# R08 · Compact source-distribution critic

Reproduce the selected compact source-distribution critic with frozen query plans, exact raw labels, byte-matched controls and live query-cost measurement.

**Question:** can five bytes of within-cell distribution information produce a more accurate and useful
error assessment than SA or a finer count sidecar under the same storage budget?

This notebook reproduces one **already selected** candidate. It does not search models, tune the formula,
recalibrate an acceptance gate or change the ARM histogram answer. The selected critic uses measured source-cell
means, variances and covariance. It is an analytic reference estimator with **zero learned model weights**.
Moment matching and Gaussian copulas are established techniques; this implementation alone is not a novelty claim.

## Run on Databricks

1. Pull this branch in your personal Git folder. Open R08 and attach the same runtime used for R07.
2. Run the dependency/configuration cells. Normally **leave `run_dir` blank** to choose your latest completed
   **R07 `fresh-query-validation-...`** result. If an old widget value persists, clear it first. To reproduce a specific
   input, enter that completed R07 directory. Do not enter R05 or R06, a notebook filename or a ZIP path.
3. Leave `seeds=0,1,2`, `fresh_query_seed=2026111602` and `replication_cells=1000` unchanged for the planned replication.
   The seed controls query geometry; rerunning it is a replication of the same queries, not a new independent test.
4. Normally leave relocation fields blank. Previous R05/R01 artifacts and the matching raw point files must remain
   accessible. `raw_data_dir` is a folder containing the unchanged0.csv,1.csv,2.csv.gz and3.csv.gz files.
5. Run All. New moment payloads and three count-only controls are constructed from verified raw rows. Query geometry,
   implementation/payload hashes and all evaluator scores are frozen **before** new exact labels are computed.
6. Send **`compact_distribution_results.zip`** from the printed new result directory. Existing runs are preserved.

No LightGBM is required by default. `aggregate_dir` is an **optional** saved R1 H-aware control directory containing
`aggregate_seed_<seed>/aggregate_relative_l1/{model.txt,schema.json}`. Leave it blank if those development artifacts
are unavailable. The skipped control is listed explicitly; another model is never substituted. This notebook does not
train that control and does not claim superiority over it when it is missing.

## What is stored and what is scored?

- The original merged ARM histogram still returns H. Every evaluator assesses exactly that same H.
- The candidate retains each exact source histogram and a64-byte-header payload with five bytes per source cell:
  mean x, mean y, variance x, variance y and covariance xy. It estimates mass in partially intersected source cells;
  fully contained cells contribute their exact source mass.
- A moment-matched beta distribution models each axis. A fixed Gaussian copula approximates dependence. Its Pearson
  correlation plug-in does not exactly preserve covariance with beta marginals. Quantization and model assumptions
  can introduce error; this is **not** a confidence bound.
- The count-derived error proxy is `abs(H-reference)/max(reference,1)`, shared by all reference-count controls.
  It is compared with actual `abs(H-truth)/truth`, never with another method's changed answer.

## Frozen controls

| Evaluator | Information | Current-dataset auxiliary bytes |
|---|---|---:|
| Selected beta + Gaussian copula | Exact coarse source histograms + five quantized moments per cell |108,064|
| Independent beta ablation | Same payload, ignores covariance dependence |108,064|
| Matched finer count critic | Largest compact square count grid fitting the candidate auxiliary cap:164×164 |107,648|
| Larger finer count critic | Largest grid fitting raw2's auxiliary cap:186×186 |138,448|
| Raw2 proxy | Existing exact2×2 source subdivisions |139,664|
| Original SA | Original live SA formula |8 for saved alpha, plus unchanged H|

A separate **direct-answer** control uses the candidate's complete storage budget for a247×247 count grid
(244,100B on the current dataset). It can change the returned answer and needs no original H. Its answer-quality
comparison is reported separately; its storage must not be mixed with a fixed-H critic that still retains H.

The unchanged H adds137,320B on the current dataset. Actual persisted files, derived memory and initialization costs
are reported; these figures are not inferred from feature counts. Count grids are built directly from raw points,
including non-dyadic resolutions. Coarse histograms plus moments are a retained source union; arbitrary heterogeneous
ARM merges are not implemented as a bounded-size guarantee.

## Evaluation scope

The fixed query seed creates new source-cell-sized, diagonally shifted rectangles. Exact and near duplicates of all
original queries and17 geometry are rejected before reading new truth. `new_groups` is PRIMARY;
`existing_test_groups` is SUPPLEMENTARY. Their generating populations were already examined in17: these are new
rectangles/labels on the **same raw dataset**, not a new dataset or strict geographic holdout.

Report full-cohort error-prediction MAE/P90, bad-answer AUC, complete-tie10/25/50/100% ranking diagnostics and paired
generating-cell effects. Seed appearances are averaged before clustered bootstrap; they are not independent samples.
Actual error at truth0 is0 if H0 and infinity otherwise. The inherited truth>=10 cohort is an offline reporting filter,
never a substitute for an unknown count at inference. Zero/small-count queries remain in diagnostics.

There is no5% accepted-risk guarantee or newly fitted operational gate. Resident evaluator timing measures live H,
features, analytic calculation/inference and score mapping. It separately records loading/initialization, construction,
offline exact labeling, warmups, repeats and cache state. It does not include raw fallback and is not an E2E speedup claim.
## Position in the retained series

R05 establishes the value of within-cell information; R06 and R07 retain the budget and generalization controls.
R08 is the selected candidate. Legacy source15/source17 artifact fields remain unchanged, and `run_dir` still
selects a completed `fresh-query-validation-...` result produced by R07.


## Dependencies

Only missing packages are installed. The selected analytic evaluator needs NumPy, pandas and SciPy; plots use matplotlib.
For comparable numbers, use the same Databricks runtime as17. Installed versions are recorded in the run manifest.
If you explicitly provide the optional saved LightGBM control, install its matching LightGBM version separately.

In [ ]:
import importlib.util
import subprocess
import sys

_required = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "matplotlib": "matplotlib"}
_missing = [package for module, package in _required.items() if importlib.util.find_spec(module) is None]
if _missing:
    try:
        get_ipython().run_line_magic("pip", "install " + " ".join(_missing))
    except (NameError, AttributeError):
        subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
print("Dependencies ready. No LightGBM is needed for the default comparison.")

## Configuration

Defaults locate your current Git folder and personal result directory. Relocation overrides must still match inherited
SHA256 identities. Settings are frozen before labels; do not search query seeds after inspecting outcomes.

In [ ]:
import json
import tempfile
from pathlib import Path

def _defaults19():
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        folder = context.notebookPath().get().split("/ml/notebooks/")[0]
        repo = folder if folder.startswith("/Workspace/") else "/Workspace" + folder
        output = "/Workspace/Users/" + context.userName().get() + "/synopsislake-results"
    except (NameError, AttributeError):
        repo, output = str(Path.cwd()), str(Path(tempfile.gettempdir()) / "synopsislake-results")
    return dict(repo_root=repo, output_root=output, run_dir="", source15_dir="", source08d_dir="",
                raw_data_dir="", queries_file="", histogram_dir="", raw2_dir="", aggregate_dir="",
                seeds="0,1,2", fresh_query_seed="2026111602", replication_cells="1000",
                timing_queries="30", timing_repeats="5")

DEFAULTS19 = _defaults19()
try:
    for key, value in DEFAULTS19.items():
        dbutils.widgets.text(key, value)
    _get19 = dbutils.widgets.get
except NameError:
    def _get19(key):
        return DEFAULTS19[key]

cfg19 = {key: str(_get19(key)).strip() for key in DEFAULTS19}
cfg19["seeds"] = [int(value) for value in cfg19["seeds"].split(",")]
if not cfg19["seeds"] or len(set(cfg19["seeds"])) != len(cfg19["seeds"]):
    raise ValueError("Use distinct seeds present in the saved17 run")
for key in ("fresh_query_seed", "replication_cells", "timing_queries", "timing_repeats"):
    cfg19[key] = int(cfg19[key])
    if cfg19[key] < (0 if key == "replication_cells" else 1):
        raise ValueError(key + " has an invalid value")
print(json.dumps(cfg19, indent=2))

## Embedded, reviewable implementation

These helpers are bundled inside this notebook. They are materialized in a unique private run directory, then imported
as an isolated package. This keeps `__file__` and hash-based provenance meaningful without requiring extra repository files.
Only harmless histogram/raw readers are reused from the offline preparation module. No earlier notebook entry point,
model fitting, candidate search or calibration function is executed.

In [ ]:
EMBEDDED19 = {}
EMBEDDED19_SHA256 = {'input_core.py': 'af871e95a7b763eef10069da0645ee360dd74e0e46f63cda5c371ffbcc350975', 'prepare_data.py': 'bf69f5dc1077a9bb9eaccbe6783b51747db5af4b24939630cb353136e97f8255', 'runtime_features.py': 'c239ac45f79f323d063e6d07afc7bb3546b1c1bb7e7951214f266372aeeb3914', 'moment_metadata.py': 'a14c82cab9bb6c544cf0d90ed96a283949a910714ed764849b5f30f55d6ba673', 'analytic_moments_fast.py': '6102c3feecf3f67139e3a08a3aa0df5471f52bdc899c11674cc5c23b9cf520cc', 'resident_critic.py': '98a8763773d6ee57cea9d65914aaed3a2458f1b2139e14c50d1dd86a543b0956', 'budget_fine.py': '5199e21fdf97fe61c1aee34d32f6eb289b014720559dba4d4fbd14314cec5a6c', 'generation19.py': '3310aa54bf35b927777b3d0d8aed59c4a5dce89590fbdcf004a100b80e4eadc2', 'exact_counter.py': 'ada14e6fa989b7fe1843f6946cf0103cf2173de7616b1b9c365f96370ff8fd3b', 'sa_core.py': '1620ce0b7c5c8266bde93140a06ba02fe5b230fadd528778dc071c1413e2c858', 'evaluate.py': '9bdb4172c22dedd4b304b641f24541b241f46b9a604a53d5deefbbdd75e6437f', 'confirm_portable.py': '05ccfedda718b5cc90066e9d9b028ed16dda677eecd925fde7f5a1ff4712c8c2', 'workflow.py': '5d61b528352e74a19618cc528f65f50d6ea39fe09df6f4e5a94bd6c990752c13'}


In [ ]:
# Embedded helper: input_core.py
EMBEDDED19['input_core.py'] = r'''"""Verified 17/15 inputs and exact, one-pass raw count construction for 18.

Only old training geometry is returned for layout selection. Fresh labels are
returned separately for offline validation; they are not construction inputs.
No model, fitted feature provider or training entry point is loaded.
"""
from pathlib import Path
import ast
import base64
import gzip
import hashlib
import json
import math
import struct
import time
import numpy as np
import pandas as pd

def input_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def _input_json(path):
    return json.loads(Path(path).read_text())

def _input_verified(path, digest, description):
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError('Missing ' + description + ': ' + str(path))
    if input_sha(path) != digest:
        raise ValueError('SHA256 mismatch for ' + description + ': ' + str(path))
    return path.resolve()

def _input_run_dir(cfg):
    entered = str(cfg.get('run_dir', '')).strip()
    if entered:
        path = Path(entered).expanduser()
        if not (path / 'run.json').is_file():
            raise FileNotFoundError('run_dir must be a completed 17 fresh-query-validation directory')
        return path.resolve()
    candidates = []
    root = Path(cfg['output_root']).expanduser()
    for file in root.glob('fresh-query-validation-*/run.json'):
        try:
            record = _input_json(file)
            if record.get('status') == 'complete' and record.get('study') == 'fresh-query-validation':
                candidates.append(file)
        except (OSError, ValueError):
            continue
    if not candidates:
        raise FileNotFoundError('No completed 17 run under output_root. Set run_dir to its full directory')
    return max(candidates, key=lambda p: p.stat().st_mtime_ns).parent.resolve()

def _input_parent(original, digest, cfg, description, override=''):
    choices = []
    if override:
        choices.append(Path(override).expanduser() / 'run.json')
    else:
        choices.append(Path(original) / 'run.json')
        choices.extend(Path(cfg['output_root']).expanduser().glob('*/run.json'))
    for file in choices:
        if file.is_file() and input_sha(file) == digest:
            return (file.parent.resolve(), _input_json(file))
    raise FileNotFoundError('Missing matching ' + description + ' manifest. Preserve its original result folder or set its relocation override')

def _input_hist_loader(repo):
    file = Path(repo) / 'ml/notebooks/14_compact_metadata_baseline.ipynb'
    try:
        raw = file.read_bytes()
        document = json.loads(raw)
        method = 'workspace filesystem'
    except (OSError, ValueError, UnicodeError):
        if not str(file).startswith('/Workspace/'):
            raise
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.workspace import ExportFormat
        client = WorkspaceClient()
        last = None
        for candidate in (str(file)[len('/Workspace'):], str(file)[len('/Workspace'):].removesuffix('.ipynb')):
            try:
                exported = client.workspace.export(path=candidate, format=ExportFormat.JUPYTER)
                raw = base64.b64decode(exported.content)
                document = json.loads(raw)
                method = 'Workspace API using runtime identity'
                break
            except Exception as error:
                last = error
        else:
            raise ValueError('Cannot read notebook12; check repo_root and workspace permissions: ' + str(last)) from last
    digest = '1ac8d26a90d6e402c1223960b898fee17f79e9cea33c7656a7b699c6b30da8d3'
    matches = []
    for cell in document['cells']:
        if cell.get('cell_type') != 'code':
            continue
        try:
            candidate = ast.parse(''.join(cell['source']))
        except SyntaxError:
            continue
        if hashlib.sha256(ast.dump(candidate, include_attributes=False).encode()).hexdigest() == digest:
            matches.append(candidate)
    if len(matches) != 1:
        raise ValueError('Notebook12 histogram codec definitions changed/missing; review compatibility before running18')
    tree = matches[0]
    names = {'validate_hist', 'load_hist_payload'}
    selected = ast.Module(body=[node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in names], type_ignores=[])
    if {node.name for node in selected.body} != names:
        raise ValueError('Missing inherited histogram reader definitions')
    namespace = dict(np=np, Path=Path, _HIST_MAGIC=b'SLHIST01', _HIST_HEADER=struct.Struct('<IId'))
    exec(compile(selected, str(file) + '::readers', 'exec'), namespace)
    return (namespace['load_hist_payload'], dict(path=str(file), read_method=method, document_sha256=hashlib.sha256(raw).hexdigest(), definition_ast_sha256=digest))

def load_inputs(cfg, destination, state):
    """Read immutable identities/geometry; never fit or initialize a model."""
    tick = time.perf_counter()
    destination = Path(destination)
    source17 = _input_run_dir(cfg)
    prior17 = _input_json(source17 / 'run.json')
    if prior17.get('status') != 'complete' or prior17.get('study') != 'fresh-query-validation':
        raise ValueError('run_dir must identify completed notebook 17, not 15/16 or an old diagnostic')
    required17 = {'new_queries.csv': prior17['new_query_labels_sha256'], 'protocol_lock.json': prior17['protocol_sha256'], 'query_plan.csv': prior17['query_plan_sha256'], 'query_membership.csv': prior17['query_membership_sha256']}
    for name, digest in required17.items():
        _input_verified(source17 / name, digest, 'frozen17 ' + name)
    protocol = _input_json(source17 / 'protocol_lock.json')
    if protocol['query_plan_sha256'] != required17['query_plan.csv'] or protocol['membership_sha256'] != required17['query_membership.csv']:
        raise ValueError('17 manifest and protocol disagree about geometry/membership')
    labels = pd.read_csv(source17 / 'new_queries.csv', float_precision='round_trip')
    membership = pd.read_csv(source17 / 'query_membership.csv')
    plan = pd.read_csv(source17 / 'query_plan.csv', float_precision='round_trip')
    if labels.query_id.duplicated().any() or plan.query_id.duplicated().any() or membership.duplicated(['seed', 'query_id']).any():
        raise ValueError('Duplicate fresh query or membership identity')
    geometry = ['query_id', 'family', 'cell', 'x0', 'x1', 'y0', 'y1']
    if not labels[geometry].reset_index(drop=True).equals(plan[geometry].reset_index(drop=True)):
        raise ValueError('17 labelled geometry differs from the frozen pre-label plan')
    truth = labels.truth.to_numpy(float)
    rectangles = labels[['x0', 'x1', 'y0', 'y1']].to_numpy(float)
    if not np.isfinite(truth).all() or (truth < 0).any() or (not np.equal(truth, np.floor(truth)).all()) or (not np.isfinite(rectangles).all()) or (rectangles[:, 0] > rectangles[:, 1]).any() or (rectangles[:, 2] > rectangles[:, 3]).any():
        raise ValueError('Invalid exact count or rectangle in saved17 labels')

    def saved_bool(column):
        values = labels[column].astype(str).str.lower()
        if not values.isin(['true', 'false']).all():
            raise ValueError('Invalid saved17 boolean: ' + column)
        return values.eq('true').to_numpy()
    if not np.array_equal(saved_bool('eligible'), truth >= 10) or not np.array_equal(saved_bool('zero_truth'), truth == 0):
        raise ValueError('Saved17 eligibility/zero flags changed')
    labels['eligible'] = truth >= 10
    labels['zero_truth'] = truth == 0
    if not set(membership.query_id).issubset(set(labels.query_id)):
        raise ValueError('17 membership contains unknown queries')
    linked = membership.merge(labels[['query_id', 'family', 'cell']], on='query_id', validate='many_to_one', suffixes=('_member', '_label'))
    if not linked.family_member.equals(linked.family_label) or not linked.cell_member.equals(linked.cell_label):
        raise ValueError('17 family/cell membership identity mismatch')
    requested = list(map(int, cfg['seeds']))
    if not requested or len(set(requested)) != len(requested) or (not set(requested).issubset(set(membership.seed))):
        raise ValueError('Requested seeds are absent/duplicated in saved17 memberships')
    membership = membership[membership.seed.isin(requested)].copy()
    source15, prior15 = _input_parent(prior17['source15_run'], prior17['source15_manifest_sha256'], cfg, 'source15', str(cfg.get('source15_dir', '')).strip())
    for name in ('test_predictions.csv', 'outer_splits.csv', 'raw_input_manifest.json', 'payloads.csv'):
        _input_verified(source15 / name, prior17['source15_artifact_sha256'][name], 'source15 ' + name)
    outer = pd.read_csv(source15 / 'outer_splits.csv')
    if 'split_kind' in outer and set(outer.split_kind) != {'cell'}:
        raise ValueError('18 requires the saved cell-group outer split')
    outer = outer[outer.seed.isin(requested)].copy()
    if outer.duplicated(['seed', 'qId']).any():
        raise ValueError('Duplicate old outer role assignments')
    for seed in requested:
        part = outer[outer.seed == seed]
        if set(part.split) != {'train', 'calibration', 'test'}:
            raise ValueError('Incomplete original outer roles for seed ' + str(seed))
        sets = [set(part.loc[part.split == role, 'cell']) for role in ('train', 'calibration', 'test')]
        if any((sets[i] & sets[j] for i in range(3) for j in range(i))):
            raise ValueError('Original outer generating cells overlap between roles')
    all_outer = pd.read_csv(source15 / 'outer_splits.csv')
    if set(membership.loc[membership.family == 'new_groups', 'cell']) & set(all_outer.cell):
        raise ValueError('Saved17 primary group unexpectedly overlaps an old outer group')
    for seed in requested:
        supplement = set(membership.loc[(membership.seed == seed) & (membership.family == 'existing_test_groups'), 'cell'])
        heldout = set(outer.loc[(outer.seed == seed) & (outer.split == 'test'), 'cell'])
        if not supplement.issubset(heldout):
            raise ValueError('Saved17 supplement is not the corresponding seed held-out group')
    query_item = prior17['verified_inputs']['data/queries-Ans.txt']
    query_path = _input_verified(str(cfg.get('queries_file', '')).strip() or query_item['path'], query_item['sha256'], 'original training query geometry')
    old = pd.read_csv(query_path, header=None).to_numpy(float)
    if old.ndim != 2 or old.shape[1] != 5 or (not np.isfinite(old).all()):
        raise ValueError('Original query file must contain four coordinates and a count')
    if outer.qId.min() < 0 or outer.qId.max() >= len(old):
        raise ValueError('Old outer IDs lie outside original query file')
    old_queries = old[:, :4].copy()
    del old
    saved = pd.read_csv(source15 / 'test_predictions.csv', usecols=['histogram_key', 'base_rate'])
    pairs = saved.drop_duplicates()
    if len(pairs) != 1:
        raise ValueError('18 currently requires one fixed original merged histogram/rate; do not select by new quality')
    base_rate = float(pairs.iloc[0].base_rate)
    key = str(pairs.iloc[0].histogram_key)
    table = pd.read_csv(source15 / 'payloads.csv')
    row = table[(table.variant == 'raw2_source') & np.isclose(table.base_rate, base_rate, rtol=0, atol=1e-12)]
    if len(row) != 1:
        raise ValueError('Missing unique raw2 byte budget in15 payload table')
    inherited_budget = int(row.iloc[0].total_synopsis_bytes)
    entered_budget = int(cfg.get('budget_bytes', 0) or 0)
    budget = entered_budget if entered_budget else inherited_budget
    if budget < 80:
        raise ValueError('Budget must fit the common64-byte header and at least one refinement')
    source08d, original = _input_parent(prior15['source_run'], prior15['source_manifest_sha256'], cfg, 'original08d', str(cfg.get('source08d_dir', '')).strip())
    export_key = 'run/' + key + '.txt'
    manifest_file = source15 / 'preparation/input_manifest.json'
    manifest = _input_json(manifest_file) if manifest_file.is_file() else {}
    export_item = manifest.get(export_key, dict(path=str(source08d / 'inputs' / export_key), sha256=original['input_sha256'][export_key]))
    if export_item['sha256'] != original['input_sha256'][export_key]:
        raise ValueError('15 staged histogram identity differs from original08d')
    export_path = Path(cfg['histogram_dir']) / (key + '.txt') if str(cfg.get('histogram_dir', '')).strip() else Path(export_item['path'])
    export_path = _input_verified(export_path, export_item['sha256'], 'original merged histogram export')
    lines = export_path.read_text().splitlines()
    bx = np.asarray(lines[1].split(','), float)
    by = np.asarray(lines[2].split(','), float)
    if not np.isfinite(np.r_[bx, by]).all() or (np.diff(bx) <= 0).any() or (np.diff(by) <= 0).any():
        raise ValueError('Invalid original merged histogram bounds')
    loader, reuse = _input_hist_loader(cfg['repo_root'])
    binary_path = source15 / 'preparation/payloads' / (key + '.hist')
    if binary_path.is_file():
        hist = loader(binary_path)
        export_counts = np.asarray([line.split(',') for line in lines[3:]], float)
        if not np.array_equal(hist[1], bx) or not np.array_equal(hist[2], by) or (not np.array_equal(hist[3], export_counts)):
            raise ValueError('15 binary merged histogram differs from verified original export')
        if binary_path.stat().st_size != int(row.iloc[0].base_histogram_bytes):
            raise ValueError('Original merged histogram byte budget changed')
    raw_manifest = _input_json(source15 / 'raw_input_manifest.json')
    if not raw_manifest or len({str(r['source_id']) for r in raw_manifest}) != len(raw_manifest):
        raise ValueError('Missing/duplicate raw sources')
    parity = pd.read_csv(source15 / 'raw_source_parity.csv')
    if parity.source_id.duplicated().any() or not parity.subcell_mass_conserved.astype(str).str.lower().eq('true').all() or (not parity.max_coarse_count_gap.eq(0).all()):
        raise ValueError('Source15 exact-raw/coarse parity did not pass')
    parity = parity.set_index(parity.source_id.astype(str))
    resolved = []
    for record in raw_manifest:
        source_id = str(record['source_id'])
        name = Path(record['raw_path']).name
        choices = [Path(cfg['raw_data_dir']) / name] if str(cfg.get('raw_data_dir', '')).strip() else [Path(record['raw_path']), Path(cfg['repo_root']) / 'range_query' / name]
        path = next((p for p in choices if p.is_file()), None)
        if path is None:
            raise FileNotFoundError('Matching raw data missing: ' + name + '. Set raw_data_dir to the unchanged raw-file folder')
        expected = int(parity.loc[source_id, 'raw_rows'])
        if expected < 0:
            raise ValueError('Invalid expected raw row count')
        resolved.append(dict(record, raw_path=str(path.resolve()), expected_rows=expected))
    if int(row.iloc[0].auxiliary_bytes) != sum((int(r['payload_bytes']) for r in raw_manifest)):
        raise ValueError('Inherited raw2 auxiliary budget does not match frozen payload sizes')
    if inherited_budget != int(row.iloc[0].base_histogram_bytes) + int(row.iloc[0].auxiliary_bytes):
        raise ValueError('Inherited raw2 total byte budget is inconsistent')
    if sum((r['expected_rows'] for r in resolved)) > np.iinfo(np.uint32).max:
        raise OverflowError('This prototype codec requires the entire row count to fit uint32')
    outer.to_csv(destination / 'outer_splits.csv', index=False)
    labels.to_csv(destination / 'new_queries.csv', index=False, float_format='%.17g')
    membership.to_csv(destination / 'query_membership.csv', index=False)
    state.update(source17_run=str(source17), source17_manifest_sha256=input_sha(source17 / 'run.json'), source17_artifact_sha256=required17, source15_run=str(source15), source15_manifest_sha256=input_sha(source15 / 'run.json'), source08d_run=str(source08d), source08d_manifest_sha256=input_sha(source08d / 'run.json'), old_query_file=dict(path=str(query_path), sha256=query_item['sha256']), original_histogram=dict(key=key, path=str(export_path), sha256=export_item['sha256'], base_rate=base_rate), inherited_raw2_budget_bytes=inherited_budget, budget_bytes=budget, reuse_histogram_codec=reuse)
    state.setdefault('timings', {})['input_validation_seconds'] = time.perf_counter() - tick
    return dict(labels=labels, membership=membership, old_queries=old_queries, outer=outer, root=(float(bx[0]), float(bx[-1]), float(by[0]), float(by[-1])), raw_manifest=resolved, budget_bytes=budget, base_rate=base_rate)

def iter_count_raw_points(path, chunk_size=100000):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as stream:
        for part in pd.read_csv(stream, sep='\t', header=None, usecols=[0, 1], dtype=float, chunksize=chunk_size, float_precision='round_trip', skip_blank_lines=False):
            points = part.to_numpy(float)
            if points.shape[1] != 2 or not np.isfinite(points).all():
                raise ValueError('Invalid/nonfinite raw coordinates in ' + str(path))
            yield points

def construct_count_inputs(data, max_depth, uniform_n, destination, state):
    """One raw pass constructs both exact dyadic pilots and non-dyadic grids.

    Uniform263 is counted directly. Resampling a1024 pilot would lose exactness
    and invalidate the subsequent lower/upper count guarantees.
    """
    max_depth = int(max_depth)
    uniform_n = int(uniform_n)
    if not 0 <= max_depth <= 12 or uniform_n < 1:
        raise ValueError('Pilot max_depth must be in0..12 and uniform_n positive')
    side = 2 ** max_depth
    x0, x1, y0, y1 = map(float, data['root'])
    pilot_edges = (np.linspace(x0, x1, side + 1), np.linspace(y0, y1, side + 1))
    uniform_edges = (np.linspace(x0, x1, uniform_n + 1), np.linspace(y0, y1, uniform_n + 1))
    if any(((np.diff(edges) <= 0).any() for edges in (*pilot_edges, *uniform_edges))):
        raise ValueError('Requested count-grid boundaries collapse in float64')
    total_tick = time.perf_counter()
    sources = []
    costs = []
    persistent_numeric = 0
    peak_numeric = 0
    for record in data['raw_manifest']:
        path = Path(record['raw_path'])
        tick = time.perf_counter()
        _input_verified(path, record['raw_sha256'], 'raw source' + str(record['source_id']))
        hash_before_seconds = time.perf_counter() - tick
        pilot = np.zeros((side, side), np.uint32)
        uniform = np.zeros((uniform_n, uniform_n), np.uint32)
        rows = 0
        scan_tick = time.perf_counter()
        largest_chunk = 0
        for points in iter_count_raw_points(path):
            rows += len(points)
            largest_chunk = max(largest_chunk, len(points))
            if ((points[:, 0] < x0) | (points[:, 0] > x1) | (points[:, 1] < y0) | (points[:, 1] > y1)).any():
                raise ValueError('Raw point lies outside the inherited common root; cannot certify dropped points')
            for counts, edges, n in ((pilot, pilot_edges, side), (uniform, uniform_edges, uniform_n)):
                ix = np.minimum(np.searchsorted(edges[0], points[:, 0], side='right') - 1, n - 1)
                iy = np.minimum(np.searchsorted(edges[1], points[:, 1], side='right') - 1, n - 1)
                additions = np.bincount(ix * n + iy, minlength=n * n).reshape(n, n)
                summed = counts.astype(np.uint64) + additions.astype(np.uint64)
                if summed.max(initial=0) > np.iinfo(np.uint32).max:
                    raise OverflowError('Exact source-cell mass exceeds uint32 codec')
                counts[:] = summed
                numeric = persistent_numeric + pilot.nbytes + uniform.nbytes + points.nbytes + ix.nbytes + iy.nbytes + additions.nbytes + summed.nbytes + max(pilot.size, uniform.size) * 16 + len(points) * 16
                peak_numeric = max(peak_numeric, int(numeric))
            del points, ix, iy, additions, summed
        scan_seconds = time.perf_counter() - scan_tick
        tick = time.perf_counter()
        _input_verified(path, record['raw_sha256'], 'raw source after construction')
        hash_after_seconds = time.perf_counter() - tick
        if rows != int(record['expected_rows']) or int(pilot.sum(dtype=np.uint64)) != rows or int(uniform.sum(dtype=np.uint64)) != rows:
            raise ValueError('Raw row count or exact count-grid conservation mismatch for source' + str(record['source_id']))
        sources.append(dict(source_id=str(record['source_id']), pilot=pilot, uniform=uniform))
        persistent_numeric += pilot.nbytes + uniform.nbytes
        costs.append(dict(source_id=str(record['source_id']), raw_path=str(path), raw_sha256=record['raw_sha256'], raw_file_bytes=path.stat().st_size, raw_rows=rows, pilot_side=side, uniform_side=uniform_n, pilot_array_bytes=pilot.nbytes, uniform_array_bytes=uniform.nbytes, hash_before_seconds=hash_before_seconds, raw_scan_both_grids_seconds=scan_seconds, hash_after_seconds=hash_after_seconds, largest_chunk_rows=largest_chunk, row_mass_conserved=True, uniform_counted_directly_from_raw=True))
    accumulator = np.zeros((side, side), np.uint64)
    for source in sources:
        accumulator += source['pilot']
    if accumulator.max(initial=0) > np.iinfo(np.uint32).max:
        raise OverflowError('Global pilot-cell mass exceeds uint32')
    global_pilot = accumulator.astype(np.uint32)
    peak_numeric = max(peak_numeric, persistent_numeric + accumulator.nbytes + global_pilot.nbytes)
    del accumulator
    pd.DataFrame(costs).to_csv(Path(destination) / 'raw_construction.csv', index=False)
    state['raw_input_identity'] = [dict(source_id=str(r['source_id']), raw_sha256=r['raw_sha256'], raw_rows=int(r['expected_rows'])) for r in data['raw_manifest']]
    state.setdefault('construction', {}).update(pilot_side=side, uniform_side=uniform_n, pilot_total_rows=int(global_pilot.sum(dtype=np.uint64)), persistent_count_input_array_bytes=persistent_numeric + global_pilot.nbytes, raw_scan_numeric_array_allocation_bound_bytes=int(peak_numeric), memory_measure='Raw-scan numeric-array allocation bound only; excludes pandas/parser/Python objects and later layout-search/summary-index transients; not process peak RSS', shared_layout_pilot_is_coordinator_input=True, uniform_counted_directly_from_raw=True, raw_semantics='TAB first2 fields; round-trip float parser; every row one point; duplicates retained; trailing fields ignored; internal half-open cells and inclusive outer maxima')
    state.setdefault('timings', {})['raw_construction_with_hashes_seconds'] = time.perf_counter() - total_tick
    return (sources, global_pilot)
'''

In [ ]:
# Embedded helper: prepare_data.py
EMBEDDED19['prepare_data.py'] = r'''"""Exact partial-source-cell labels for a local occupancy learning experiment.

Offline diagnostic preparation only. The raw labels and query truth are saved
separately from the explicit source-only feature matrix. No model is trained.
"""
from pathlib import Path
import argparse
import gzip
import hashlib
import json
import time
import numpy as np
import pandas as pd
LOCAL_FEATURE_NAMES = ['local_x0', 'local_x1', 'local_y0', 'local_y1', 'quadrant_00_fraction', 'quadrant_01_fraction', 'quadrant_10_fraction', 'quadrant_11_fraction', 'log1p_cell_mass', 'coverage', 'relative_query_width', 'relative_query_height', 'log_aspect', 'raw2_fraction', 'raw2_lower_fraction', 'raw2_upper_fraction']

def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def load_hist(path):
    path = Path(path)
    if path.suffix == '.hist':
        from .runtime_features import _load_source_payload
        return _load_source_payload(path)
    lines = path.read_text().splitlines()
    hist = (float(lines[0]), np.asarray(lines[1].split(','), float), np.asarray(lines[2].split(','), float), np.asarray([line.split(',') for line in lines[3:]], float))
    if hist[3].shape != (len(hist[1]) - 1, len(hist[2]) - 1) or not np.isfinite(hist[3]).all() or (hist[3] < 0).any():
        raise ValueError('Invalid histogram export: ' + str(path))
    return hist

def iter_raw(path, chunk_size=100000):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as stream:
        for part in pd.read_csv(stream, sep='\t', header=None, usecols=[0, 1], dtype=float, chunksize=chunk_size, float_precision='round_trip', skip_blank_lines=False):
            values = part.to_numpy(float)
            if not np.isfinite(values).all():
                raise ValueError('Nonfinite raw point')
            yield values

def estimate_count(q, hist):
    _, bx, by, counts = hist
    i0 = max(0, int(np.searchsorted(bx, q[0], side='right') - 1))
    i1 = min(len(bx) - 1, int(np.searchsorted(bx, q[1], side='left')))
    j0 = max(0, int(np.searchsorted(by, q[2], side='right') - 1))
    j1 = min(len(by) - 1, int(np.searchsorted(by, q[3], side='left')))
    if i1 <= i0 or j1 <= j0:
        return 0.0
    fx = np.maximum(0.0, np.minimum(q[1], bx[i0 + 1:i1 + 1]) - np.maximum(q[0], bx[i0:i1])) / np.diff(bx[i0:i1 + 1])
    fy = np.maximum(0.0, np.minimum(q[3], by[j0 + 1:j1 + 1]) - np.maximum(q[2], by[j0:j1])) / np.diff(by[j0:j1 + 1])
    return float(np.sum(counts[i0:i1, j0:j1] * fx[:, None] * fy[None, :]))
'''

In [ ]:
# Embedded helper: runtime_features.py
EMBEDDED19['runtime_features.py'] = r'''"""Resident source-summary geometry features with no raw points or truth.

The sole feature-provider inputs are query rectangles, loaded exact source
histograms/raw2 payloads and a fixed root. All supervised labels stay in the
separate offline preparation module. Local and aggregate feature lists are
frozen before model fitting; this module reproduces those definitions exactly.
"""
from pathlib import Path
import re
import struct
import numpy as np
import pandas as pd
LOCAL_FEATURE_NAMES = ['local_x0', 'local_x1', 'local_y0', 'local_y1', 'quadrant_00_fraction', 'quadrant_01_fraction', 'quadrant_10_fraction', 'quadrant_11_fraction', 'log1p_cell_mass', 'coverage', 'relative_query_width', 'relative_query_height', 'log_aspect', 'raw2_fraction', 'raw2_lower_fraction', 'raw2_upper_fraction']
AGG_FEATURE_NAMES = ['log1p_full_mass', 'log1p_partial_mass', 'log1p_raw2_boundary_estimate', 'log1p_raw2_remaining_mass', 'raw2_boundary_fraction', 'full_share', 'log1p_n_partial', 'log1p_n_full', 'query_width_root_fraction', 'query_height_root_fraction', 'raw2_partial_contrast_mean', 'raw2_partial_empty_quadrant_share', 'n_source_overlap', 'coarse_boundary_fraction']
COARSE_FEATURE_NAMES = ['local_x0', 'local_x1', 'local_y0', 'local_y1', 'log1p_cell_mass', 'coverage', 'relative_query_width', 'relative_query_height', 'log_aspect']
_COARSE_MAGIC = b'SLHIST01'
_COARSE_HEADER = struct.Struct('<IId')

def _load_source_export(path):
    lines = Path(path).read_text().splitlines()
    return (float(lines[0]), np.asarray(lines[1].split(','), float), np.asarray(lines[2].split(','), float), np.asarray([line.split(',') for line in lines[3:]], float))

def save_coarse_payload(path, hist):
    """Existing12 SLHIST01 codec; actual file size includes full boundaries."""
    rate, bx, by, counts = hist
    bx = np.asarray(bx, float)
    by = np.asarray(by, float)
    counts = np.asarray(counts, float)
    if counts.ndim != 2 or counts.shape != (len(bx) - 1, len(by) - 1) or min(counts.shape) < 1:
        raise ValueError('Invalid coarse source histogram dimensions')
    if not np.isfinite(rate) or rate < 0 or (not np.isfinite(np.r_[bx, by, counts.ravel()]).all()):
        raise ValueError('Invalid coarse source values')
    if not (np.diff(bx) > 0).all() or not (np.diff(by) > 0).all() or (counts < 0).any():
        raise ValueError('Invalid coarse source geometry/counts')
    path = Path(path)
    with path.open('wb') as stream:
        stream.write(_COARSE_MAGIC)
        stream.write(_COARSE_HEADER.pack(counts.shape[0], counts.shape[1], float(rate)))
        for array in (bx, by, counts):
            stream.write(np.asarray(array, dtype='<f8').tobytes(order='C'))
    return path.stat().st_size

def _load_source_payload(path):
    """Strict existing12 binary histogram reader, with owned resident arrays."""
    payload = Path(path).read_bytes()
    header = 8 + _COARSE_HEADER.size
    if len(payload) < header or payload[:8] != _COARSE_MAGIC:
        raise ValueError('Invalid coarse histogram artifact header')
    nx, ny, rate = _COARSE_HEADER.unpack_from(payload, 8)
    if nx < 1 or ny < 1 or len(payload) != header + 8 * (nx + 1 + (ny + 1) + nx * ny):
        raise ValueError('Invalid coarse histogram artifact dimensions/length')
    numbers = np.frombuffer(payload, dtype='<f8', offset=header)
    bx = numbers[:nx + 1].copy()
    by = numbers[nx + 1:nx + ny + 2].copy()
    counts = numbers[nx + ny + 2:].reshape(nx, ny).copy()
    if not np.isfinite(rate) or rate < 0:
        raise ValueError('Invalid coarse histogram rate')
    return (float(rate), bx, by, counts)

def _validate_source_pair(hist, raw2):
    _, bx, by, coarse = hist
    _, fx, fy, fine = raw2
    for boundaries in (bx, by, fx, fy):
        if boundaries.ndim != 1 or len(boundaries) < 2 or (not np.isfinite(boundaries).all()) or (not (np.diff(boundaries) > 0).all()):
            raise ValueError('Source boundaries must be finite and increasing')
    nx, ny = coarse.shape
    if nx * ny > 65536:
        raise ValueError('Prototype coarse cell_id must fit uint16')
    if coarse.shape != (len(bx) - 1, len(by) - 1) or fine.shape != (2 * nx, 2 * ny):
        raise ValueError('Source/raw2 dimensions mismatch')
    if not np.array_equal(fx[::2], bx) or not np.array_equal(fy[::2], by):
        raise ValueError('Raw2 boundaries do not exactly refine coarse sources')
    for counts in (coarse, fine):
        if not np.isfinite(counts).all() or (counts < 0).any() or (not np.equal(counts, np.floor(counts)).all()):
            raise ValueError('Source summaries must store exact nonnegative integer masses')
        if counts.max(initial=0) > np.iinfo(np.uint32).max:
            raise OverflowError('Source cell mass exceeds prototype uint32')
    if not np.array_equal(fine.reshape(nx, 2, ny, 2).sum(axis=(1, 3)), coarse):
        raise ValueError('Raw2 source counts do not conserve original exact masses')

def source_contexts(histogram_dir, raw2_dir, validate_exports=False):
    """Load raw2 summaries once and derive their redundant coarse parents.

    Production features require only raw2 files. Original coarse exports may
    optionally be read for an initialization audit; they are never an extra
    independent persisted input. The loading change leaves every feature value
    and all frozen model inputs unchanged.
    """
    histogram_dir = Path(histogram_dir)
    raw2_dir = Path(raw2_dir)
    paths = []
    for path in raw2_dir.glob('src_*-raw2.npz'):
        match = re.fullmatch('src_(\\d+)-raw2\\.npz', path.name)
        if match:
            paths.append((int(match.group(1)), path))
    if not paths:
        raise FileNotFoundError('No raw2 source summaries found in raw2_dir')
    contexts = []
    for source_id, raw2_path in sorted(paths):
        if not 0 <= source_id <= 255:
            raise ValueError('Prototype source_id must fit uint8')
        with np.load(raw2_path, allow_pickle=False) as payload:
            if payload['version'].tolist() != [1]:
                raise ValueError('Unsupported raw2 source payload version')
            raw2 = (float(payload['rate'][0]), payload['bx'].copy(), payload['by'].copy(), payload['counts'].astype(float))
        sx, sy = raw2[3].shape
        if sx % 2 or sy % 2:
            raise ValueError('Raw2 source dimensions must be even')
        hist = (raw2[0], raw2[1][::2].copy(), raw2[2][::2].copy(), raw2[3].reshape(sx // 2, 2, sy // 2, 2).sum(axis=(1, 3)))
        _validate_source_pair(hist, raw2)
        path = histogram_dir / f'src_{source_id}.txt'
        if validate_exports:
            original = _load_source_export(path)
            if not all((np.array_equal(a, b) for a, b in zip(hist, original))):
                raise ValueError('Raw2-derived source differs from original export')
        contexts.append(dict(source_id=source_id, hist=hist, raw2=raw2, source_path=str(path) if validate_exports else None, raw2_path=str(raw2_path), source_export_bytes=path.stat().st_size if validate_exports else 0, coarse_arrays_derived_from_raw2=True, source_export_validation_performed=bool(validate_exports), raw2_payload_bytes=raw2_path.stat().st_size, numeric_array_bytes=sum((a.nbytes for a in (*hist[1:], *raw2[1:])))))
    return contexts

def _source_describe(rectangles, context):
    """Geometry and summary-mass operations only; no supervised argument."""
    source_id = context['source_id']
    _, bx, by, coarse = context['hist']
    _, fx, fy, fine = context['raw2']
    nx, ny = coarse.shape
    nq = len(rectangles)
    aggregates = {name: np.zeros(nq, float) for name in ('L', 'U', 'S', 'Scoarse', 'L_fine', 'U_fine', 'n_partial', 'n_full', 'n_source_overlap', 'contrast_mass', 'empty_quadrant_mass')}
    blocks = {name: [] for name in ('X', 'mass', 'query_index', 'source_id', 'cell_id', 'lower_fraction', 'upper_fraction', 'uniform_fraction', 'coarse_uniform_fraction')}
    for qi, q in enumerate(rectangles):
        if q[1] < bx[0] or q[0] > bx[-1] or q[3] < by[0] or (q[2] > by[-1]):
            continue
        i0 = max(0, int(np.searchsorted(bx, q[0], side='left') - 1))
        i1 = min(nx - 1, int(np.searchsorted(bx, q[1], side='right') - 1))
        j0 = max(0, int(np.searchsorted(by, q[2], side='left') - 1))
        j1 = min(ny - 1, int(np.searchsorted(by, q[3], side='right') - 1))
        if i1 < i0 or j1 < j0:
            continue
        a, b = np.meshgrid(np.arange(i0, i1 + 1), np.arange(j0, j1 + 1), indexing='ij')
        a = a.ravel()
        b = b.ravel()
        masses = coarse[a, b]
        nonempty = masses > 0
        a = a[nonempty]
        b = b[nonempty]
        masses = masses[nonempty]
        if not len(a):
            continue
        xlo, xhi, ylo, yhi = (bx[a], bx[a + 1], by[b], by[b + 1])
        touched = (q[1] >= xlo) & (q[0] <= xhi) & (q[3] >= ylo) & (q[2] <= yhi)
        a = a[touched]
        b = b[touched]
        masses = masses[touched]
        xlo = xlo[touched]
        xhi = xhi[touched]
        ylo = ylo[touched]
        yhi = yhi[touched]
        if not len(a):
            continue
        aggregates['n_source_overlap'][qi] = 1
        full = (q[0] <= xlo) & (q[1] >= xhi) & (q[2] <= ylo) & (q[3] >= yhi)
        full_mass = float(masses[full].sum())
        aggregates['L'][qi] += full_mass
        for name in ('U', 'S', 'Scoarse', 'L_fine', 'U_fine'):
            aggregates[name][qi] += full_mass
        aggregates['n_full'][qi] += int(full.sum())
        partial = ~full
        a = a[partial]
        b = b[partial]
        masses = masses[partial]
        xlo = xlo[partial]
        xhi = xhi[partial]
        ylo = ylo[partial]
        yhi = yhi[partial]
        if not len(a):
            continue
        n = len(a)
        dx = xhi - xlo
        dy = yhi - ylo
        local = np.column_stack((np.clip((q[0] - xlo) / dx, 0, 1), np.clip((q[1] - xlo) / dx, 0, 1), np.clip((q[2] - ylo) / dy, 0, 1), np.clip((q[3] - ylo) / dy, 0, 1)))
        coverage = (local[:, 1] - local[:, 0]) * (local[:, 3] - local[:, 2])
        quadrants = np.stack((fine[2 * a, 2 * b], fine[2 * a, 2 * b + 1], fine[2 * a + 1, 2 * b], fine[2 * a + 1, 2 * b + 1]), axis=1)
        fractions = quadrants / masses[:, None]
        quad_xlo = np.column_stack((fx[2 * a], fx[2 * a], fx[2 * a + 1], fx[2 * a + 1]))
        quad_xhi = np.column_stack((fx[2 * a + 1], fx[2 * a + 1], fx[2 * a + 2], fx[2 * a + 2]))
        quad_ylo = np.column_stack((fy[2 * b], fy[2 * b + 1], fy[2 * b], fy[2 * b + 1]))
        quad_yhi = np.column_stack((fy[2 * b + 1], fy[2 * b + 2], fy[2 * b + 1], fy[2 * b + 2]))
        overlaps = np.maximum(0, np.minimum(q[1], quad_xhi) - np.maximum(q[0], quad_xlo)) / (quad_xhi - quad_xlo) * np.maximum(0, np.minimum(q[3], quad_yhi) - np.maximum(q[2], quad_ylo)) / (quad_yhi - quad_ylo)
        raw2_fraction = np.sum(fractions * overlaps, axis=1)
        fine_full = (q[0] <= quad_xlo) & (q[1] >= quad_xhi) & (q[2] <= quad_ylo) & (q[3] >= quad_yhi)
        fine_touch = (q[1] >= quad_xlo) & (q[0] <= quad_xhi) & (q[3] >= quad_ylo) & (q[2] <= quad_yhi)
        lower_counts = np.sum(quadrants * fine_full, axis=1)
        upper_counts = np.sum(quadrants * fine_touch, axis=1)
        lower_fraction = lower_counts / masses
        upper_fraction = upper_counts / masses
        X = np.column_stack((local, fractions, np.log1p(masses), coverage, np.full(n, q[1] - q[0]) / dx, np.full(n, q[3] - q[2]) / dy, np.log(np.maximum((q[1] - q[0]) / dx, 1e-12) / np.maximum((q[3] - q[2]) / dy, 1e-12)), raw2_fraction, lower_fraction, upper_fraction))
        if X.shape[1] != len(LOCAL_FEATURE_NAMES) or not np.isfinite(X).all():
            raise ValueError('Invalid local source-only features')
        blocks['X'].append(X.astype(np.float32))
        blocks['mass'].append(masses.astype(np.uint32))
        blocks['query_index'].append(np.full(n, qi, np.int32))
        blocks['source_id'].append(np.full(n, int(source_id), np.uint8))
        blocks['cell_id'].append((a * ny + b).astype(np.uint16))
        blocks['lower_fraction'].append(lower_fraction.astype(np.float64))
        blocks['upper_fraction'].append(upper_fraction.astype(np.float64))
        blocks['uniform_fraction'].append(raw2_fraction.astype(np.float64))
        blocks['coarse_uniform_fraction'].append(coverage.astype(np.float64))
        aggregates['U'][qi] += masses.sum()
        aggregates['S'][qi] += np.sum(masses * raw2_fraction)
        aggregates['Scoarse'][qi] += np.sum(masses * coverage)
        aggregates['L_fine'][qi] += np.sum(quadrants * fine_full)
        aggregates['U_fine'][qi] += np.sum(quadrants * fine_touch)
        aggregates['n_partial'][qi] += n
        aggregates['contrast_mass'][qi] += np.sum(masses * (fractions.max(axis=1) - fractions.min(axis=1)))
        aggregates['empty_quadrant_mass'][qi] += np.sum(masses * np.mean(quadrants == 0, axis=1))
    dtypes = dict(X=np.float32, mass=np.uint32, query_index=np.int32, source_id=np.uint8, cell_id=np.uint16, lower_fraction=np.float64, upper_fraction=np.float64, uniform_fraction=np.float64, coarse_uniform_fraction=np.float64)
    arrays = {name: np.concatenate(parts, axis=0) if parts else np.empty((0, len(LOCAL_FEATURE_NAMES)), np.float32) if name == 'X' else np.empty(0, dtypes[name]) for name, parts in blocks.items()}
    return (arrays, aggregates)

def describe_queries(rectangles, contexts, root):
    """Compute frozen source-only features from query geometry and summaries.

    `uniform_fraction` is the float64 raw2 area baseline for each partial coarse
    cell. `coarse_uniform_fraction` is its float64 whole-cell area fraction.
    No raw point arrays, query IDs, true answers or original ARM answer enter.
    Source-major, query-major record ordering matches offline cache preparation.
    """
    rectangles = np.asarray(rectangles, dtype=float)
    root = tuple(map(float, root))
    if rectangles.size == 0:
        rectangles = np.empty((0, 4), float)
    if rectangles.ndim != 2 or rectangles.shape[1] != 4 or (not np.isfinite(rectangles).all()) or (rectangles[:, 0] > rectangles[:, 1]).any() or (rectangles[:, 2] > rectangles[:, 3]).any():
        raise ValueError('Expected finite closed rectangles, including optional zero-area contacts')
    if len(root) != 4 or not np.isfinite(root).all() or (not root[0] < root[1]) or (not root[2] < root[3]):
        raise ValueError('Expected finite increasing root bounds')
    if not contexts:
        raise ValueError('No source summary contexts loaded')
    arrays_by_name = {name: [] for name in ('X', 'mass', 'query_index', 'source_id', 'cell_id', 'lower_fraction', 'upper_fraction', 'uniform_fraction', 'coarse_uniform_fraction')}
    sums = None
    for context in contexts:
        arr, agg = _source_describe(rectangles, context)
        for name, values in arr.items():
            arrays_by_name[name].append(values)
        if sums is None:
            sums = agg
        else:
            for name, values in agg.items():
                sums[name] += values
    arrays = {name: np.concatenate(parts, axis=0) for name, parts in arrays_by_name.items()}
    B = sums['U'] - sums['L']
    raw2b = np.clip(sums['S'] - sums['L'], 0, B)
    remaining = np.clip(sums['U'] - sums['S'], 0, B)
    qlevel = pd.DataFrame(dict(S=sums['S'], Scoarse=sums['Scoarse'], L=sums['L'], U=sums['U'], L_fine=sums['L_fine'], U_fine=sums['U_fine'], partial_mass=B, n_partial=sums['n_partial'].astype(int), n_full=sums['n_full'].astype(int)))
    qlevel['log1p_full_mass'] = np.log1p(sums['L'])
    qlevel['log1p_partial_mass'] = np.log1p(B)
    qlevel['log1p_raw2_boundary_estimate'] = np.log1p(raw2b)
    qlevel['log1p_raw2_remaining_mass'] = np.log1p(remaining)
    qlevel['raw2_boundary_fraction'] = raw2b / np.maximum(B, 1)
    qlevel['full_share'] = sums['L'] / np.maximum(sums['S'], 1)
    qlevel['log1p_n_partial'] = np.log1p(sums['n_partial'])
    qlevel['log1p_n_full'] = np.log1p(sums['n_full'])
    qlevel['query_width_root_fraction'] = (rectangles[:, 1] - rectangles[:, 0]) / (root[1] - root[0])
    qlevel['query_height_root_fraction'] = (rectangles[:, 3] - rectangles[:, 2]) / (root[3] - root[2])
    qlevel['raw2_partial_contrast_mean'] = sums['contrast_mass'] / np.maximum(B, 1)
    qlevel['raw2_partial_empty_quadrant_share'] = sums['empty_quadrant_mass'] / np.maximum(B, 1)
    qlevel['n_source_overlap'] = sums['n_source_overlap'].astype(int)
    qlevel['coarse_boundary_fraction'] = np.clip(sums['Scoarse'] - sums['L'], 0, B) / np.maximum(B, 1)
    if not np.isfinite(qlevel[AGG_FEATURE_NAMES].to_numpy(float)).all():
        raise ValueError('Nonfinite aggregate source-only features')
    arrays.update(full_mass=sums['L'], qlevel=qlevel, agg_X=qlevel[AGG_FEATURE_NAMES].to_numpy(np.float32))
    return arrays

def coarse_source_contexts(histogram_dir):
    """Load coarse sources only; prefer persisted12 .hist, allow .txt setup.

    Each compact .hist includes its actual24-byte header and float64 boundary
    and count arrays. No raw2 payload is opened or required by this pure path.
    """
    by_id = {}
    for path in Path(histogram_dir).glob('src_*'):
        match = re.fullmatch('src_(\\d+)\\.(hist|txt)', path.name)
        if match:
            source_id = int(match.group(1))
            if source_id not in by_id or path.suffix == '.hist':
                by_id[source_id] = path
    paths = sorted(by_id.items())
    if not paths:
        raise FileNotFoundError('No coarse source exports found in histogram_dir')
    contexts = []
    for source_id, path in paths:
        if not 0 <= source_id <= 255:
            raise ValueError('Prototype source_id must fit uint8')
        hist = _load_source_payload(path) if path.suffix == '.hist' else _load_source_export(path)
        _, bx, by, counts = hist
        for boundaries in (bx, by):
            if boundaries.ndim != 1 or len(boundaries) < 2 or (not np.isfinite(boundaries).all()) or (not (np.diff(boundaries) > 0).all()):
                raise ValueError('Invalid coarse source boundaries')
        if counts.shape != (len(bx) - 1, len(by) - 1) or not np.isfinite(counts).all() or (counts < 0).any() or (not np.equal(counts, np.floor(counts)).all()):
            raise ValueError('Expected exact coarse source integer counts')
        if counts.size > 65536:
            raise ValueError('Prototype coarse cell_id must fit uint16')
        if counts.max(initial=0) > np.iinfo(np.uint32).max:
            raise OverflowError('Source mass exceeds prototype uint32')
        contexts.append(dict(source_id=source_id, hist=hist, source_path=str(path), source_export_bytes=path.stat().st_size, source_payload_bytes=path.stat().st_size, source_payload_format='SLHIST01' if path.suffix == '.hist' else 'legacy text export', raw2_payload_bytes=0, raw2_loaded=False, numeric_array_bytes=sum((a.nbytes for a in hist[1:]))))
    return contexts

def describe_coarse_queries(rectangles, contexts, root):
    """Nine geometry/mass features using coarse sources only, with no raw2.

    A moment-only candidate may append its separately loaded compact metadata
    to this matrix. Ignoring raw2 feature columns in the other provider is not
    equivalent: this implementation never reads or requires quadrant arrays.
    No raw points, labels, original ARM answers or query IDs are arguments.
    """
    rectangles = np.asarray(rectangles, float)
    root = tuple(map(float, root))
    if rectangles.size == 0:
        rectangles = np.empty((0, 4), float)
    if rectangles.ndim != 2 or rectangles.shape[1] != 4 or (not np.isfinite(rectangles).all()) or (rectangles[:, 0] > rectangles[:, 1]).any() or (rectangles[:, 2] > rectangles[:, 3]).any():
        raise ValueError('Expected finite closed rectangles')
    if len(root) != 4 or not np.isfinite(root).all() or (not root[0] < root[1]) or (not root[2] < root[3]):
        raise ValueError('Expected increasing finite root bounds')
    if not contexts:
        raise ValueError('No coarse source summaries loaded')
    nq = len(rectangles)
    sums = {name: np.zeros(nq, float) for name in ('L', 'U', 'S', 'n_partial', 'n_full', 'n_source_overlap')}
    blocks = {name: [] for name in ('X', 'mass', 'query_index', 'source_id', 'cell_id', 'uniform_fraction')}
    for context in contexts:
        source_id = context['source_id']
        _, bx, by, counts = context['hist']
        nx, ny = counts.shape
        source_estimate = np.zeros(nq, float)
        for qi, q in enumerate(rectangles):
            if q[1] < bx[0] or q[0] > bx[-1] or q[3] < by[0] or (q[2] > by[-1]):
                continue
            i0 = max(0, int(np.searchsorted(bx, q[0], side='left') - 1))
            i1 = min(nx - 1, int(np.searchsorted(bx, q[1], side='right') - 1))
            j0 = max(0, int(np.searchsorted(by, q[2], side='left') - 1))
            j1 = min(ny - 1, int(np.searchsorted(by, q[3], side='right') - 1))
            if i1 < i0 or j1 < j0:
                continue
            a, b = np.meshgrid(np.arange(i0, i1 + 1), np.arange(j0, j1 + 1), indexing='ij')
            a = a.ravel()
            b = b.ravel()
            masses = counts[a, b]
            nonempty = masses > 0
            a = a[nonempty]
            b = b[nonempty]
            masses = masses[nonempty]
            if not len(a):
                continue
            xlo, xhi, ylo, yhi = (bx[a], bx[a + 1], by[b], by[b + 1])
            touched = (q[1] >= xlo) & (q[0] <= xhi) & (q[3] >= ylo) & (q[2] <= yhi)
            a = a[touched]
            b = b[touched]
            masses = masses[touched]
            xlo = xlo[touched]
            xhi = xhi[touched]
            ylo = ylo[touched]
            yhi = yhi[touched]
            if not len(a):
                continue
            sums['n_source_overlap'][qi] += 1
            full = (q[0] <= xlo) & (q[1] >= xhi) & (q[2] <= ylo) & (q[3] >= yhi)
            full_mass = float(masses[full].sum())
            sums['L'][qi] += full_mass
            sums['U'][qi] += full_mass
            source_estimate[qi] += full_mass
            sums['n_full'][qi] += int(full.sum())
            partial = ~full
            a = a[partial]
            b = b[partial]
            masses = masses[partial]
            xlo = xlo[partial]
            xhi = xhi[partial]
            ylo = ylo[partial]
            yhi = yhi[partial]
            if not len(a):
                continue
            n = len(a)
            dx = xhi - xlo
            dy = yhi - ylo
            local = np.column_stack((np.clip((q[0] - xlo) / dx, 0, 1), np.clip((q[1] - xlo) / dx, 0, 1), np.clip((q[2] - ylo) / dy, 0, 1), np.clip((q[3] - ylo) / dy, 0, 1)))
            coverage = (local[:, 1] - local[:, 0]) * (local[:, 3] - local[:, 2])
            X = np.column_stack((local, np.log1p(masses), coverage, np.full(n, q[1] - q[0]) / dx, np.full(n, q[3] - q[2]) / dy, np.log(np.maximum((q[1] - q[0]) / dx, 1e-12) / np.maximum((q[3] - q[2]) / dy, 1e-12))))
            if X.shape[1] != len(COARSE_FEATURE_NAMES) or not np.isfinite(X).all():
                raise ValueError('Invalid coarse-only features')
            blocks['X'].append(X.astype(np.float32))
            blocks['mass'].append(masses.astype(np.uint32))
            blocks['query_index'].append(np.full(n, qi, np.int32))
            blocks['source_id'].append(np.full(n, int(source_id), np.uint8))
            blocks['cell_id'].append((a * ny + b).astype(np.uint16))
            blocks['uniform_fraction'].append(coverage.astype(np.float64))
            sums['U'][qi] += masses.sum()
            source_estimate[qi] += np.sum(masses * coverage)
            sums['n_partial'][qi] += n
        sums['S'] += source_estimate
    dtypes = dict(X=np.float32, mass=np.uint32, query_index=np.int32, source_id=np.uint8, cell_id=np.uint16, uniform_fraction=np.float64)
    arrays = {name: np.concatenate(parts, axis=0) if parts else np.empty((0, len(COARSE_FEATURE_NAMES)), np.float32) if name == 'X' else np.empty(0, dtypes[name]) for name, parts in blocks.items()}
    nrecords = len(arrays['mass'])
    arrays.update(full_mass=sums['L'], lower_fraction=np.zeros(nrecords, float), upper_fraction=np.ones(nrecords, float), qlevel=pd.DataFrame(dict(S=sums['S'], Scoarse=sums['S'], L=sums['L'], U=sums['U'], partial_mass=sums['U'] - sums['L'], n_partial=sums['n_partial'].astype(int), n_full=sums['n_full'].astype(int), n_source_overlap=sums['n_source_overlap'].astype(int))))
    return arrays
'''

In [ ]:
# Embedded helper: moment_metadata.py
EMBEDDED19['moment_metadata.py'] = r'''"""Five quantized source-cell moments; no query labels or raw2 inputs.

Construction preserves the original arithmetic-floor source-cell ownership.
Only the saved five-byte values are decoded into model features. Unquantized
sums can add on an identical source grid before encoding. Different source
grids are retained as a source union; this is not a bounded-size heterogeneous
merge algorithm, and quantized moments are not exact sufficient statistics.
"""
from pathlib import Path
import argparse
import json
import struct
import time
import numpy as np
from .prepare_data import iter_raw, load_hist, sha, LOCAL_FEATURE_NAMES
MAGIC = b'SLMOM001'
HEADER = struct.Struct('<8sBBHIII4d8x')
MOMENT_NAMES = ['source_mean_x', 'source_mean_y', 'source_var_x', 'source_var_y', 'source_cov_xy']
COARSE_INDICES = [0, 1, 2, 3, 8, 9, 10, 11, 12]
MOMENT_ONLY_NAMES = [LOCAL_FEATURE_NAMES[i] for i in COARSE_INDICES] + MOMENT_NAMES
AUGMENTED_NAMES = LOCAL_FEATURE_NAMES + MOMENT_NAMES

def _histogram(hist):
    _, bx, by, counts = hist
    bx = np.asarray(bx, float)
    by = np.asarray(by, float)
    counts = np.asarray(counts)
    if bx.ndim != 1 or by.ndim != 1 or len(bx) < 2 or (len(by) < 2) or (not np.isfinite(bx).all()) or (not np.isfinite(by).all()) or (not (np.diff(bx) > 0).all()) or (not (np.diff(by) > 0).all()) or (counts.shape != (len(bx) - 1, len(by) - 1)) or (not np.isfinite(counts).all()) or (counts < 0).any() or (not np.equal(counts, np.floor(counts)).all()):
        raise ValueError('Expected exact nonnegative source histogram with increasing boundaries')
    nx, ny = counts.shape
    if not np.allclose(np.diff(bx), (bx[-1] - bx[0]) / nx, rtol=1e-11, atol=0) or not np.allclose(np.diff(by), (by[-1] - by[0]) / ny, rtol=1e-11, atol=0):
        raise ValueError('Arithmetic-floor ownership requires a regular source grid')
    return (bx, by, counts, nx, ny)

def accumulate_source_moments(chunks, hist):
    """Return construction-only count and additive normalized first/second sums."""
    bx, by, coarse, nx, ny = _histogram(hist)
    counts = np.zeros(nx * ny, np.uint64)
    sums = np.zeros((nx * ny, 5), np.float64)
    rows = 0
    clipped = 0
    if isinstance(chunks, np.ndarray):
        chunks = [chunks]
    for points in chunks:
        points = np.asarray(points, float)
        if points.ndim != 2 or points.shape[1] != 2 or (not np.isfinite(points).all()):
            raise ValueError('Expected finite raw x/y points')
        if ((points[:, 0] < bx[0]) | (points[:, 0] >= bx[-1]) | (points[:, 1] < by[0]) | (points[:, 1] >= by[-1])).any():
            raise ValueError('Raw point outside original half-open source domain')
        ix = ((points[:, 0] - bx[0]) / ((bx[-1] - bx[0]) / nx)).astype(np.int64)
        iy = ((points[:, 1] - by[0]) / ((by[-1] - by[0]) / ny)).astype(np.int64)
        if (ix < 0).any() or (ix >= nx).any() or (iy < 0).any() or (iy >= ny).any():
            raise ValueError('Original arithmetic source ownership produced invalid cell')
        cell = ix * ny + iy
        u = (points[:, 0] - bx[ix]) / np.diff(bx)[ix]
        v = (points[:, 1] - by[iy]) / np.diff(by)[iy]
        if ((u < -1e-10) | (u > 1 + 1e-10) | (v < -1e-10) | (v > 1 + 1e-10)).any():
            raise ValueError('Assigned point lies beyond source-cell roundoff tolerance')
        clipped += int(((u < 0) | (u > 1) | (v < 0) | (v > 1)).sum())
        u = np.clip(u, 0, 1)
        v = np.clip(v, 0, 1)
        counts += np.bincount(cell, minlength=nx * ny).astype(np.uint64)
        for k, values in enumerate((u, v, u * u, v * v, u * v)):
            sums[:, k] += np.bincount(cell, weights=values, minlength=nx * ny)
        rows += len(points)
    if not np.array_equal(counts.reshape(nx, ny), coarse):
        raise ValueError('Moment construction does not reproduce original exact coarse counts')
    return dict(counts=counts, sums=sums, rows=rows, numerical_local_coordinate_clips=clipped)

def moments_from_sufficient(counts, sums):
    counts = np.asarray(counts)
    sums = np.asarray(sums, np.float64)
    if counts.ndim != 1 or sums.shape != (len(counts), 5) or (not np.isfinite(counts).all()) or (not np.isfinite(sums).all()) or (counts < 0).any() or (not np.equal(counts, np.floor(counts)).all()):
        raise ValueError('Invalid count/sufficient-statistic arrays')
    avg = np.divide(sums, counts[:, None], out=np.zeros_like(sums), where=counts[:, None] > 0)
    moments = np.column_stack((avg[:, 0], avg[:, 1], avg[:, 2] - avg[:, 0] ** 2, avg[:, 3] - avg[:, 1] ** 2, avg[:, 4] - avg[:, 0] * avg[:, 1]))
    if (moments[:, 2:4] < -1e-10).any():
        raise ValueError('Negative variance beyond roundoff')
    moments[:, 2:4] = np.maximum(moments[:, 2:4], 0)
    return moments

def encode_moments(moments):
    values = np.asarray(moments, float)
    if values.ndim != 2 or values.shape[1] != 5 or (not np.isfinite(values).all()):
        raise ValueError('Expected finite N by 5 moments')
    tolerance = 1e-10
    if ((values[:, :2] < -tolerance) | (values[:, :2] > 1 + tolerance)).any() or ((values[:, 2:4] < -tolerance) | (values[:, 2:4] > 0.25 + tolerance)).any() or (np.abs(values[:, 4]) > 0.25 + tolerance).any():
        raise ValueError('Normalized moments outside supported physical ranges')
    encoded = np.empty(values.shape, np.uint8)
    encoded[:, :2] = np.rint(255 * np.clip(values[:, :2], 0, 1)).astype(np.uint8)
    encoded[:, 2:4] = np.rint(1020 * np.clip(values[:, 2:4], 0, 0.25)).astype(np.uint8)
    covariance = np.rint(508 * np.clip(values[:, 4], -0.25, 0.25)).astype(np.int8)
    encoded[:, 4] = covariance.view(np.uint8)
    return encoded

def decode_moments(encoded):
    encoded = np.asarray(encoded)
    if encoded.ndim != 2 or encoded.shape[1] != 5 or encoded.dtype != np.uint8:
        raise ValueError('Expected N by 5 uint8 moment bytes')
    covariance = np.ascontiguousarray(encoded[:, 4]).view(np.int8)
    if (covariance == -128).any():
        raise ValueError('Reserved covariance byte -128')
    return np.column_stack((encoded[:, :2].astype(float) / 255, encoded[:, 2:4].astype(float) / 1020, covariance.astype(float) / 508)).astype(np.float32)

def save_moments(path, moments, source_id, root, nx, ny):
    root = tuple(map(float, root))
    nx = int(nx)
    ny = int(ny)
    source_id = int(source_id)
    if len(root) != 4 or not np.isfinite(root).all() or (not root[0] < root[1]) or (not root[2] < root[3]) or (nx < 1) or (ny < 1) or (not 0 <= source_id < 2 ** 32):
        raise ValueError('Invalid moment geometry/source identity')
    encoded = encode_moments(moments)
    if len(encoded) != nx * ny:
        raise ValueError('Moments do not match source dimensions')
    payload = HEADER.pack(MAGIC, 1, 5, 0, source_id, nx, ny, *root) + encoded.tobytes(order='C')
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    return len(payload)

def load_moments(path):
    payload = Path(path).read_bytes()
    if len(payload) < HEADER.size:
        raise ValueError('Truncated moment header')
    magic, version, components, flags, source_id, nx, ny, *root = HEADER.unpack_from(payload)
    if magic != MAGIC or version != 1 or components != 5 or flags or (nx < 1) or (ny < 1) or (payload[56:64] != b'\x00' * 8) or (len(payload) != HEADER.size + 5 * nx * ny):
        raise ValueError('Invalid moment header or payload size')
    if not np.isfinite(root).all() or not root[0] < root[1] or (not root[2] < root[3]):
        raise ValueError('Invalid moment root')
    encoded = np.frombuffer(payload, np.uint8, offset=HEADER.size).reshape(nx * ny, 5).copy()
    decoded = decode_moments(encoded)
    encoded.setflags(write=False)
    decoded.setflags(write=False)
    return dict(source_id=source_id, root=tuple(root), nx=nx, ny=ny, encoded=encoded, decoded=decoded, payload_bytes=len(payload), path=str(Path(path)))

def load_moments_contexts(files):
    if isinstance(files, (str, Path)):
        path = Path(files)
        files = sorted(path.glob('src_*-moments.bin')) if path.is_dir() else [path]
    result = {}
    for path in files:
        item = load_moments(path)
        if item['source_id'] in result:
            raise ValueError('Duplicate source moment identity')
        result[item['source_id']] = item
    if not result:
        raise FileNotFoundError('No moment payloads')
    return result

def augment_records(described, decoded_by_source):
    """Feature-only source/cell join; labels and query outcomes are never read."""
    X = np.asarray(described['X'], np.float32)
    source_ids = np.asarray(described['source_id'])
    cell_ids = np.asarray(described['cell_id'])
    if X.ndim != 2 or X.shape[1] not in (9, 16) or source_ids.ndim != 1 or (cell_ids.ndim != 1) or (len(source_ids) != len(X)) or (len(cell_ids) != len(X)):
        raise ValueError('Expected original16 or coarse9 feature records with source/cell keys')
    if not np.isfinite(X).all() or not np.isfinite(source_ids).all() or (not np.isfinite(cell_ids).all()) or (source_ids < 0).any() or (source_ids >= 2 ** 32).any() or (not np.equal(source_ids, np.floor(source_ids)).all()) or (cell_ids < 0).any() or (not np.equal(cell_ids, np.floor(cell_ids)).all()):
        raise ValueError('Invalid source-only records')
    moments = np.empty((len(X), 5), np.float32)
    for source_id in np.unique(source_ids):
        key = int(source_id)
        if key not in decoded_by_source:
            raise KeyError('Missing moment source ' + str(key))
        item = decoded_by_source[key]
        decoded = np.asarray(item['decoded'] if isinstance(item, dict) else item, np.float32)
        if decoded.ndim != 2 or decoded.shape[1] != 5 or (not np.isfinite(decoded).all()):
            raise ValueError('Invalid decoded moments')
        if not np.array_equal(decoded, decode_moments(encode_moments(decoded))):
            raise ValueError('Moment features must use the saved quantized codec values')
        selected = source_ids == source_id
        ids = cell_ids[selected].astype(np.int64)
        if ids.max(initial=0) >= len(decoded):
            raise ValueError('Moment cell ID out of range')
        moments[selected] = decoded[ids]
    coarse = X[:, COARSE_INDICES] if X.shape[1] == 16 else X
    result = dict(X_moment_only=np.column_stack((coarse, moments)).astype(np.float32), moment_features=moments, coarse_uniform_fraction=coarse[:, 5].copy())
    if X.shape[1] == 16:
        result['X_aug'] = np.column_stack((X, moments)).astype(np.float32)
    return result

def build_moments(contexts, raw_files, destination, expected_hashes=None, chunk_size=100000):
    """Scan raw source files once, verify exact masses, persist and reload codec."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    manifest_path = destination / 'moment_manifest.json'
    if manifest_path.exists():
        raise FileExistsError('Moment destination is preserved; use a new directory')
    records = []
    started = time.perf_counter()
    for context in contexts:
        sid = int(context['source_id'])
        hist = context['hist']
        bx, by, coarse, nx, ny = _histogram(hist)
        raw_path = Path(raw_files[sid])
        before = sha(raw_path)
        if expected_hashes is not None and before != expected_hashes[sid]:
            raise ValueError('Raw identity changed before moment scan')
        tick = time.perf_counter()
        stats = accumulate_source_moments(iter_raw(raw_path, chunk_size), hist)
        after = sha(raw_path)
        if before != after:
            raise ValueError('Raw identity changed during moment scan')
        moments = moments_from_sufficient(stats['counts'], stats['sums'])
        path = destination / f'src_{sid}-moments.bin'
        size = save_moments(path, moments, sid, (bx[0], bx[-1], by[0], by[-1]), nx, ny)
        loaded = load_moments(path)
        if not np.array_equal(loaded['encoded'], encode_moments(moments)):
            raise ValueError('Moment codec round trip failed')
        nonempty = stats['counts'] > 0
        w = stats['counts'][nonempty].astype(float)
        d = loaded['decoded'][nonempty].astype(float)
        delta = d - moments[nonempty]
        variance = d[:, 2:4]
        cov = d[:, 4]
        eigen_gap = np.sqrt((variance[:, 0] - variance[:, 1]) ** 2 + 4 * cov ** 2)
        major = (variance.sum(axis=1) + eigen_gap) / 2
        minor = (variance.sum(axis=1) - eigen_gap) / 2
        anisotropy = (major - minor) / np.maximum(major + minor, 1e-12)
        row = dict(source_id=sid, raw_path=str(raw_path), raw_sha256_before=before, raw_sha256_after=after, raw_rows=stats['rows'], coarse_cells=nx * ny, nonempty_cells=int(nonempty.sum()), coarse_mass_verified=True, numerical_local_coordinate_clips=stats['numerical_local_coordinate_clips'], payload_path=str(path), payload_bytes=size, payload_sha256=sha(path), raw_scan_and_encoding_seconds=time.perf_counter() - tick, decoded_mass_weighted_mean_x=float(np.average(d[:, 0], weights=w)), decoded_mass_weighted_mean_y=float(np.average(d[:, 1], weights=w)), decoded_mass_weighted_var_x=float(np.average(d[:, 2], weights=w)), decoded_mass_weighted_var_y=float(np.average(d[:, 3], weights=w)), decoded_mass_weighted_cov_xy=float(np.average(d[:, 4], weights=w)), mass_share_variance_x_below_half_uniform=float(np.average(d[:, 2] < 1 / 24, weights=w)), mass_share_variance_y_below_half_uniform=float(np.average(d[:, 3] < 1 / 24, weights=w)), decoded_mass_weighted_anisotropy=float(np.average(anisotropy, weights=w)), quantized_non_psd_cells=int((minor < -1e-12).sum()), max_abs_quantization_error=[float(x) for x in np.max(np.abs(delta), axis=0)], construction_numeric_accumulators_bytes=stats['counts'].nbytes + stats['sums'].nbytes, construction_memory_scope='count + five float64 sum buffers only; excludes raw chunk and arithmetic temporaries; not peak RSS')
        records.append(row)
        print(f"Moment source {sid}: {stats['rows']} rows, {size} bytes, {row['raw_scan_and_encoding_seconds']:.2f}s", flush=True)
    manifest = dict(status='complete', version=1, feature_names=MOMENT_NAMES, encoding='64-byte header + 5 bytes per source coarse cell', rounding='means uint8 rint(255*mean); variance uint8 rint(1020*variance); covariance int8 rint(508*covariance), stored uint8 view', empty_cells='all five bytes zero; empty cells never create partial-cell model records', quantization_not_confidence_bounds=True, uniform_variance_reference=1 / 12, merge_scope='unquantized sums add on an identical source grid before encoding; different source grids retained separately; no bounded heterogeneous merge claim', training=False, query_labels_read=False, raw2_read=False, total_payload_bytes=sum((row['payload_bytes'] for row in records)), sources=records, total_seconds=time.perf_counter() - started)
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return load_moments_contexts([row['payload_path'] for row in records])
'''

In [ ]:
# Embedded helper: analytic_moments_fast.py
EMBEDDED19['analytic_moments_fast.py'] = r'''"""Fixed moment-matched source-cell reference estimators.

Both candidates consume exactly the same decoded five-byte source-cell moment
payload and query intersections. They never receive raw points, labels, raw2
counts, fitted models or the candidate ARM answer. The reference is a structural
distribution approximation, not a confidence interval or an exact answer.
Beta moment matching and Gaussian copulas are classical components; no novelty
or effectiveness claim follows from this implementation alone.
"""
import numpy as np
from scipy.special import betainc, ndtri
VARIANTS = ('beta_independent', 'beta_gaussian_copula')
MOMENT_COLUMNS = ('mean_x', 'mean_y', 'variance_x', 'variance_y', 'covariance_xy')
EPSILON = 1e-06
RHO_LIMIT = 0.995
QUADRATURE_ORDER = 16
CDF_BLOCK_SIZE = 2048
_GL_NODES, _GL_WEIGHTS = np.polynomial.legendre.leggauss(QUADRATURE_ORDER)

def _validated_inputs(rectangles, moments):
    rectangles = np.asarray(rectangles, float)
    moments = np.asarray(moments, float)
    if rectangles.ndim != 2 or rectangles.shape[1] != 4 or moments.shape != (len(rectangles), 5):
        raise ValueError('Expected Nx4 local rectangles and Nx5 decoded moments')
    if not np.isfinite(rectangles).all() or not np.isfinite(moments).all():
        raise ValueError('Nonfinite local geometry or moments')
    if ((rectangles < 0) | (rectangles > 1)).any() or (rectangles[:, 0] > rectangles[:, 1]).any() or (rectangles[:, 2] > rectangles[:, 3]).any():
        raise ValueError('Query intersections must be closed subrectangles of[0,1]^2')
    if ((moments[:, :2] < 0) | (moments[:, :2] > 1)).any() or (moments[:, 2:4] < 0).any():
        raise ValueError('Decoded means must lie in[0,1]; variances must be nonnegative')
    return (rectangles, moments)

def _marginal_limits(lo, hi, mean, variance):
    """Return F(lo-) and F(hi), including exact Dirac-axis handling.

    Quantization can yield an inconsistent mean/variance pair. The fixed policy
    clamps nondegenerate means to[eps,1-eps] and variances to the largest valid
    beta moment, multiplied by1-eps. It never uses labels to choose a clamp.
    A decoded zero variance is represented by a point mass at the decoded mean.
    """
    point = variance == 0
    lower = np.empty(len(lo), float)
    upper = np.empty(len(lo), float)
    lower[point] = (lo[point] > mean[point]).astype(float)
    upper[point] = (hi[point] >= mean[point]).astype(float)
    continuous = ~point
    if continuous.any():
        m = np.clip(mean[continuous], EPSILON, 1 - EPSILON)
        v = np.minimum(variance[continuous], m * (1 - m) * (1 - EPSILON))
        k = m * (1 - m) / v - 1.0
        alpha = m * k
        beta = (1 - m) * k
        lower[continuous] = betainc(alpha, beta, lo[continuous])
        upper[continuous] = betainc(alpha, beta, hi[continuous])
    if not np.isfinite(np.r_[lower, upper]).all():
        raise ValueError('Moment-matched beta produced a nonfinite marginal CDF')
    return (np.clip(lower, 0.0, 1.0), np.clip(upper, 0.0, 1.0))

def gaussian_copula_cdf(u, v, rho):
    """Deterministic Gaussian-copula CDF using a fixed16-point integral.

    C(u,v;r)=uv+integral[0,r] phi2(ndtri(u),ndtri(v);t)dt.
    The substitution t=sin(theta) removes1/sqrt(1-t²), so the fixed16-point
    rule integrates over[0,asin(r)] and stays accurate near the rho clamp.
    Endpoint marginals are handled exactly before inverse normal evaluation.
    No scipy.stats multivariate CDF or randomized numerical integration is used.
    Interior CDF rows are processed in blocks of at most2048 by16 quadrature
    nodes. This changes array execution only, not the fixed numerical rule.
    """
    u, v, rho = np.broadcast_arrays(np.asarray(u, float), np.asarray(v, float), np.asarray(rho, float))
    original_shape = u.shape
    u = u.reshape(-1)
    v = v.reshape(-1)
    rho = rho.reshape(-1)
    if not np.isfinite(np.r_[u, v, rho]).all() or ((u < 0) | (u > 1) | (v < 0) | (v > 1)).any():
        raise ValueError('Copula probabilities must be finite in[0,1]')
    rho = np.clip(rho, -RHO_LIMIT, RHO_LIMIT)
    result = u * v
    interior = (u > 0) & (u < 1) & (v > 0) & (v < 1) & (rho != 0)
    if interior.any():
        h = ndtri(u[interior])
        k = ndtri(v[interior])
        angle = np.arcsin(rho[interior])
        integral = np.empty(len(h), float)
        for start in range(0, len(h), CDF_BLOCK_SIZE):
            stop = min(start + CDF_BLOCK_SIZE, len(h))
            hs = h[start:stop, None]
            ks = k[start:stop, None]
            t = np.sin(0.5 * angle[start:stop, None] * (_GL_NODES[None, :] + 1.0))
            one_minus = 1.0 - t * t
            exponent = -0.5 * ((hs - ks * t) ** 2 / one_minus + ks * ks)
            integral[start:stop] = np.sum(np.exp(exponent) * _GL_WEIGHTS[None, :], axis=1)
        result[interior] += 0.5 * angle * integral / (2.0 * np.pi)
    upper = np.minimum(u, v)
    lower = np.minimum(np.maximum(u + v - 1.0, 0.0), upper)
    result = np.maximum(np.minimum(result, upper), lower)
    return result.reshape(original_shape)

def boundary_probability(rectangles, moments, variant='beta_independent'):
    """Approximate local rectangle probability from decoded five moments."""
    if variant not in VARIANTS:
        raise ValueError('Unknown fixed analytic moment variant')
    rectangles, moments = _validated_inputs(rectangles, moments)
    if not len(rectangles):
        return np.empty(0, float)
    lx, ux = _marginal_limits(rectangles[:, 0], rectangles[:, 1], moments[:, 0], moments[:, 2])
    ly, uy = _marginal_limits(rectangles[:, 2], rectangles[:, 3], moments[:, 1], moments[:, 3])
    if variant == 'beta_independent':
        probability = (ux - lx) * (uy - ly)
    else:
        denominator = np.sqrt(moments[:, 2] * moments[:, 3])
        rho = np.divide(moments[:, 4], denominator, out=np.zeros(len(moments)), where=denominator > 0)
        rho = np.clip(rho, -RHO_LIMIT, RHO_LIMIT)
        corners = gaussian_copula_cdf(np.concatenate((ux, lx, ux, lx)), np.concatenate((uy, uy, ly, ly)), np.tile(rho, 4)).reshape(4, len(rho))
        probability = corners[0] - corners[1] - corners[2] + corners[3]
    if not np.isfinite(probability).all():
        raise ValueError('Nonfinite analytic moment probability')
    return np.clip(probability, 0.0, 1.0)

def estimate_reference(rectangles, moments, mass, query_index, L_full, variant='beta_independent'):
    """Aggregate exact fully contained mass plus estimated boundary mass.

    Only source/query metadata is an argument. query_index selects the output
    query for each partial-cell row; it is not an identity feature or a label.
    """
    rectangles, moments = _validated_inputs(rectangles, moments)
    mass = np.asarray(mass, float)
    query_index = np.asarray(query_index)
    full = np.asarray(L_full, float)
    if mass.shape != (len(rectangles),) or query_index.shape != (len(rectangles),) or full.ndim != 1 or (not np.isfinite(mass).all()) or (not np.isfinite(full).all()) or (mass < 0).any() or (full < 0).any():
        raise ValueError('Invalid partial masses, query indices or fully-contained masses')
    if not np.issubdtype(query_index.dtype, np.integer) or (query_index < 0).any() or (query_index >= len(full)).any():
        raise ValueError('Partial row query index outside the reference-count output')
    p = boundary_probability(rectangles, moments, variant)
    return full + np.bincount(query_index, weights=mass * p, minlength=len(full))

def protocol():
    return dict(variants=list(VARIANTS), moment_columns=list(MOMENT_COLUMNS), mean_clamp_epsilon=EPSILON, variance_policy='zero=>decoded point mass; positive=>min(v,m*(1-m)*(1-eps)) after mean clamp', covariance_policy='rho=cov/sqrt(vx*vy), clipped to[-.995,.995]; zero variance=>rho0', gaussian_copula_integration='fixed16-point Gauss-Legendre Plackett integral with t=sin(theta) substitution; exact probability endpoints', quadrature_order=QUADRATURE_ORDER, probability_clipping='CDF Fréchet bounds, then final rectangle[0,1]', execution='blocked2048 CDF rows x16 quadrature nodes; all4 rectangle corners in one CDF call', source_model='moment-matched beta marginals; independent or fixed Gaussian copula', quantization='caller supplies round-trip decoded mu/255, variance/1020, covariance/508 five-byte moments', training=False, labels_at_inference=False, raw2_at_inference=False, H_at_reference_inference=False, confidence_or_accuracy_guarantee=False, novelty_claim=False)
'''

In [ ]:
# Embedded helper: resident_critic.py
EMBEDDED19['resident_critic.py'] = r'''"""Resident, geometry-indexed execution of the frozen moment critic.

This changes execution only: normalized intersections retain the frozen
float32 feature contract; beta clamps, Pearson plug-in correlation and the
16-point deterministic quadrature are unchanged. No raw points, labels,
candidate answers or saved per-query values are retained.
"""
import time
import numpy as np
from scipy.special import betainc, ndtri
_EPSILON = 1e-06
_RHO_LIMIT = 0.995
_GL_NODES, _GL_WEIGHTS = np.polynomial.legendre.leggauss(16)

class ResidentCritic:

    def __init__(self, coarse_contexts, moments, root=None, directory_side=64, cache_quadrature=False):
        tick = time.perf_counter()
        if not coarse_contexts or int(directory_side) != directory_side or directory_side < 1:
            raise ValueError('Nonempty source summaries and a positive directory side are required')
        bounds = []
        mass = []
        values = []
        source_keys = []
        extents = []
        self.cache_quadrature = bool(cache_quadrature)
        seen = set()
        for context in sorted(coarse_contexts, key=lambda item: int(item['source_id'])):
            sid = int(context['source_id'])
            if sid in seen or sid not in moments:
                raise ValueError('Duplicate source or missing moment payload')
            seen.add(sid)
            _, bx, by, counts = context['hist']
            bx = np.asarray(bx, float)
            by = np.asarray(by, float)
            counts = np.asarray(counts)
            if counts.shape != (len(bx) - 1, len(by) - 1) or not np.isfinite(counts).all() or (counts < 0).any() or (not np.equal(counts, np.floor(counts)).all()):
                raise ValueError('Source summaries require exact nonnegative integer counts')
            if not (np.diff(bx) > 0).all() or not (np.diff(by) > 0).all():
                raise ValueError('Invalid source cell geometry')
            entry = moments[sid]
            decoded = np.asarray(entry['decoded'] if isinstance(entry, dict) else entry, np.float32)
            if decoded.shape != (counts.size, 5) or not np.isfinite(decoded).all():
                raise ValueError('Decoded moment cells do not match source summary')
            if isinstance(entry, dict):
                expected = (float(bx[0]), float(bx[-1]), float(by[0]), float(by[-1]))
                if tuple(entry['root']) != expected or entry['nx'] != counts.shape[0] or entry['ny'] != counts.shape[1]:
                    raise ValueError('Moment and count source geometry differ')
            nx, ny = counts.shape
            extents.append((float(bx[0]), float(bx[-1]), float(by[0]), float(by[-1])))
            cells = np.flatnonzero(counts.ravel() > 0)
            ix, iy = (cells // ny, cells % ny)
            bounds.append(np.column_stack((bx[ix], bx[ix + 1], by[iy], by[iy + 1])))
            mass.append(counts.ravel()[cells].astype(np.uint64))
            values.append(decoded[cells].astype(float))
            source_keys.append(np.column_stack((np.full(len(cells), sid, np.uint32), cells.astype(np.uint32))))
        self.bounds = np.concatenate(bounds)
        self.mass = np.concatenate(mass)
        self.moments = np.concatenate(values)
        self.source_keys = np.concatenate(source_keys)
        if ((self.moments[:, :2] < 0) | (self.moments[:, :2] > 1)).any() or (self.moments[:, 2:4] < 0).any():
            raise ValueError('Decoded moments lie outside their permitted ranges')
        if root is None:
            root = (min((e[0] for e in extents)), max((e[1] for e in extents)), min((e[2] for e in extents)), max((e[3] for e in extents)))
        self.root = tuple(map(float, root))
        self.side = int(directory_side)
        if len(self.root) != 4 or not np.isfinite(self.root).all() or (not self.root[0] < self.root[1]) or (not self.root[2] < self.root[3]) or (self.bounds[:, 0] < self.root[0]).any() or (self.bounds[:, 1] > self.root[1]).any() or (self.bounds[:, 2] < self.root[2]).any() or (self.bounds[:, 3] > self.root[3]).any():
            raise ValueError('Directory root must contain every source cell')
        self._build_directory()
        self._cache_distribution_parameters()
        self.resident_numpy_bytes = int(sum((value.nbytes for value in vars(self).values() if isinstance(value, np.ndarray))))
        self.initialization_seconds = time.perf_counter() - tick

    def _build_directory(self):
        side = self.side
        b = self.bounds
        self.directory_x = np.linspace(self.root[0], self.root[1], side + 1)
        self.directory_y = np.linspace(self.root[2], self.root[3], side + 1)
        sx = np.clip(np.searchsorted(self.directory_x, b[:, 0], 'right') - 1, 0, side - 1)
        ex = np.clip(np.searchsorted(self.directory_x, b[:, 1], 'left') - 1, 0, side - 1)
        sy = np.clip(np.searchsorted(self.directory_y, b[:, 2], 'right') - 1, 0, side - 1)
        ey = np.clip(np.searchsorted(self.directory_y, b[:, 3], 'left') - 1, 0, side - 1)
        widths = ey - sy + 1
        lengths = (ex - sx + 1) * widths
        starts = np.r_[0, np.cumsum(lengths, dtype=np.int64)]
        n = int(starts[-1])
        ids = np.repeat(np.arange(len(b), dtype=np.uint32), lengths)
        relative = np.arange(n, dtype=np.int64) - np.repeat(starts[:-1], lengths)
        wy = np.repeat(widths, lengths)
        bins = (np.repeat(sx, lengths) + relative // wy) * side + np.repeat(sy, lengths) + relative % wy
        order = np.argsort(bins, kind='stable')
        counts = np.bincount(bins, minlength=side * side)
        self.directory_offsets = np.r_[0, np.cumsum(counts, dtype=np.int64)]
        self.directory_ids = ids[order]
        self.directory_construction_named_nbytes = int(sum((a.nbytes for a in (sx, ex, sy, ey, widths, lengths, starts, ids, relative, wy, bins, order, counts))))

    def _cache_distribution_parameters(self):
        m = self.moments[:, :2]
        v = self.moments[:, 2:4]
        self.point_axis = v == 0
        mu = np.clip(m, _EPSILON, 1 - _EPSILON)
        variance = np.minimum(v, mu * (1 - mu) * (1 - _EPSILON))
        k = np.divide(mu * (1 - mu), variance, out=np.ones_like(variance), where=variance > 0) - 1.0
        self.alpha = mu * k
        self.beta = (1 - mu) * k
        denominator = np.sqrt(v[:, 0] * v[:, 1])
        self.rho = np.clip(np.divide(self.moments[:, 4], denominator, out=np.zeros(len(v)), where=denominator > 0), -_RHO_LIMIT, _RHO_LIMIT)
        self.angle = np.arcsin(self.rho)
        if self.cache_quadrature:
            self.quadrature_t = np.sin(0.5 * self.angle[:, None] * (_GL_NODES[None, :] + 1.0))
            self.quadrature_one_minus = 1.0 - self.quadrature_t * self.quadrature_t

    def _candidate_ids(self, q):
        side = self.side
        i0 = max(0, min(side - 1, int(np.searchsorted(self.directory_x, q[0], 'left') - 1)))
        i1 = max(0, min(side - 1, int(np.searchsorted(self.directory_x, q[1], 'right') - 1)))
        j0 = max(0, min(side - 1, int(np.searchsorted(self.directory_y, q[2], 'left') - 1)))
        j1 = max(0, min(side - 1, int(np.searchsorted(self.directory_y, q[3], 'right') - 1)))
        pieces = []
        for i in range(i0, i1 + 1):
            for j in range(j0, j1 + 1):
                lo, hi = self.directory_offsets[i * side + j:i * side + j + 2]
                if hi > lo:
                    pieces.append(self.directory_ids[lo:hi])
        return np.unique(np.concatenate(pieces)) if pieces else np.empty(0, np.uint32)

    def _marginal(self, lo, hi, ids, axis):
        point = self.point_axis[ids, axis]
        lower = np.empty(len(ids))
        upper = np.empty(len(ids))
        lower[point] = (lo[point] > self.moments[ids[point], axis]).astype(float)
        upper[point] = (hi[point] >= self.moments[ids[point], axis]).astype(float)
        continuous = ~point
        if continuous.any():
            selected = ids[continuous]
            lower[continuous] = betainc(self.alpha[selected, axis], self.beta[selected, axis], lo[continuous])
            upper[continuous] = betainc(self.alpha[selected, axis], self.beta[selected, axis], hi[continuous])
        return (np.clip(lower, 0.0, 1.0), np.clip(upper, 0.0, 1.0))

    def _cdf(self, u, v, ids):
        result = u * v
        interior = (u > 0) & (u < 1) & (v > 0) & (v < 1) & (self.rho[ids] != 0)
        if interior.any():
            selected = ids[interior]
            h = ndtri(u[interior])[:, None]
            k = ndtri(v[interior])[:, None]
            if self.cache_quadrature:
                t = self.quadrature_t[selected]
                om = self.quadrature_one_minus[selected]
            else:
                t = np.sin(0.5 * self.angle[selected, None] * (_GL_NODES[None, :] + 1.0))
                om = 1.0 - t * t
            exponent = -0.5 * ((h - k * t) ** 2 / om + k * k)
            integral = np.sum(np.exp(exponent) * _GL_WEIGHTS[None, :], axis=1)
            result[interior] += 0.5 * self.angle[selected] * integral / (2.0 * np.pi)
        upper = np.minimum(u, v)
        lower = np.minimum(np.maximum(u + v - 1.0, 0.0), upper)
        return np.maximum(np.minimum(result, upper), lower)

    def query(self, q, variant='beta_gaussian_copula'):
        q = np.asarray(q, float)
        if q.shape != (4,) or not np.isfinite(q).all() or q[0] > q[1] or (q[2] > q[3]):
            raise ValueError('Expected one finite closed rectangle')
        if variant not in ('beta_gaussian_copula', 'beta_independent'):
            raise ValueError('Unknown frozen moment variant')
        if q[1] < self.root[0] or q[0] > self.root[1] or q[3] < self.root[2] or (q[2] > self.root[3]):
            return dict(reference_count=0.0, full_mass=0, touched_cells=0, partial_cells=0)
        ids = self._candidate_ids(q)
        b = self.bounds[ids]
        touched = (b[:, 0] <= q[1]) & (b[:, 1] >= q[0]) & (b[:, 2] <= q[3]) & (b[:, 3] >= q[2])
        ids = ids[touched]
        b = b[touched]
        full = (q[0] <= b[:, 0]) & (b[:, 1] <= q[1]) & (q[2] <= b[:, 2]) & (b[:, 3] <= q[3])
        L = int(self.mass[ids[full]].sum(dtype=np.uint64))
        partial = ids[~full]
        b = b[~full]
        if not len(partial):
            return dict(reference_count=float(L), full_mass=L, touched_cells=len(ids), partial_cells=0)
        local = np.column_stack(((q[0] - b[:, 0]) / (b[:, 1] - b[:, 0]), (q[1] - b[:, 0]) / (b[:, 1] - b[:, 0]), (q[2] - b[:, 2]) / (b[:, 3] - b[:, 2]), (q[3] - b[:, 2]) / (b[:, 3] - b[:, 2])))
        local = np.clip(local, 0.0, 1.0).astype(np.float32).astype(float)
        lx, ux = self._marginal(local[:, 0], local[:, 1], partial, 0)
        ly, uy = self._marginal(local[:, 2], local[:, 3], partial, 1)
        if variant == 'beta_independent':
            p = (ux - lx) * (uy - ly)
        else:
            corners = self._cdf(np.concatenate((ux, lx, ux, lx)), np.concatenate((uy, uy, ly, ly)), np.tile(partial, 4)).reshape(4, len(partial))
            p = corners[0] - corners[1] - corners[2] + corners[3]
        p = np.clip(p, 0.0, 1.0)
        reference = float(L) + float(np.sum(self.mass[partial].astype(float) * p))
        return dict(reference_count=reference, full_mass=L, touched_cells=len(ids), partial_cells=len(partial))

    def score(self, q, H, variant='beta_gaussian_copula'):
        H = float(H)
        if not np.isfinite(H) or H < 0:
            raise ValueError('Candidate answer must be finite and nonnegative')
        reference = self.query(q, variant)['reference_count']
        return abs(H - reference) / max(reference, 1.0)
'''

In [ ]:
# Embedded helper: budget_fine.py
EMBEDDED19['budget_fine.py'] = r'''"""Exact count-only fine sidecars under actual byte budgets.

The same compact64-byte codec as the support-critic control is used. Every
requested non-dyadic grid is counted directly from raw rows in a shared pass;
no area projection from another grid and no query truth is used.
"""
from pathlib import Path
import gzip
import hashlib
import json
import math
import struct
import time
import numpy as np
import pandas as pd
MAGIC = b'SLCNT001'
HEADER = struct.Struct('<8sII4d16x')

def fine_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def largest_fine_side(cap):
    cap = int(cap)
    if cap < 68:
        raise ValueError('Fine sidecar budget must fit64-byte header and one uint32 count')
    n = math.isqrt((cap - 64) // 4)
    if not 64 + 4 * n * n <= cap < 64 + 4 * (n + 1) * (n + 1):
        raise AssertionError('Nonmaximal grid budget calculation')
    return n

def _fine_root(root):
    root = tuple(map(float, root))
    if len(root) != 4 or not np.isfinite(root).all() or (not root[0] < root[1]) or (not root[2] < root[3]):
        raise ValueError('Expected finite increasing (xmin,xmax,ymin,ymax) root')
    return root

def save_fine(path, counts, root):
    counts = np.asarray(counts)
    root = _fine_root(root)
    if counts.ndim != 2 or counts.shape[0] != counts.shape[1] or counts.shape[0] < 1:
        raise ValueError('Fine counts must be a nonempty square array')
    if not np.isfinite(counts).all() or (counts < 0).any() or (not np.equal(counts, np.floor(counts)).all()):
        raise ValueError('Fine cell masses must be finite nonnegative integers')
    if counts.max(initial=0) > np.iinfo(np.uint32).max:
        raise OverflowError('Fine cell mass exceeds uint32')
    path = Path(path)
    path.write_bytes(HEADER.pack(MAGIC, counts.shape[0], 1, *root) + np.asarray(counts, dtype='<u4').tobytes(order='C'))
    return path.stat().st_size

def load_fine(path, expected_sha256=None):
    path = Path(path)
    if expected_sha256 is not None and fine_sha(path) != expected_sha256:
        raise ValueError('Fine sidecar SHA256 mismatch')
    payload = path.read_bytes()
    if len(payload) < HEADER.size:
        raise ValueError('Truncated fine header')
    magic, n, version, *root = HEADER.unpack_from(payload)
    if magic != MAGIC or version != 1 or n < 1 or (payload[48:64] != b'\x00' * 16) or (len(payload) != HEADER.size + 4 * n * n):
        raise ValueError('Invalid fine sidecar header, version, flags or length')
    root = _fine_root(root)
    counts = np.frombuffer(payload, dtype='<u4', offset=HEADER.size).reshape(n, n).copy()
    counts.setflags(write=False)
    bx = np.linspace(root[0], root[1], n + 1)
    by = np.linspace(root[2], root[3], n + 1)
    if (np.diff(bx) <= 0).any() or (np.diff(by) <= 0).any():
        raise ValueError('Fine boundaries collapse in float64')
    return dict(counts=counts, root=root, bx=bx, by=by, n=int(n), serialized_bytes=len(payload), total_mass=int(counts.sum(dtype=np.uint64)))

def fine_reference(rectangles, hist):
    """Uniform-area reference count from the resident sidecar, never truth.

    One rectangle returns a float; N×4 rectangles return a float64 vector.
    Closed-query boundary contact has zero area, as in the existing16 kernel.
    """
    values = np.asarray(rectangles, float)
    single = values.ndim == 1 and values.size > 0
    if single:
        values = values.reshape(1, -1)
    if values.size == 0:
        values = np.empty((0, 4), float)
    if values.ndim != 2 or values.shape[1] != 4 or (not np.isfinite(values).all()) or (values[:, 0] > values[:, 1]).any() or (values[:, 2] > values[:, 3]).any():
        raise ValueError('Expected finite closed query rectangles')
    bx, by, counts = (hist['bx'], hist['by'], hist['counts'])
    nx, ny = counts.shape
    answers = np.zeros(len(values), float)
    for k, q in enumerate(values):
        i0 = max(0, int(np.searchsorted(bx, q[0], side='right') - 1))
        i1 = min(nx, int(np.searchsorted(bx, q[1], side='left')))
        j0 = max(0, int(np.searchsorted(by, q[2], side='right') - 1))
        j1 = min(ny, int(np.searchsorted(by, q[3], side='left')))
        if i1 <= i0 or j1 <= j0:
            continue
        fx = np.maximum(0.0, np.minimum(q[1], bx[i0 + 1:i1 + 1]) - np.maximum(q[0], bx[i0:i1])) / np.diff(bx[i0:i1 + 1])
        fy = np.maximum(0.0, np.minimum(q[3], by[j0 + 1:j1 + 1]) - np.maximum(q[2], by[j0:j1])) / np.diff(by[j0:j1 + 1])
        answers[k] = float(np.sum(counts[i0:i1, j0:j1] * fx[:, None] * fy[None, :]))
    return float(answers[0]) if single else answers

def _fine_raw_points(path, chunk_size=100000):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as stream:
        for frame in pd.read_csv(stream, sep='\t', header=None, usecols=[0, 1], dtype=float, chunksize=chunk_size, float_precision='round_trip', skip_blank_lines=False):
            points = frame.to_numpy(float)
            if not np.isfinite(points).all():
                raise ValueError('Nonfinite raw point')
            yield points

def build_fine_counts(rawpaths, root, budgets, outdir, expectedhashes):
    """Build all requested count grids directly in ONE shared raw-data pass.

    `expectedhashes` is keyed by absolute path, filename or integer source ID.
    Its values may be SHA256 strings or dictionaries containing raw_sha256
    (or sha256) and optional expected_rows (or raw_rows). Per-grid construction
    times are intentionally not invented from the shared input scan.
    """
    root = _fine_root(root)
    caps = sorted(set(map(int, budgets)))
    if not caps:
        raise ValueError('No fine budgets requested')
    sides = {cap: largest_fine_side(cap) for cap in caps}
    paths = [Path(p).resolve() for p in rawpaths]
    if not paths or len(set(paths)) != len(paths):
        raise ValueError('Raw paths must be nonempty and distinct')
    identities = []
    for path in paths:
        sid = path.name.split('.')[0]
        keys = (str(path), path.name, int(sid) if sid.isdigit() else sid, sid)
        item = next((expectedhashes[key] for key in keys if key in expectedhashes), None)
        if item is None:
            raise ValueError('Expected raw identity missing for ' + path.name)
        if isinstance(item, str):
            digest = item
            rows = None
        else:
            digest = item.get('raw_sha256', item.get('sha256'))
            rows = item.get('expected_rows', item.get('raw_rows'))
        if not isinstance(digest, str) or len(digest) != 64:
            raise ValueError('Invalid expected raw SHA256')
        identities.append(dict(path=str(path), sha256=digest, expected_rows=None if rows is None else int(rows)))
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    manifest_path = outdir / 'run.json'
    identity = dict(root=list(root), budgets=caps, raw_identities=identities, codec='SLCNT001-v1-header64-u32')
    if manifest_path.is_file():
        prior = json.loads(manifest_path.read_text())
        if prior.get('status') != 'complete' or prior.get('identity') != identity:
            raise FileExistsError('Use a new output directory; a partial/different fine build is preserved')
        mapping = {int(cap): info for cap, info in prior['payloads'].items()}
        for cap, info in mapping.items():
            hist = load_fine(info['path'], info['sha256'])
            if hist['n'] != sides[cap] or hist['serialized_bytes'] != info['actual_bytes']:
                raise ValueError('Existing fine metadata mismatch')
        return mapping
    state = dict(status='running', study='budgeted-count-only-fine-construction', identity=identity, construction_uses_queries=False, construction_uses_truth=False, construction_uses_models=False, raw_sources=[], cost_scope='One raw scan builds every requested grid; shared cost, not separate per-grid timings')
    manifest_path.write_text(json.dumps(state, indent=2))
    grids = {}
    for n in sorted(set(sides.values())):
        bx = np.linspace(root[0], root[1], n + 1)
        by = np.linspace(root[2], root[3], n + 1)
        if (np.diff(bx) <= 0).any() or (np.diff(by) <= 0).any():
            raise ValueError('Fine grid boundaries collapse')
        grids[n] = dict(bx=bx, by=by, counts=np.zeros((n, n), np.uint64))
    started = time.perf_counter()
    total_rows = 0
    for item in identities:
        path = Path(item['path'])
        tick = time.perf_counter()
        if fine_sha(path) != item['sha256']:
            raise ValueError('Raw identity mismatch before fine construction')
        hash_before = time.perf_counter() - tick
        scan_start = time.perf_counter()
        rows = 0
        for points in _fine_raw_points(path):
            if ((points[:, 0] < root[0]) | (points[:, 0] > root[1]) | (points[:, 1] < root[2]) | (points[:, 1] > root[3])).any():
                raise ValueError('Raw point outside the inherited common root')
            rows += len(points)
            for n, grid in grids.items():
                ix = np.minimum(np.searchsorted(grid['bx'], points[:, 0], side='right') - 1, n - 1)
                iy = np.minimum(np.searchsorted(grid['by'], points[:, 1], side='right') - 1, n - 1)
                grid['counts'] += np.bincount(ix * n + iy, minlength=n * n).reshape(n, n).astype(np.uint64)
        scan_seconds = time.perf_counter() - scan_start
        tick = time.perf_counter()
        if fine_sha(path) != item['sha256']:
            raise ValueError('Raw identity changed during fine construction')
        hash_after = time.perf_counter() - tick
        if item['expected_rows'] is not None and rows != item['expected_rows']:
            raise ValueError('Raw row count mismatch')
        total_rows += rows
        source = dict(item, raw_rows=rows, file_bytes=path.stat().st_size, hash_before_seconds=hash_before, one_pass_all_grids_seconds=scan_seconds, hash_after_seconds=hash_after)
        state['raw_sources'].append(source)
        print('Fine count construction source', path.name, 'rows', rows, 'seconds', round(scan_seconds, 3), flush=True)
    mapping = {}
    for n, grid in grids.items():
        if int(grid['counts'].sum(dtype=np.uint64)) != total_rows:
            raise ValueError('Fine grid does not conserve raw row mass')
        if grid['counts'].max(initial=0) > np.iinfo(np.uint32).max:
            raise OverflowError('Fine count exceeds uint32')
        path = outdir / f'count_n{n}.summary'
        actual = save_fine(path, grid['counts'], root)
        loaded = load_fine(path)
        if not np.array_equal(loaded['counts'], grid['counts']):
            raise ValueError('Fine codec round-trip mismatch')
        for cap in caps:
            if sides[cap] != n:
                continue
            if actual > cap or 64 + 4 * (n + 1) * (n + 1) <= cap:
                raise ValueError('Fine payload violates or underuses maximal integer-side budget')
            mapping[cap] = dict(path=str(path.resolve()), n=n, actual_bytes=actual, budget_bytes=cap, unused_bytes=cap - actual, next_side_bytes=64 + 4 * (n + 1) * (n + 1), sha256=fine_sha(path), raw_rows=total_rows, input_counts_exact=True, direct_raw_counts=True, query_truth_used=False, model_bytes_included=False)
    state.update(status='complete', payloads=mapping, total_raw_rows=total_rows, one_pass_construction_with_hashes_seconds=time.perf_counter() - started, resident_grid_count_array_bytes=sum((g['counts'].nbytes for g in grids.values())), allocation_rule='largest square side N with64+4N² <= given cap; boundaries implicit from root')
    manifest_path.write_text(json.dumps(state, indent=2))
    return mapping
'''

In [ ]:
# Embedded helper: generation19.py
EMBEDDED19['generation19.py'] = r'''"""Frozen, geometry-only query generation for notebook 17.

No function in this module receives labels, model predictions or acceptance
outcomes. Coarse count arrays are used only to validate histogram dimensions.
"""
import hashlib
import math
from itertools import product
import numpy as np
import pandas as pd
GENERATION_VERSION = '17-fresh-diagonal-v1'
MAX_DIAGONAL_SHIFT = math.sqrt(0.05)
NEAR_GEOMETRY_TOL = 1e-10
MAX_GEOMETRY_ATTEMPTS = 256
PLAN_COLUMNS = ['query_id', 'family', 'cell', 'source_id', 'cell_x', 'cell_y', 'x0', 'x1', 'y0', 'y1', 'shift', 'attempt']
MEMBERSHIP_COLUMNS = ['query_id', 'family', 'cell', 'seed']

def _ordered_grids(coarse_hists, source_ids):
    if len(coarse_hists) != len(source_ids) or not len(coarse_hists):
        raise ValueError('Each coarse histogram requires one distinct source ID')
    ids = []
    for value in source_ids:
        try:
            number = float(value)
        except (TypeError, ValueError) as error:
            raise ValueError('Source IDs must be distinct nonnegative integers') from error
        if not np.isfinite(number) or int(number) != number or int(number) < 0:
            raise ValueError('Source IDs must be distinct nonnegative integers')
        ids.append(int(number))
    if len(set(ids)) != len(ids):
        raise ValueError('Duplicate source IDs')
    ordered = []
    for source_id, hist in sorted(zip(ids, coarse_hists), key=lambda item: item[0]):
        if len(hist) != 4:
            raise ValueError('Histogram must be (rate, bx, by, counts)')
        bx, by = (np.asarray(hist[1], float), np.asarray(hist[2], float))
        if bx.ndim != 1 or by.ndim != 1 or len(bx) < 2 or (len(by) < 2) or (not np.isfinite(bx).all()) or (not np.isfinite(by).all()) or (np.diff(bx) <= 0).any() or (np.diff(by) <= 0).any() or (np.shape(hist[3]) != (len(bx) - 1, len(by) - 1)):
            raise ValueError('Invalid source boundaries or dimensions')
        width, height = (float(np.diff(bx).mean()), float(np.diff(by).mean()))
        if not np.allclose(np.diff(bx), width, rtol=1e-08, atol=1e-12) or not np.allclose(np.diff(by), height, rtol=1e-08, atol=1e-12):
            raise ValueError('Fresh queries require the original uniform source grids')
        ordered.append(dict(source_id=source_id, bx=bx, by=by, width=width, height=height))
    return ordered

def _rectangles(rectangles):
    values = np.asarray(rectangles, float)
    if values.ndim != 2 or values.shape[1] != 4 or (not np.isfinite(values).all()) or (values[:, 0] >= values[:, 1]).any() or (values[:, 2] >= values[:, 3]).any():
        raise ValueError('Rectangles must be an N x 4 finite increasing array')
    return values

def _infer_with_grids(rectangles, grids):
    q = _rectangles(rectangles)
    if not len(q):
        return np.empty(0, dtype=str)
    width, height = (q[:, 1] - q[:, 0], q[:, 3] - q[:, 2])
    centres_x, centres_y = ((q[:, 0] + q[:, 1]) / 2, (q[:, 2] + q[:, 3]) / 2)
    size_gap = np.asarray([[abs(w - g['width']) + abs(h - g['height']) for g in grids] for w, h in zip(width, height)])
    source_position = size_gap.argmin(axis=1)
    i = np.asarray([np.searchsorted(grids[s]['bx'], x) - 1 for s, x in zip(source_position, centres_x)])
    j = np.asarray([np.searchsorted(grids[s]['by'], y) - 1 for s, y in zip(source_position, centres_y)])
    return np.asarray([f"{grids[s]['source_id']}_{a}_{b}" for s, a, b in zip(source_position, i, j)], dtype=str)

def infer_cells(rectangles, coarse_hists, source_ids):
    """Match model_study.source_cells on numerically sorted source file IDs."""
    return _infer_with_grids(rectangles, _ordered_grids(coarse_hists, source_ids))

def _geometry_hash(rectangle):
    values = np.asarray(rectangle, dtype='<f8').copy()
    values[values == 0] = 0.0
    return hashlib.sha256(values.tobytes()).hexdigest()

def _shift_for_cell(query_seed, cell, attempt):
    salt = f'{GENERATION_VERSION}:{query_seed}:{cell}:{attempt}'
    integer = int.from_bytes(hashlib.sha256(salt.encode()).digest()[:8], 'big') >> 11
    unit = (integer + 0.5) / 2 ** 53
    unit = min(float(np.nextafter(1.0, 0.0)), max(float(np.nextafter(0.0, 1.0)), float(unit)))
    return MAX_DIAGONAL_SHIFT * unit

def _old_family_diagnostics(q, cells, grid_by_id):
    if not len(q):
        return dict(query_count=0, one_cell_size_fraction=None, diagonal_shift_fraction=None, shift_range_fraction=None, max_xy_shift_difference=None)
    sizes, diagonal, ranges, differences = ([], [], [], [])
    for rectangle, cell in zip(q, cells):
        sid, i, j = map(int, cell.split('_'))
        g = grid_by_id[sid]
        valid = 0 <= i < len(g['bx']) - 1 and 0 <= j < len(g['by']) - 1
        if not valid:
            sizes.append(False)
            diagonal.append(False)
            ranges.append(False)
            continue
        dx, dy = (g['bx'][i + 1] - g['bx'][i], g['by'][j + 1] - g['by'][j])
        ox, oy = ((rectangle[0] - g['bx'][i]) / dx, (rectangle[2] - g['by'][j]) / dy)
        sizes.append(bool(np.isclose((rectangle[1] - rectangle[0]) / dx, 1.0, rtol=0, atol=1e-08) and np.isclose((rectangle[3] - rectangle[2]) / dy, 1.0, rtol=0, atol=1e-08)))
        differences.append(float(abs(ox - oy)))
        diagonal.append(bool(abs(ox - oy) <= 1e-08))
        ranges.append(bool(-1e-08 <= ox <= MAX_DIAGONAL_SHIFT + 1e-08 and -1e-08 <= oy <= MAX_DIAGONAL_SHIFT + 1e-08))
    return dict(query_count=len(q), one_cell_size_fraction=float(np.mean(sizes)), diagonal_shift_fraction=float(np.mean(diagonal)), shift_range_fraction=float(np.mean(ranges)), max_xy_shift_difference=max(differences) if differences else None)

def generate_plan(old_rectangles, coarse_hists, source_ids, outer, seeds, query_seed=2026111602, replication_cells=1000, extra_forbidden_geometry=None):
    """Return a frozen query plan, seed membership and geometry-only audit.

    All old rectangles must be supplied, including rows later excluded by the
    exact-count cohort. No candidate is selected or replaced using new truth.
    A geometry is preallocated for EVERY possible source cell in a fixed order,
    making collision handling independent of which model seeds are requested.
    """
    q = _rectangles(old_rectangles)
    grids = _ordered_grids(coarse_hists, source_ids)
    if not seeds or len(set(seeds)) != len(seeds) or any((not np.isfinite(s) or int(s) != s for s in seeds)):
        raise ValueError('Use distinct integer model seeds')
    seeds = sorted(map(int, seeds))
    if not np.isfinite(query_seed) or int(query_seed) != query_seed or (not np.isfinite(replication_cells)) or (int(replication_cells) != replication_cells) or (replication_cells < 0):
        raise ValueError('query_seed must be integer; replication_cells must be a nonnegative integer')
    query_seed, replication_cells = (int(query_seed), int(replication_cells))
    grid_by_id = {g['source_id']: g for g in grids}
    extra = np.empty((0, 4), float) if extra_forbidden_geometry is None else _rectangles(extra_forbidden_geometry)
    from scipy.spatial import cKDTree
    global_scale = np.asarray([max((g['width'] for g in grids))] * 2 + [max((g['height'] for g in grids))] * 2)
    forbidden = np.vstack((q, extra))
    forbidden_tree = cKDTree(forbidden / global_scale) if len(forbidden) else None
    rejected_extra = 0
    inventory = []
    for g in grids:
        for i in range(len(g['bx']) - 1):
            for j in range(len(g['by']) - 1):
                inventory.append((f"{g['source_id']}_{i}_{j}", g['source_id'], i, j))
    possible = {cell for cell, _, _, _ in inventory}
    old_cell_array = _infer_with_grids(q, grids)
    old_cells = set(old_cell_array)
    grouped_old = {}
    for cell, rectangle in zip(old_cell_array, q):
        grouped_old.setdefault(cell, []).append(rectangle)
    grouped_old = {cell: np.asarray(values, float) for cell, values in grouped_old.items()}
    old_hashes = {_geometry_hash(rectangle) for rectangle in q}
    if not {'seed', 'cell', 'split'} <= set(outer.columns):
        raise ValueError('Outer membership requires seed, cell and split')
    outside = outer.copy()
    if outside[['seed', 'cell', 'split']].isna().any().any():
        raise ValueError('Missing outer role identifiers')
    if not np.equal(outside.seed.to_numpy(float), np.floor(outside.seed.to_numpy(float))).all():
        raise ValueError('Outer model seeds must be integers')
    outside['seed'] = outside.seed.astype(int)
    outside['cell'] = outside.cell.astype(str)
    outside = outside[outside.seed.isin(seeds)]
    selected_secondary = {}
    for seed in seeds:
        part = outside[outside.seed == seed]
        if set(part.split) != {'train', 'calibration', 'test'}:
            raise ValueError('Each requested seed needs train, calibration and test roles')
        if not set(part.cell) <= possible:
            raise ValueError('Outer membership contains invalid source-cell IDs')
        if not set(part.cell) <= old_cells:
            raise ValueError('Outer cell IDs disagree with all-old query geometry')
        roles = part[['cell', 'split']].drop_duplicates()
        if roles.cell.duplicated().any():
            raise ValueError('A source cell overlaps outer train/calibration/test roles')
        test_cells = set(roles.loc[roles.split == 'test', 'cell'])
        ordered = sorted(test_cells, key=lambda cell: hashlib.sha256(f'{GENERATION_VERSION}:replication:{query_seed}:{cell}'.encode()).digest())
        selected_secondary[seed] = set(ordered[:replication_cells])
    primary = possible - old_cells
    if not primary:
        raise ValueError('No globally unused source-cell groups; primary validation cannot be substituted')
    secondary = set().union(*selected_secondary.values())
    if primary & secondary:
        raise ValueError('Primary and secondary populations overlap')
    generated = {}
    new_hashes = set()
    rejected_near_old = 0
    rejected_duplicate = 0
    new_geometry_bins = {}
    bucket_scale = NEAR_GEOMETRY_TOL * np.asarray([max((g['width'] for g in grids))] * 2 + [max((g['height'] for g in grids))] * 2)
    neighbor_offsets = list(product((-1, 0, 1), repeat=4))
    for cell, sid, i, j in inventory:
        g = grid_by_id[sid]
        dx = g['bx'][i + 1] - g['bx'][i]
        dy = g['by'][j + 1] - g['by'][j]
        old = grouped_old.get(cell)
        scale = np.asarray([dx, dx, dy, dy])
        for attempt in range(MAX_GEOMETRY_ATTEMPTS):
            shift = _shift_for_cell(query_seed, cell, attempt)
            x0 = g['bx'][i] + shift * dx
            y0 = g['by'][j] + shift * dy
            rectangle = np.asarray([x0, x0 + dx, y0, y0 + dy], float)
            digest = _geometry_hash(rectangle)
            if digest in old_hashes or (old is not None and np.any(np.max(np.abs((old - rectangle) / scale), axis=1) <= NEAR_GEOMETRY_TOL)):
                rejected_near_old += 1
                continue
            if forbidden_tree is not None:
                neighbours = forbidden_tree.query_ball_point(rectangle / global_scale, r=NEAR_GEOMETRY_TOL * 1.001, p=np.inf)
                if neighbours and np.any(np.max(np.abs((forbidden[neighbours] - rectangle) / scale), axis=1) <= NEAR_GEOMETRY_TOL):
                    rejected_extra += 1
                    continue
            bucket = tuple((int(np.floor(value / unit)) for value, unit in zip(rectangle, bucket_scale)))
            near_new = False
            for delta in neighbor_offsets:
                key = tuple((a + b for a, b in zip(bucket, delta)))
                for previous in new_geometry_bins.get(key, []):
                    if np.max(np.abs((previous - rectangle) / scale)) <= NEAR_GEOMETRY_TOL:
                        near_new = True
                        break
                if near_new:
                    break
            if digest in new_hashes or near_new:
                rejected_duplicate += 1
                continue
            inferred = _infer_with_grids(rectangle.reshape(1, 4), grids)[0]
            if inferred != cell:
                raise ValueError('Source grids cannot reproduce declared generating-cell IDs; check ambiguous source sizes or boundaries')
            generated[cell] = dict(cell=cell, source_id=sid, cell_x=i, cell_y=j, x0=float(rectangle[0]), x1=float(rectangle[1]), y0=float(rectangle[2]), y1=float(rectangle[3]), shift=float(shift), attempt=attempt, geometry_hash=digest)
            new_hashes.add(digest)
            new_geometry_bins.setdefault(bucket, []).append(rectangle)
            break
        else:
            raise ValueError('Cannot generate distinct finite geometry for cell ' + cell)
    plans = []
    members = []
    for family, chosen in (('new_groups', primary), ('existing_test_groups', secondary)):
        for cell, _, _, _ in inventory:
            if cell not in chosen:
                continue
            geometry = generated[cell]
            query_id = f"q19-{family}-{cell}-{geometry['geometry_hash'][:16]}"
            row = dict(query_id=query_id, family=family, **{k: v for k, v in geometry.items() if k != 'geometry_hash'})
            plans.append(row)
            for seed in seeds:
                if family == 'new_groups' or cell in selected_secondary[seed]:
                    members.append(dict(query_id=query_id, family=family, cell=cell, seed=seed))
    plan = pd.DataFrame(plans, columns=PLAN_COLUMNS)
    membership = pd.DataFrame(members, columns=MEMBERSHIP_COLUMNS)
    if plan.query_id.duplicated().any() or membership.duplicated(['query_id', 'seed']).any():
        raise ValueError('Duplicate planned query identities')
    if plan[['x0', 'x1', 'y0', 'y1']].duplicated().any():
        raise ValueError('Duplicate planned query geometry')
    if not set(plan.loc[plan.family == 'new_groups', 'cell']).isdisjoint(old_cells):
        raise ValueError('Primary group leaked from old query inventory')
    for seed in seeds:
        assigned = membership[(membership.seed == seed) & (membership.family == 'existing_test_groups')]
        test = set(outside.loc[(outside.seed == seed) & (outside.split == 'test'), 'cell'])
        other = set(outside.loc[(outside.seed == seed) & outside.split.isin(['train', 'calibration']), 'cell'])
        if not set(assigned.cell) <= test or set(assigned.cell) & other:
            raise ValueError('Secondary membership overlaps model fitting/calibration groups')
    audit = dict(protocol=GENERATION_VERSION, query_seed=query_seed, maximum_diagonal_shift=MAX_DIAGONAL_SHIFT, near_geometry_tolerance=NEAR_GEOMETRY_TOL, all_old_query_count=len(q), all_old_cells=len(old_cells), possible_source_cells=len(possible), old_cells_outside_inventory=len(old_cells - possible), new_groups=len(primary), secondary_counts={str(s): len(selected_secondary[s]) for s in seeds}, secondary_union_cells=len(secondary), planned_queries=len(plan), planned_model_query_cases=len(membership), geometries_preallocated=len(generated), rejected_near_old=rejected_near_old, rejected_duplicate_geometry=rejected_duplicate, geometry_selection_uses_outcomes=False, extra_forbidden_geometry_count=len(extra), rejected_near_extra_or_other_old=rejected_extra, primary_old_group_overlap=0, secondary_fit_calibration_overlap=0, old_family_geometrymatch=_old_family_diagnostics(q, old_cell_array, grid_by_id), scope='new rectangles/labels on same raw dataset; model-unseen generating IDs were already inspected in17; not untouched geography or a new dataset')
    return (plan, membership, audit)
'''

In [ ]:
# Embedded helper: exact_counter.py
EMBEDDED19['exact_counter.py'] = r'''"""Offline exact 2D closed-rectangle counts directly over raw points.

The spatial tree is only an index: counts are either the number of original
rows wholly enclosed by a query or direct comparisons of original leaf points.
No histogram/label/model value enters the answer. Duplicate points count once
per raw row. One source is loaded/indexed/released at a time.
"""
from pathlib import Path
import time
import numpy as np
from scipy.spatial import cKDTree

def _exact_rectangles(rectangles):
    q = np.asarray(rectangles, dtype=np.float64)
    if q.ndim != 2 or q.shape[1] != 4 or (not np.isfinite(q).all()):
        raise ValueError('Rectangles must be finite [min_x,max_x,min_y,max_y] rows')
    if (q[:, 0] > q[:, 1]).any() or (q[:, 2] > q[:, 3]).any():
        raise ValueError('Rectangle lower bounds exceed upper bounds')
    return q

def exact_rectangle_count(tree, rectangle):
    """Count one closed rectangle using public cKDTree node attributes.

    Root bounds and split coordinates are existing float64 point coordinates.
    All comparisons use <=/>= directly; no epsilon expansion/normalization or
    approximate distance search changes the raw query semantics.
    """
    x0, x1, y0, y1 = map(float, rectangle)
    if tree.n == 0:
        return 0
    pending = [(tree.tree, float(tree.mins[0]), float(tree.maxes[0]), float(tree.mins[1]), float(tree.maxes[1]))]
    answer = 0
    while pending:
        node, lx, hx, ly, hy = pending.pop()
        if hx < x0 or lx > x1 or hy < y0 or (ly > y1):
            continue
        if x0 <= lx and hx <= x1 and (y0 <= ly) and (hy <= y1):
            answer += int(node.children)
            continue
        if node.split_dim < 0:
            points = tree.data[tree.indices[node.start_idx:node.end_idx]]
            answer += int(np.count_nonzero((points[:, 0] >= x0) & (points[:, 0] <= x1) & (points[:, 1] >= y0) & (points[:, 1] <= y1)))
        elif node.split_dim == 0:
            split = float(node.split)
            pending.append((node.lesser, lx, split, ly, hy))
            pending.append((node.greater, split, hx, ly, hy))
        elif node.split_dim == 1:
            split = float(node.split)
            pending.append((node.lesser, lx, hx, ly, split))
            pending.append((node.greater, lx, hx, split, hy))
        else:
            raise ValueError('Expected a two-dimensional point tree')
    return answer

def count_point_rectangles(points, rectangles, leafsize=128):
    """Index raw coordinates and count each rectangle, returning setup/query cost."""
    points = np.asarray(points, dtype=np.float64, order='C')
    rectangles = _exact_rectangles(rectangles)
    if points.ndim != 2 or points.shape[1] != 2 or (not np.isfinite(points).all()):
        raise ValueError('Points must be finite two-dimensional coordinates')
    if int(leafsize) != leafsize or leafsize < 1:
        raise ValueError('leafsize must be positive')
    tick = time.perf_counter()
    tree = cKDTree(points, leafsize=int(leafsize), compact_nodes=True, balanced_tree=True, copy_data=False)
    tree.tree
    build_seconds = time.perf_counter() - tick
    tick = time.perf_counter()
    counts = np.fromiter((exact_rectangle_count(tree, q) for q in rectangles), dtype=np.int64, count=len(rectangles))
    query_seconds = time.perf_counter() - tick
    return (counts, dict(raw_points=len(points), query_count=len(rectangles), index_build_seconds=build_seconds, exact_query_seconds=query_seconds, raw_coordinate_bytes=points.nbytes, leafsize=int(leafsize), algorithm='exact closed 2D range count over cKDTree nodes/raw leaf points'))

def exact_range_counts(points, rectangles, leafsize=128):
    """Return int64 exact inclusive raw-point counts; no labels or histograms.

    Zero-width/height closed boxes are well-defined and count boundary points.
    The new-query generator can impose its stricter positive-area requirement.
    """
    return count_point_rectangles(points, rectangles, leafsize=leafsize)[0]

def direct_point_counts(points, rectangles):
    """Independent brute-force oracle, no index or precomputed answers."""
    points = np.asarray(points, dtype=np.float64)
    rectangles = _exact_rectangles(rectangles)
    return np.array([np.count_nonzero((points[:, 0] >= q[0]) & (points[:, 0] <= q[1]) & (points[:, 1] >= q[2]) & (points[:, 1] <= q[3])) for q in rectangles], dtype=np.int64)

def count_raw_rectangles(raw_paths, rectangles, iter_points, expected_rows=None, audit_indices=(), leafsize=128):
    """Build exact new labels from files, one source at a time.

    iter_points(path) must yield finite Nx2 raw arrays (e.g. verified notebook15
    TAB/gzip reader). expected_rows, when given, avoids a concatenate double copy
    and detects changed/truncated sources. File hashes belong to the caller's
    provenance checks. audit_indices are fixed independently of outcomes.

    Returns (total_counts, source_diagnostics, brute_audit_rows). No true labels,
    model scores, features or histogram answers are accepted.
    """
    rectangles = _exact_rectangles(rectangles)
    raw_paths = list(raw_paths)
    if expected_rows is not None and len(expected_rows) != len(raw_paths):
        raise ValueError('One expected row count is required per source')
    audit_indices = np.asarray(audit_indices, dtype=np.int64)
    if (audit_indices < 0).any() or (audit_indices >= len(rectangles)).any():
        raise ValueError('Audit query index outside new query set')
    if len(set(audit_indices.tolist())) != len(audit_indices):
        raise ValueError('Duplicate audit indices')
    total = np.zeros(len(rectangles), dtype=np.int64)
    brute_total = np.zeros(len(audit_indices), dtype=np.int64)
    diagnostics = []
    for source_index, path in enumerate(raw_paths):
        tick = time.perf_counter()
        if expected_rows is None:
            pieces = list(iter_points(path))
            points = np.concatenate(pieces, axis=0) if pieces else np.empty((0, 2), dtype=np.float64)
            del pieces
        else:
            n = int(expected_rows[source_index])
            if n < 0:
                raise ValueError('Expected source row count cannot be negative')
            points = np.empty((n, 2), dtype=np.float64)
            position = 0
            for part in iter_points(path):
                part = np.asarray(part, dtype=np.float64)
                if part.ndim != 2 or part.shape[1] != 2 or (not np.isfinite(part).all()):
                    raise ValueError('Raw reader emitted invalid coordinates')
                if position + len(part) > n:
                    raise ValueError('Raw source has more rows than its saved identity')
                points[position:position + len(part)] = part
                position += len(part)
            if position != n:
                raise ValueError('Raw source has fewer rows than its saved identity')
        load_seconds = time.perf_counter() - tick
        counts, info = count_point_rectangles(points, rectangles, leafsize=leafsize)
        total += counts
        tick = time.perf_counter()
        brute = direct_point_counts(points, rectangles[audit_indices])
        brute_total += brute
        if not np.array_equal(counts[audit_indices], brute):
            raise ValueError('Indexed exact counts disagree with independent raw-point brute scan')
        info.update(source_index=source_index, raw_path=str(path), raw_load_seconds=load_seconds, brute_audit_seconds=time.perf_counter() - tick)
        diagnostics.append(info)
        del points
    rows = [dict(query_index=int(i), indexed_exact_count=int(total[i]), brute_raw_count=int(v), matches=bool(total[i] == v)) for i, v in zip(audit_indices, brute_total)]
    return (total, diagnostics, rows)
'''

In [ ]:
# Embedded helper: sa_core.py
EMBEDDED19['sa_core.py'] = r'''"""Count-only estimation and the repository's Skewness-Align query score.

These functions use the same positive-area overlap semantics as features.py and
SimpleSpatialHistogramOpt.RangeQueryWithRatio. SA is an area-fraction-weighted
heuristic, not a prediction expressed in relative-error percentage units.

sa_alpha implements the mathematical density flattening (correct row stride).
The original logger's legacy stride implementation is equivalent on the square
ARM grids and the square finer grids used in this comparison. It is not a
promise to reproduce the legacy rectangular-grid stride bug.
"""
import numpy as np
from scipy.special import betainc

def _sa_hist_parts(hist):
    if len(hist) != 4:
        raise ValueError('Histogram must be (rate, bx, by, counts)')
    _, bx, by, counts = hist
    bx, by, counts = (np.asarray(bx), np.asarray(by), np.asarray(counts))
    if bx.ndim != 1 or by.ndim != 1 or len(bx) < 2 or (len(by) < 2) or (counts.ndim != 2) or (counts.shape != (len(bx) - 1, len(by) - 1)):
        raise ValueError('Invalid histogram dimensions')
    return (bx, by, counts)

def sa_alpha(hist):
    """Compute target-histogram beta shape once at initialization, without sources.

    Constants and the median/MAD convention match CountHistExp/ErrorPredictionLogExp.
    Zero-density cells are retained. All-empty or zero-median/zero-MAD grids
    legitimately produce alpha=500001 and may make SA scores heavily tied.
    """
    bx, by, counts = _sa_hist_parts(hist)
    if not np.isfinite(bx).all() or not np.isfinite(by).all() or (not np.isfinite(counts).all()) or (counts < 0).any() or (np.diff(bx) <= 0).any() or (np.diff(by) <= 0).any():
        raise ValueError('Invalid histogram boundaries or counts')
    density = counts / np.outer(np.diff(bx), np.diff(by))
    median = float(np.median(density))
    mad = float(np.median(np.abs(density - median)))
    alpha = 0.5 / (mad / (median + 1e-06) + 1e-06) + 1.0
    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError('Invalid SA beta shape')
    return float(alpha)

def _sa_overlap(boundaries, lower, upper):
    n = len(boundaries) - 1
    i0 = int(np.clip(np.searchsorted(boundaries, lower, 'right') - 1, 0, n - 1))
    i1 = int(np.clip(np.searchsorted(boundaries, upper, 'left') - 1, 0, n - 1))
    left, right = (boundaries[i0:i1 + 1], boundaries[i0 + 1:i1 + 2])
    fraction = np.clip((np.minimum(upper, right) - np.maximum(lower, left)) / (right - left), 0, 1)
    return (i0, i1, fraction)

def _sa_query_overlap(q, hist):
    """Return positive-area geometry plus aligned counts; hist is setup-validated.

    Per-query code intentionally avoids scanning/revalidating the entire grid or
    allocating a float64 copy of every count. Query construction never receives
    truth, training labels, a source file path or raw points.
    """
    bx, by, counts = _sa_hist_parts(hist)
    q = np.asarray(q, dtype=float)
    if q.shape != (4,) or not np.isfinite(q).all() or q[0] >= q[1] or (q[2] >= q[3]):
        raise ValueError('Query must contain four finite increasing rectangle bounds')
    if q[1] <= bx[0] or q[0] >= bx[-1] or q[3] <= by[0] or (q[2] >= by[-1]):
        return (np.empty((0, 0)), np.empty((0, 0)))
    i0, i1, fx = _sa_overlap(bx, q[0], q[1])
    j0, j1, fy = _sa_overlap(by, q[2], q[3])
    weight = np.outer(fx, fy)
    return (weight, counts[i0:i1 + 1, j0:j1 + 1])

def histogram_count(q, hist):
    """Uniform-area range-count estimate only, without ML/source feature work."""
    weight, counts = _sa_query_overlap(q, hist)
    return float(np.sum(weight * counts))

def query_answer_sa(q, hist, alpha):
    """Return (count estimate, original SA heuristic) in one geometry traversal.

    All positive-overlap cells participate in SA, INCLUDING empty cells. Each
    cell's weight is its area-overlap fraction divided by the sum of fractions,
    never its point count. This is not a certified numerical error estimate.
    """
    alpha = float(alpha)
    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError('SA alpha must be finite and positive')
    weight, counts = _sa_query_overlap(q, hist)
    estimate = float(np.sum(weight * counts))
    ratios = weight[weight > 0]
    if not len(ratios):
        return (estimate, 0.0)
    errors = 1.0 - betainc(alpha, alpha, 0.5 + ratios / 2) + betainc(alpha, alpha, 0.5 - ratios / 2)
    score = float(np.sum(ratios * errors) / np.sum(ratios))
    if not np.isfinite(estimate) or not np.isfinite(score):
        raise ValueError('Nonfinite histogram estimate or SA score')
    return (estimate, score)
'''

In [ ]:
# Embedded helper: evaluate.py
EMBEDDED19['evaluate.py'] = r'''"""Frozen fixed-H evaluation only; no fitting, method selection or gate calibration."""
from pathlib import Path
import hashlib
import json
import math
import numpy as np
import pandas as pd
EPSILON = 0.25
MIN_EXACT = 10
COVERAGES = (0.1, 0.25, 0.5, 1.0)

def relative_error(answer, truth):
    h, t = np.broadcast_arrays(np.asarray(answer, float), np.asarray(truth, float))
    if not np.isfinite(h).all() or not np.isfinite(t).all() or (h < 0).any() or (t < 0).any():
        raise ValueError('Counts must be finite and nonnegative')
    out = np.full(h.shape, np.inf, float)
    np.divide(np.abs(h - t), t, out=out, where=t > 0)
    return np.where((h == 0) & (t == 0), 0.0, out)

def predicted_error_from_count(answer, reference_count):
    """17's common finite proxy mapping; no truth is an argument.

    This is a score definition, not an uncertainty guarantee. Actual error at
    zero truth is still infinite when H>0. The floor is1, never min_exact10.
    """
    h, t = np.broadcast_arrays(np.asarray(answer, float), np.asarray(reference_count, float))
    if not np.isfinite(h).all() or not np.isfinite(t).all() or (h < 0).any() or (t < 0).any():
        raise ValueError('Original and reference counts must be finite and nonnegative')
    return np.abs(h - t) / np.maximum(t, 1.0)

def auc_bad(labels, scores):
    labels = np.asarray(labels, bool)
    scores = np.asarray(scores, float)
    if labels.shape != scores.shape or np.isnan(scores).any():
        raise ValueError('Bad-label and score vectors must align without NaNs')
    positive = int(labels.sum())
    negative = len(labels) - positive
    if not positive or not negative:
        return np.nan
    ranks = pd.Series(scores).rank(method='average').to_numpy(float)
    return float((ranks[labels].sum() - positive * (positive + 1) / 2) / (positive * negative))

def fixed_coverage(errors, scores, fractions=COVERAGES):
    """No operational gate: complete ties at predeclared diagnostic fractions."""
    errors = np.asarray(errors, float)
    scores = np.asarray(scores, float)
    if errors.shape != scores.shape or errors.ndim != 1 or np.isnan(errors).any() or np.isnan(scores).any():
        raise ValueError('One aligned non-NaN error and reliability score per query is required')
    n = len(errors)
    rows = []
    masks = {}
    for fraction in fractions:
        if not 0 < fraction <= 1:
            raise ValueError('Coverage fractions must be in(0,1]')
        if n:
            rank = max(1, int(math.ceil(fraction * n)))
            cutoff = float(np.partition(scores, rank - 1)[rank - 1])
            selected = scores <= cutoff
        else:
            rank = 0
            cutoff = np.nan
            selected = np.zeros(0, bool)
        count = int(selected.sum())
        bad = int((errors[selected] > EPSILON).sum())
        masks[float(fraction)] = selected
        rows.append(dict(n_queries=n, nominal_coverage=float(fraction), nominal_count=rank, score_cutoff=cutoff, selected=count, violations=bad, actual_coverage=count / n if n else np.nan, violation_rate=bad / count if count else np.nan, complete_ties=True))
    return (pd.DataFrame(rows), masks)

def _normal_queries(querydf):
    frame = querydf.copy().reset_index(drop=True)
    if 'query_id' not in frame and 'qId' in frame:
        frame['query_id'] = frame.qId.astype(str)
    if 'H' not in frame and 'answer' in frame:
        frame['H'] = frame.answer
    if not {'query_id', 'truth', 'H'}.issubset(frame):
        raise ValueError('querydf needs query_id/qId,truth,H/answer')
    frame['truth'] = pd.to_numeric(frame.truth, errors='raise')
    frame['H'] = pd.to_numeric(frame.H, errors='raise')
    frame['query_id'] = frame.query_id.astype(str)
    for column, default in [('family', 'old_test'), ('seed', 0), ('cell', None), ('base_rate', 0.0)]:
        if column not in frame:
            frame[column] = frame.query_id if column == 'cell' else default
    frame['cell'] = frame.cell.astype(str)
    if frame[['query_id', 'truth', 'H', 'family', 'seed', 'cell', 'base_rate']].isna().any().any():
        raise ValueError('Query identities/counts must not be missing')
    if frame.duplicated(['family', 'seed', 'base_rate', 'query_id']).any():
        raise ValueError('Duplicate physical query/model membership')
    if (frame.groupby(['family', 'base_rate', 'query_id'])[['H', 'truth', 'cell']].nunique() > 1).any().any():
        raise ValueError('Repeated seed appearances must retain identical H,truth and cell')
    frame['actual_rel_error'] = relative_error(frame.H, frame.truth)
    frame['bad'] = frame.actual_rel_error > EPSILON
    frame['eligible'] = frame.truth >= MIN_EXACT
    return frame

def _extended_quantile_nonnegative(values, fraction):
    """Linear quantile with mathematically defined nonnegative infinity tails."""
    array = np.sort(np.asarray(values, float))
    if not len(array):
        return np.nan
    position = (len(array) - 1) * fraction
    lo = int(np.floor(position))
    hi = int(np.ceil(position))
    if lo == hi or array[lo] == array[hi]:
        return float(array[lo])
    if np.isinf(array[hi]):
        return np.inf
    return float(array[lo] + (position - lo) * (array[hi] - array[lo]))

def evaluate_predictions(querydf, scores_by_method):
    """Return tables, never choose methods or change scores.

    scores_by_method[name] is a vector, or {'score':vector,
    'predicted_error':vector}. Optional 'reference_count' is preserved. Vectors
    follow querydf row order. All fixed-H numerical metrics use truth>=10.
    """
    queries = _normal_queries(querydf)
    predictions = []
    for method, values in scores_by_method.items():
        info = values if isinstance(values, dict) else {'score': values}
        if {'H', 'answer', 'truth'} & set(info):
            raise ValueError('Methods cannot replace the fixed answer or truth')
        score = np.asarray(info['score'], float)
        if score.shape != (len(queries),) or np.isnan(score).any():
            raise ValueError('Each method must score every identical query without NaNs')
        part = queries.copy()
        part['method'] = str(method)
        part['score'] = score
        numeric = info.get('predicted_error')
        if numeric is None:
            part['predicted_error'] = np.nan
            part['numeric_error_prediction'] = False
        else:
            numeric = np.asarray(numeric, float)
            if numeric.shape != score.shape or np.isnan(numeric).any() or (numeric < 0).any():
                raise ValueError('Predicted relative errors must be aligned nonnegative non-NaN values')
            part['predicted_error'] = numeric
            part['numeric_error_prediction'] = True
        if 'reference_count' in info:
            reference = np.asarray(info['reference_count'], float)
            if reference.shape != score.shape or not np.isfinite(reference).all() or (reference < 0).any():
                raise ValueError('Reference counts must be aligned finite nonnegative values')
            part['reference_count'] = reference
        predictions.append(part)
    if not predictions:
        raise ValueError('No evaluator supplied')
    perquery = pd.concat(predictions, ignore_index=True)
    metrics = []
    coverage = []
    diagnostics = []
    groupcols = ['family', 'seed', 'base_rate', 'method']
    for keys, allpart in perquery.groupby(groupcols, sort=True, dropna=False):
        identity = dict(zip(groupcols, keys))
        part = allpart[allpart.eligible]
        n = len(part)
        row = dict(identity, n_queries=n, unique_queries=part.query_id.nunique(), generating_groups=part.cell.nunique(), bad_fraction=float(part.bad.mean()) if n else np.nan, auc_bad=auc_bad(part.bad, part.score), mae_pp=np.nan, p90_ae_pp=np.nan, bias_pp=np.nan, rmse_pp=np.nan, numeric_error_prediction=bool(allpart.numeric_error_prediction.all()), infinite_predicted_errors=int(np.isinf(part.predicted_error).sum()))
        if n and row['numeric_error_prediction']:
            residual = part.predicted_error.to_numpy(float) - part.actual_rel_error.to_numpy(float)
            gap = np.abs(residual)
            row.update(mae_pp=float(100 * np.mean(gap)), p90_ae_pp=float(100 * _extended_quantile_nonnegative(gap, 0.9)), bias_pp=float(100 * np.mean(residual)), rmse_pp=float(100 * np.sqrt(np.mean(residual ** 2))))
        metrics.append(row)
        cov, masks = fixed_coverage(part.actual_rel_error, part.score)
        for _, value in cov.iterrows():
            coverage.append(dict(identity, **value.to_dict()))
        for fraction, mask in masks.items():
            perquery.loc[part.index, 'selected_' + str(int(100 * fraction))] = mask
        zeros = allpart[allpart.truth == 0]
        diagnostics.append(dict(identity, all_candidates=len(allpart), eligible=n, excluded_below10=int((~allpart.eligible).sum()), zero_truth=len(zeros), zero_H_zero=int((zeros.H == 0).sum()), zero_H_positive=int((zeros.H > 0).sum()), all_candidate_auc_bad=auc_bad(allpart.bad, allpart.score), numerical_MAE_all_candidates='undefined if any actual error is infinite; never silently drop zero cases'))
    for fraction in COVERAGES:
        name = 'selected_' + str(int(100 * fraction))
        perquery[name] = perquery[name].astype('boolean')
    return dict(metrics=pd.DataFrame(metrics), fixed_coverage=pd.DataFrame(coverage), all_candidate_diagnostics=pd.DataFrame(diagnostics), per_query=perquery)

def paired_group_bootstrap(per_query, method_a, method_b, repeats=2000, seed=19001):
    """Conditional descriptive CIs; average seed appearances before cell draws.

    Positive MAE/risk delta means A is worse. AUC is supplied as a descriptive
    mean of matched seed deltas only, without claiming independent-seed CIs.
    """
    if repeats < 0:
        raise ValueError('Bootstrap repeats must be nonnegative')
    output = []
    eligible = per_query[per_query.eligible]
    for (family, rate), part in eligible.groupby(['family', 'base_rate'], sort=True):
        a = part[part.method == method_a]
        b = part[part.method == method_b]
        keys = ['seed', 'query_id']
        paired = a.merge(b, on=keys, suffixes=('_a', '_b'), validate='one_to_one')
        if len(paired) != len(a) or len(paired) != len(b):
            raise ValueError('Paired methods must have every same query membership')
        if not len(paired):
            continue
        for field in ('truth', 'H', 'actual_rel_error', 'cell'):
            if not paired[field + '_a'].equals(paired[field + '_b']):
                raise ValueError('Paired evaluators have different fixed-H targets')
        base = dict(family=family, base_rate=rate, method_a=method_a, method_b=method_b, conditional_scope='descriptive generating-cell bootstrap; seed appearances averaged per query; fitted models fixed; geography may overlap')
        work = pd.DataFrame({'query_id': paired.query_id, 'cell': paired.cell_a})
        stats = []
        if paired.numeric_error_prediction_a.all() and paired.numeric_error_prediction_b.all():
            work['mae_delta_pp'] = 100 * (np.abs(paired.predicted_error_a - paired.actual_rel_error_a) - np.abs(paired.predicted_error_b - paired.actual_rel_error_b))
            if np.isfinite(work.mae_delta_pp).all():
                stats.append(('mae_delta_pp', 'mean', 'mae_delta_pp'))
        for fraction in COVERAGES:
            suffix = str(int(100 * fraction))
            sa = paired['selected_' + suffix + '_a'].astype(float)
            sb = paired['selected_' + suffix + '_b'].astype(float)
            work['selected_a_' + suffix] = sa
            work['selected_b_' + suffix] = sb
            work['bad_a_' + suffix] = sa * paired.bad_a.astype(float)
            work['bad_b_' + suffix] = sb * paired.bad_b.astype(float)
            stats.append(('violation_delta_pp_at_' + suffix, 'risk', suffix))
        averaged = work.groupby(['query_id', 'cell'], as_index=False).mean(numeric_only=True)
        numeric = [col for col in averaged if col not in ('query_id', 'cell')]
        groups = averaged.groupby('cell')[numeric].sum()
        groups['n_query'] = averaged.groupby('cell').size()
        values = groups.to_numpy(float)
        cols = {name: i for i, name in enumerate(groups.columns)}
        rng = np.random.default_rng(seed)
        draws = []
        for _ in range(repeats if len(groups) > 1 else 0):
            draws.append(values[rng.integers(0, len(groups), len(groups))].sum(axis=0))
        totals = values.sum(axis=0)

        def metric(total, kind, column):
            if kind == 'mean':
                return total[cols[column]] / total[cols['n_query']]
            na, nb = (total[cols['selected_a_' + column]], total[cols['selected_b_' + column]])
            return 100 * (total[cols['bad_a_' + column]] / na - total[cols['bad_b_' + column]] / nb) if na > 0 and nb > 0 else np.nan
        for name, kind, column in stats:
            samples = np.asarray([metric(draw, kind, column) for draw in draws], float)
            samples = samples[np.isfinite(samples)]
            output.append(dict(base, statistic=name, estimate=float(metric(totals, kind, column)), ci95_low=float(np.quantile(samples, 0.025)) if len(samples) else np.nan, ci95_high=float(np.quantile(samples, 0.975)) if len(samples) else np.nan, unique_queries=len(averaged), generating_groups=len(groups), bootstrap_repeats=len(samples)))
        auc_deltas = []
        for _, seedpart in paired.groupby('seed'):
            aa = auc_bad(seedpart.bad_a, seedpart.score_a)
            bb = auc_bad(seedpart.bad_b, seedpart.score_b)
            if np.isfinite(aa) and np.isfinite(bb):
                auc_deltas.append(aa - bb)
        output.append(dict(base, statistic='mean_matched_seed_auc_delta', estimate=float(np.mean(auc_deltas)) if auc_deltas else np.nan, ci95_low=np.nan, ci95_high=np.nan, unique_queries=len(averaged), generating_groups=len(groups), bootstrap_repeats=0))
    return pd.DataFrame(output)
'''

In [ ]:
# Embedded helper: confirm_portable.py
EMBEDDED19['confirm_portable.py'] = r'''"""Independent frozen-candidate confirmation on new raw-count query labels.

No fitting, iteration selection or candidate comparison/selection occurs here.
Actual labels cannot be generated without an explicit allow_label_generation
flag and a pre-existing frozen selection whose input/module hashes verify.
"""
from pathlib import Path
import hashlib
import json
import math
import time
from uuid import uuid4
import numpy as np
import pandas as pd
from .generation19 import generate_plan
from .prepare_data import load_hist, iter_raw, estimate_count, sha
from .runtime_features import coarse_source_contexts, source_contexts, describe_coarse_queries, describe_queries, AGG_FEATURE_NAMES
from .moment_metadata import load_moments_contexts, augment_records
from .analytic_moments_fast import estimate_reference
from .budget_fine import load_fine, fine_reference
from .resident_critic import ResidentCritic
from .evaluate import evaluate_predictions, paired_group_bootstrap, predicted_error_from_count
from .evaluate import relative_error
METHODS = ('beta_gaussian_copula', 'beta_independent', 'raw2_proxy', 'fine164_critic', 'fine186_critic', 'original_SA')
DEFAULT_FRESH_SEED = 2026111602

def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False, default=str) + '\n')

def _stable_ids(values, n, salt):
    return sorted(sorted(set(map(str, values)), key=lambda x: hashlib.sha256(f'{salt}:{x}'.encode()).digest())[:n])

def verify_selection(selection_path, cfg):
    path = Path(selection_path)
    if not path.is_file():
        raise FileNotFoundError('Freeze selection.json before confirmation')
    selection = json.loads(path.read_text())
    if selection.get('status') != 'frozen' or selection.get('chosen_method') != 'beta_gaussian_copula':
        raise ValueError('Only the pre-selected frozen beta_gaussian_copula is authorized')
    for key, default in [('fresh_query_seed', DEFAULT_FRESH_SEED), ('replication_cells', 1000), ('seeds', [0, 1, 2])]:
        if selection.get(key) != cfg.get(key, default):
            raise ValueError('Confirmation config differs from frozen selection: ' + key)
    artifacts = selection.get('artifacts')
    if not isinstance(artifacts, list) or not artifacts:
        raise ValueError('Frozen selection needs module, payload and model artifact hashes')
    seen = set()
    for item in artifacts:
        artifact = Path(item['path']).resolve()
        if not artifact.is_file() or sha(artifact) != item['sha256']:
            raise ValueError('Frozen artifact missing/changed: ' + str(artifact))
        if str(artifact) in seen:
            raise ValueError('Duplicate frozen artifact path')
        seen.add(str(artifact))
    return (selection, sha(path))

def _load_sa_functions(cfg):
    import importlib.util
    spec = importlib.util.spec_from_file_location('confirmation_frozen_sa', cfg['sa_core_file'])
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

class ResidentProviders:
    """Runtime state contains no raw path, query identity, label or saved score."""

    def __init__(self, cfg):
        self.initialization = []
        self.costs = []
        self.seeds = list(cfg['seeds'])
        tick = time.perf_counter()
        self.base = load_hist(cfg['base_histogram_file'])
        self.root = (self.base[1][0], self.base[1][-1], self.base[2][0], self.base[2][-1])
        self.sa = _load_sa_functions(cfg)
        self.alpha = self.sa.sa_alpha(self.base)
        self.initialization.append(dict(method='shared_original_H_and_SA', stage='load_histogram_and_alpha', seconds=time.perf_counter() - tick))
        tick = time.perf_counter()
        self.coarse = coarse_source_contexts(cfg['coarse_dir'])
        self.moments = load_moments_contexts(cfg['moment_dir'])
        if set(self.moments) != {int(context['source_id']) for context in self.coarse}:
            raise ValueError('Coarse and moment source identities differ')
        for context in self.coarse:
            _, bx, by, counts = context['hist']
            item = self.moments[int(context['source_id'])]
            if item['root'] != (bx[0], bx[-1], by[0], by[-1]) or (item['nx'], item['ny']) != counts.shape:
                raise ValueError('Moment payload geometry differs from exact coarse source')
        self.initialization.append(dict(method='analytic_moments', stage='load_coarse_and_moment_payloads', seconds=time.perf_counter() - tick))
        self.resident_critic = None
        if cfg.get('use_optimized_resident', True):
            tick = time.perf_counter()
            self.resident_critic = ResidentCritic(self.coarse, self.moments, root=self.root, cache_quadrature=False)
            self.initialization.append(dict(method='analytic_moments', stage='build_geometry_index_and_distribution_parameters', seconds=time.perf_counter() - tick, resident_numpy_bytes=self.resident_critic.resident_numpy_bytes, memory_scope='Named retained NumPy buffers in critic only; excludes Python objects, temporaries and other comparison arms; not RSS'))
        tick = time.perf_counter()
        self.raw2 = source_contexts(cfg['histogram_dir'], cfg['raw2_dir'], validate_exports=False)
        self.initialization.append(dict(method='raw2_proxy_and_Haware_control', stage='load_raw2_derive_coarse', seconds=time.perf_counter() - tick))
        self.fine = {}
        for name, key in [('fine164_critic', 'fine164_path'), ('fine186_critic', 'fine186_path')]:
            tick = time.perf_counter()
            self.fine[name] = load_fine(cfg[key])
            expected_n = int(cfg.get('fine164_n', 164) if name == 'fine164_critic' else cfg.get('fine186_n', 186))
            if self.fine[name]['n'] != expected_n or self.fine[name]['root'] != self.root:
                raise ValueError('Fine critic geometry/resolution differs from frozen method name')
            self.initialization.append(dict(method=name, stage='load_fine_payload', seconds=time.perf_counter() - tick))
        self.direct_fine = None
        if str(cfg.get('fine247_path', '')).strip():
            tick = time.perf_counter()
            self.direct_fine = load_fine(cfg['fine247_path'])
            if self.direct_fine['root'] != self.root or self.direct_fine['n'] != int(cfg.get('fine247_n', 247)):
                raise ValueError('Direct-answer fine control differs from frozen budget geometry')
            self.initialization.append(dict(method='fine_total_direct_answer', stage='load_direct_answer_quality_control', seconds=time.perf_counter() - tick))
        self.models = {}
        self.schemas = {}
        self.methods = METHODS + ('aggregate_relative_l1',) if str(cfg.get('aggregate_dir', '')).strip() else METHODS
        if str(cfg.get('aggregate_dir', '')).strip():
            import lightgbm as lgb
        for seed in self.seeds if str(cfg.get('aggregate_dir', '')).strip() else []:
            folder = Path(cfg['aggregate_dir']) / f'aggregate_seed_{seed}' / 'aggregate_relative_l1'
            tick = time.perf_counter()
            schema = json.loads((folder / 'schema.json').read_text())
            if sha(folder / 'model.txt') != schema['model_sha256']:
                raise ValueError('H-aware control model weights changed')
            expected = list(AGG_FEATURE_NAMES) + ['candidate_log1pH', 'candidate_signed_raw2_disagreement']
            if schema['features'] != expected:
                raise ValueError('H-aware control feature order differs')
            self.models[seed] = lgb.Booster(model_file=str(folder / 'model.txt'))
            self.schemas[seed] = schema
            self.initialization.append(dict(method='aggregate_relative_l1', seed=seed, stage='load_model_and_schema', seconds=time.perf_counter() - tick))
        original = int(cfg.get('original_histogram_bytes', 137320))
        coarse_bytes = sum((p.stat().st_size for p in Path(cfg['coarse_dir']).glob('src_*.hist')))
        if not coarse_bytes:
            raise ValueError('Persisted coarse binary payloads required for actual-byte comparison')
        moments_bytes = sum((int(r['payload_bytes']) for r in self.moments.values()))
        raw2_bytes = sum((int(r['raw2_payload_bytes']) for r in self.raw2))
        self.candidate_auxiliary_bytes = coarse_bytes + moments_bytes
        self.original_histogram_bytes = original
        for method in self.methods:
            for seed in self.seeds:
                model = 0
                schema_bytes = 0
                if method in ('beta_independent', 'beta_gaussian_copula'):
                    aux = coarse_bytes + moments_bytes
                elif method in self.fine:
                    aux = self.fine[method]['serialized_bytes']
                elif method == 'original_SA':
                    aux = 8
                else:
                    aux = raw2_bytes
                if method == 'aggregate_relative_l1':
                    folder = Path(cfg['aggregate_dir']) / f'aggregate_seed_{seed}' / method
                    model = (folder / 'model.txt').stat().st_size
                    schema_bytes = (folder / 'schema.json').stat().st_size
                self.costs.append(dict(method=method, seed=seed, original_H_bytes=original, auxiliary_bytes=aux, model_weight_bytes=model, model_schema_bytes=schema_bytes, total_serialized_bytes=original + aux + model + schema_bytes, memory_scope='serialized bytes only; resident derived arrays and Python objects require more RAM'))
        if self.direct_fine is not None:
            self.costs.append(dict(method='fine_total_direct_answer', seed=-1, original_H_bytes=0, auxiliary_bytes=self.direct_fine['serialized_bytes'], model_weight_bytes=0, model_schema_bytes=0, total_serialized_bytes=self.direct_fine['serialized_bytes'], comparison_role='Separate direct-answer quality alternative; not a fixed-H critic', memory_scope='serialized bytes only; no original H is required for this alternative answer'))

    @staticmethod
    def direct_matrix(described, H):
        X = described['agg_X'].astype(float)
        S = described['qlevel'].S.to_numpy(float)
        return np.column_stack((X, np.log1p(H), (H - S) / np.maximum(S, 1.0)))

    def batch(self, rectangles):
        """Calculate all scores before raw labels exist; no supervised argument."""
        q = np.asarray(rectangles, float)
        H = np.asarray([self.sa.histogram_count(rect, self.base) for rect in q])
        described = describe_coarse_queries(q, self.coarse, self.root)
        aug = augment_records(described, self.moments)
        static = {}
        bounds = []
        for variant in ('beta_gaussian_copula', 'beta_independent'):
            reference = estimate_reference(described['X'][:, :4], aug['moment_features'], described['mass'], described['query_index'], described['full_mass'], variant)
            static[variant] = dict(score=predicted_error_from_count(H, reference), predicted_error=predicted_error_from_count(H, reference), reference_count=reference)
        source = describe_queries(q, self.raw2, self.root)
        S = source['qlevel'].S.to_numpy(float)
        static['raw2_proxy'] = dict(score=predicted_error_from_count(H, S), predicted_error=predicted_error_from_count(H, S), reference_count=S)
        for name, hist in self.fine.items():
            reference = fine_reference(q, hist)
            static[name] = dict(score=predicted_error_from_count(H, reference), predicted_error=predicted_error_from_count(H, reference), reference_count=reference)
        static['original_SA'] = dict(score=np.asarray([self.sa.query_answer_sa(rect, self.base, self.alpha)[1] for rect in q]))
        scores = {}
        X = self.direct_matrix(source, H)
        exact = source['qlevel'].L_fine.to_numpy(float) == source['qlevel'].U_fine.to_numpy(float)
        exact_score = predicted_error_from_count(H, source['qlevel'].L_fine)
        for seed in self.seeds:
            scores[seed] = dict(static)
            if seed in self.models:
                score = np.maximum(self.models[seed].predict(X, num_threads=2), 0.0)
                score[exact] = exact_score[exact]
                scores[seed]['aggregate_relative_l1'] = dict(score=score, predicted_error=score)
        return (H, scores, described['qlevel'][['L', 'U']].copy())

    def call(self, rectangle, method, seed):
        """Live H, feature construction, inference and score; no cached lookup."""
        start = time.perf_counter()
        q = np.asarray(rectangle, float)
        stages = {}
        if method == 'original_SA':
            H, score = self.sa.query_answer_sa(q, self.base, self.alpha)
            stages['estimate_and_SA_seconds'] = time.perf_counter() - start
            return (float(H), float(score), np.nan, stages)
        tick = time.perf_counter()
        H = self.sa.histogram_count(q, self.base)
        stages['H_estimate_seconds'] = time.perf_counter() - tick
        if method in ('beta_gaussian_copula', 'beta_independent'):
            if self.resident_critic is not None:
                tick = time.perf_counter()
                reference = float(self.resident_critic.query(q, variant=method)['reference_count'])
                stages['indexed_features_and_reference_seconds'] = time.perf_counter() - tick
            else:
                tick = time.perf_counter()
                described = describe_coarse_queries(q.reshape(1, 4), self.coarse, self.root)
                aug = augment_records(described, self.moments)
                stages['feature_seconds'] = time.perf_counter() - tick
                tick = time.perf_counter()
                reference = float(estimate_reference(described['X'][:, :4], aug['moment_features'], described['mass'], described['query_index'], described['full_mass'], method)[0])
                stages['inference_seconds'] = time.perf_counter() - tick
        elif method in self.fine:
            tick = time.perf_counter()
            reference = float(fine_reference(q, self.fine[method]))
            stages['fine_reference_seconds'] = time.perf_counter() - tick
        elif method == 'raw2_proxy':
            tick = time.perf_counter()
            reference = float(sum((estimate_count(q, context['raw2']) for context in self.raw2)))
            stages['raw2_reference_seconds'] = time.perf_counter() - tick
        elif method == 'aggregate_relative_l1':
            tick = time.perf_counter()
            described = describe_queries(q.reshape(1, 4), self.raw2, self.root)
            X = self.direct_matrix(described, np.asarray([H]))
            stages['feature_seconds'] = time.perf_counter() - tick
            tick = time.perf_counter()
            score = float(max(self.models[int(seed)].predict(X, num_threads=2)[0], 0.0))
            row = described['qlevel'].iloc[0]
            if row.L_fine == row.U_fine:
                score = float(predicted_error_from_count(H, row.L_fine))
            stages['inference_seconds'] = time.perf_counter() - tick
            return (float(H), score, np.nan, stages)
        else:
            raise ValueError('Unknown frozen confirmation method')
        tick = time.perf_counter()
        score = float(predicted_error_from_count(H, reference))
        stages['score_mapping_seconds'] = time.perf_counter() - tick
        return (float(H), score, float(reference), stages)

def prepare_plan(cfg, providers, destination, selection_sha):
    prior17 = json.loads((Path(cfg['source17_dir']) / 'run.json').read_text())
    if sha(Path(cfg['source15_dir']) / 'run.json') != prior17['source15_manifest_sha256']:
        raise ValueError('Changed source15 identity before new geometry')
    for name in ('outer_splits.csv', 'raw_input_manifest.json'):
        if sha(Path(cfg['source15_dir']) / name) != prior17['source15_artifact_sha256'][name]:
            raise ValueError('Changed original source15 artifact: ' + name)
    if sha(cfg['queries_file']) != prior17['verified_inputs']['data/queries-Ans.txt']['sha256']:
        raise ValueError('Changed original training geometry input')
    if sha(Path(cfg['source17_dir']) / 'new_queries.csv') != prior17['new_query_labels_sha256']:
        raise ValueError('Changed inspected17 forbidden geometry')
    old = pd.read_csv(cfg['queries_file'], header=None, usecols=[0, 1, 2, 3]).to_numpy(float)
    inspected = pd.read_csv(Path(cfg['source17_dir']) / 'new_queries.csv', usecols=['x0', 'x1', 'y0', 'y1'], float_precision='round_trip').to_numpy(float)
    outer = pd.read_csv(Path(cfg['source15_dir']) / 'outer_splits.csv')
    plan, membership, audit = generate_plan(old, [c['hist'] for c in providers.coarse], [int(c['source_id']) for c in providers.coarse], outer, cfg['seeds'], query_seed=cfg['fresh_query_seed'], replication_cells=cfg['replication_cells'], extra_forbidden_geometry=inspected)
    plan.to_csv(destination / 'query_plan.csv', index=False, float_format='%.17g')
    membership.to_csv(destination / 'query_membership.csv', index=False)
    save_json(destination / 'generation_audit.json', audit)
    lock = dict(selection_sha256=selection_sha, query_plan_sha256=sha(destination / 'query_plan.csv'), membership_sha256=sha(destination / 'query_membership.csv'), query_seed=cfg['fresh_query_seed'], methods=list(providers.methods), geometry_hashes_before_truth=True, replication_cells=cfg['replication_cells'], old_geometry_file_sha256=sha(cfg['queries_file']), extra_forbidden17_sha256=sha(Path(cfg['source17_dir']) / 'new_queries.csv'), scope=audit['scope'], min_exact=10, no_refill_after_truth=True, model_training=False, gate_fitting=False, selected_method='beta_gaussian_copula', selection_uses_new_truth=False, independent_confirmation_code_sha256={str(p.resolve()): sha(p) for p in (Path(__file__), Path(__file__).with_name('generation19.py'), Path(cfg['exact_counter_file']))})
    save_json(destination / 'protocol_lock.json', lock)
    return (plan, membership, lock)

def save_scores_before_labels(plan, membership, providers, destination):
    H, scores, bounds = providers.batch(plan[['x0', 'x1', 'y0', 'y1']].to_numpy(float))
    index = dict(zip(plan.query_id, range(len(plan))))
    records = []
    for member in membership.itertuples(index=False):
        position = index[member.query_id]
        for method, values in scores[int(member.seed)].items():
            records.append(dict(query_id=member.query_id, family=member.family, cell=member.cell, seed=int(member.seed), method=method, base_rate=float(providers.base[0]), H=float(H[position]), score=float(values['score'][position]), predicted_error=float(values['predicted_error'][position]) if 'predicted_error' in values else np.nan, reference_count=float(values['reference_count'][position]) if 'reference_count' in values else np.nan))
    pd.DataFrame(records).to_csv(destination / 'scores_before_labels.csv', index=False, float_format='%.17g')
    plan[['query_id']].join(bounds).to_csv(destination / 'coarse_source_bounds.csv', index=False, float_format='%.17g')
    if providers.direct_fine is not None:
        direct = fine_reference(plan[['x0', 'x1', 'y0', 'y1']].to_numpy(float), providers.direct_fine)
        pd.DataFrame(dict(query_id=plan.query_id, reference_count=direct)).to_csv(destination / 'direct_answer_before_labels.csv', index=False, float_format='%.17g')
        save_json(destination / 'direct_answer_control_lock.json', dict(reference_sha256=sha(destination / 'direct_answer_before_labels.csv'), direct_side=providers.direct_fine['n'], direct_answer_bytes=providers.direct_fine['serialized_bytes'], candidate_auxiliary_bytes=providers.candidate_auxiliary_bytes, original_H_bytes=providers.original_histogram_bytes, predictions_saved_before_new_labels=True, scope='Separate direct-answer alternative using full critic storage cap; no H required for its answer'))
    return sha(destination / 'scores_before_labels.csv')

def build_labels(cfg, plan, destination):
    import importlib.util
    spec = importlib.util.spec_from_file_location('confirmation_exact_raw_counter', cfg['exact_counter_file'])
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    raw_manifest = json.loads((Path(cfg['source15_dir']) / 'raw_input_manifest.json').read_text())
    parity = pd.read_csv(Path(cfg['source15_dir']) / 'raw_source_parity.csv').set_index('source_id')
    paths = []
    expected = []
    for record in raw_manifest:
        path = Path(cfg['raw_data_dir']) / Path(record['raw_path']).name
        if sha(path) != record['raw_sha256']:
            raise ValueError('Raw identity mismatch before new truth')
        paths.append(path)
        expected.append(int(parity.loc[int(record['source_id']), 'raw_rows']))
    chosen = set(_stable_ids(plan.query_id, 32, '19-exact-brute-audit-v1'))
    indices = np.flatnonzero(plan.query_id.isin(chosen))
    tick = time.perf_counter()
    truth, diagnostics, audits = module.count_raw_rectangles(paths, plan[['x0', 'x1', 'y0', 'y1']].to_numpy(float), iter_raw, expected_rows=expected, audit_indices=indices, leafsize=128)
    for path, record in zip(paths, raw_manifest):
        if sha(path) != record['raw_sha256']:
            raise ValueError('Raw identity changed during new truth')
    pd.DataFrame(diagnostics).to_csv(destination / 'exact_label_costs.csv', index=False)
    audit = pd.DataFrame(audits)
    audit['query_id'] = plan.iloc[audit.query_index].query_id.to_numpy()
    audit.to_csv(destination / 'exact_label_audit.csv', index=False)
    if not audit.matches.all():
        raise ValueError('Independent raw brute audit failed')
    labels = plan.assign(truth=truth, eligible=truth >= 10, zero_truth=truth == 0)
    labels.to_csv(destination / 'new_queries.csv', index=False, float_format='%.17g')
    labels[~labels.eligible].assign(reason='truth<10; no replacement').to_csv(destination / 'excluded_queries.csv', index=False)
    return (labels, time.perf_counter() - tick)

def evaluate_frozen_scores(labels, membership, destination, score_sha, method_names=METHODS):
    if sha(destination / 'scores_before_labels.csv') != score_sha:
        raise ValueError('Scores changed after accessing new truth')
    scores = pd.read_csv(destination / 'scores_before_labels.csv', float_precision='round_trip')
    tables = {}
    outputs = []
    for seed in sorted(membership.seed.unique()):
        querydf = membership[membership.seed == seed].merge(labels[['query_id', 'truth']], on='query_id', validate='one_to_one')
        byseed = scores[scores.seed == seed]
        bymethod = {m: p.set_index('query_id') for m, p in byseed.groupby('method')}
        querydf['H'] = bymethod['original_SA'].loc[querydf.query_id, 'H'].to_numpy(float)
        querydf['base_rate'] = bymethod['original_SA'].loc[querydf.query_id, 'base_rate'].to_numpy(float)
        methods = {}
        for method in method_names:
            part = bymethod[method].loc[querydf.query_id]
            if not np.array_equal(part.H.to_numpy(), querydf.H.to_numpy()):
                raise ValueError('A method changed fixed H')
            methods[method] = dict(score=part.score.to_numpy(float))
            if method != 'original_SA':
                methods[method]['predicted_error'] = part.predicted_error.to_numpy(float)
            if part.reference_count.notna().all():
                methods[method]['reference_count'] = part.reference_count.to_numpy(float)
        result = evaluate_predictions(querydf, methods)
        for name, frame in result.items():
            tables.setdefault(name, []).append(frame)
    result = {name: pd.concat(parts, ignore_index=True) for name, parts in tables.items()}
    for name, frame in result.items():
        frame.to_csv(destination / (name + '.csv'), index=False, float_format='%.17g')
    answer_rows = []
    pq = result['per_query']
    reference_methods = [method for method, part in pq.groupby('method') if 'reference_count' in part and part.reference_count.notna().all()]
    direct_parts = []
    for method in reference_methods:
        part = pq[pq.method == method].copy()
        part['direct_answer'] = part.reference_count
        direct_parts.append(part)
    original = pq[pq.method == 'original_SA'].copy()
    original['method'] = 'original_H'
    original['direct_answer'] = original.H
    direct_parts.append(original)
    for (family, seed, method), allpart in pd.concat(direct_parts, ignore_index=True).groupby(['family', 'seed', 'method']):
        part = allpart[allpart.eligible]
        errors = relative_error(part.direct_answer, part.truth)
        zeros = allpart[allpart.truth == 0]
        answer_rows.append(dict(family=family, seed=seed, method=method, n_queries=len(part), within25_count=int((errors <= 0.25).sum()), within25_fraction=float(np.mean(errors <= 0.25)) if len(part) else np.nan, mean_actual_relative_error=float(np.mean(errors)) if len(part) else np.nan, median_actual_relative_error=float(np.median(errors)) if len(part) else np.nan, p95_actual_relative_error=float(np.quantile(errors, 0.95)) if len(part) else np.nan, mean_absolute_count_error=float(np.mean(np.abs(part.direct_answer - part.truth))) if len(part) else np.nan, zero_truth_count=len(zeros), zero_truth_predicted_positive=int((zeros.direct_answer > 0).sum()), answer_is_original_H=method == 'original_H', scope='Separate direct-answer alternative; primary evaluator metrics still assess original fixed H'))
    pd.DataFrame(answer_rows).to_csv(destination / 'reference_answer_quality.csv', index=False)
    direct_lock = destination / 'direct_answer_control_lock.json'
    if direct_lock.is_file():
        lock = json.loads(direct_lock.read_text())
        protocol = json.loads((destination / 'protocol_lock.json').read_text())
        if sha(direct_lock) != protocol.get('direct_answer_control_lock_sha256'):
            raise ValueError('Direct-answer lock differs from the pre-label protocol')
        if sha(destination / 'direct_answer_before_labels.csv') != lock['reference_sha256']:
            raise ValueError('Direct-answer control predictions changed after labels')
        direct = pd.read_csv(destination / 'direct_answer_before_labels.csv', float_precision='round_trip')
        selected = scores[scores.method == 'beta_gaussian_copula'][['query_id', 'H', 'reference_count']].drop_duplicates()
        if selected.query_id.duplicated().any():
            raise ValueError('Deterministic reference differs across seed memberships')
        frame = labels[['query_id', 'family', 'truth']].merge(selected, on='query_id', validate='one_to_one').merge(direct.rename(columns={'reference_count': 'fine_direct'}), on='query_id', validate='one_to_one')
        rows = []
        for family, part in frame.groupby('family'):
            for name, column, payload_bytes in (('original_H', 'H', lock['original_H_bytes']), ('selected_reference_as_answer', 'reference_count', lock['candidate_auxiliary_bytes']), ('fine_total_direct_answer', 'fine_direct', lock['direct_answer_bytes'])):
                eligible = part[part.truth >= 10]
                err = relative_error(eligible[column], eligible.truth)
                rows.append(dict(family=family, method=name, unique_queries=len(eligible), answer_payload_bytes=payload_bytes, mean_actual_relative_error=float(np.mean(err)) if len(err) else np.nan, median_actual_relative_error=float(np.median(err)) if len(err) else np.nan, within25_count=int((err <= 0.25).sum()), within25_fraction=float(np.mean(err <= 0.25)) if len(err) else np.nan, zero_truth_count=int((part.truth == 0).sum()), zero_truth_positive_answer=int(((part.truth == 0) & (part[column] > 0)).sum()), scope='Same unique physical queries; no repeated seeds. Direct answer comparison only, separate from fixed-H assessment.'))
        pd.DataFrame(rows).to_csv(destination / 'direct_answer_control.csv', index=False)
    pairs = []
    for comparator in ('raw2_proxy', 'beta_independent', 'fine164_critic', 'fine186_critic', 'aggregate_relative_l1'):
        if comparator not in method_names:
            continue
        pairs.append(paired_group_bootstrap(result['per_query'], 'beta_gaussian_copula', comparator))
    pd.concat(pairs, ignore_index=True).to_csv(destination / 'paired_effects.csv', index=False)
    bounds = pd.read_csv(destination / 'coarse_source_bounds.csv').merge(labels[['query_id', 'truth']], on='query_id', validate='one_to_one')
    bounds['truth_inside_coarse_source_bounds'] = (bounds.L <= bounds.truth) & (bounds.truth <= bounds.U)
    bounds.to_csv(destination / 'bounds_audit.csv', index=False)
    if not bounds.truth_inside_coarse_source_bounds.all():
        raise ValueError('Exact-source coarse count bounds exclude truth')
    return result

def time_live(cfg, plan, result, providers, destination):
    qbyid = plan.set_index('query_id')[['x0', 'x1', 'y0', 'y1']]
    positive = result['per_query'][result['per_query'].eligible]
    expected = positive.set_index(['family', 'seed', 'query_id', 'method'])[['H', 'score']]
    timings = []
    selected = []
    for (family, seed), part in positive.groupby(['family', 'seed'], sort=True):
        ids = _stable_ids(part.query_id, int(cfg.get('timing_queries', 30)), '19-common-live-timing-v1')
        for query_id in ids:
            for method in providers.methods:
                providers.call(qbyid.loc[query_id].to_numpy(float), method, seed)
        selected.extend((dict(family=family, seed=seed, query_id=qid) for qid in ids))
        for repeat in range(int(cfg.get('timing_repeats', 5))):
            calls = [(method, qid) for qid in ids for method in providers.methods]
            calls.sort(key=lambda x: hashlib.sha256(f'19-live-order-v1:{family}:{seed}:{repeat}:{x}'.encode()).digest())
            for order, (method, qid) in enumerate(calls):
                q = qbyid.loc[qid].to_numpy(float)
                tick = time.perf_counter()
                H, score, reference, stages = providers.call(q, method, seed)
                seconds = time.perf_counter() - tick
                target = expected.loc[family, seed, qid, method]
                if not np.isclose(H, target.H, rtol=1e-10, atol=1e-08) or not np.isclose(score, target.score, rtol=1e-08, atol=1e-08):
                    raise ValueError(f'Live output differs from pre-label batch: {family}/{seed}/{qid}/{method}: {score} vs {target.score}')
                timings.append(dict(family=family, seed=seed, method=method, query_id=qid, repeat=repeat, arm_order=order, seconds=seconds, cache='resident summaries/models; everyquery/arm warmed; no raw/pilot/truth', **stages))
    frame = pd.DataFrame(timings)
    frame.to_csv(destination / 'resident_timings.csv', index=False)
    pd.DataFrame(selected).to_csv(destination / 'timing_query_ids.csv', index=False)
    rows = []
    for keys, part in frame.groupby(['family', 'seed', 'method']):
        rows.append(dict(zip(['family', 'seed', 'method'], keys), n_calls=len(part), unique_queries=part.query_id.nunique(), mean_ms=float(1000 * part.seconds.mean()), p95_ms=float(1000 * part.seconds.quantile(0.95))))
    pd.DataFrame(rows).to_csv(destination / 'resident_timing_summary.csv', index=False)

def run_confirmation(selection_path, cfg, allow_label_generation=False):
    if not allow_label_generation:
        raise PermissionError('New label generation remains locked until parent explicitly authorizes GO')
    selection, selection_sha = verify_selection(selection_path, cfg)
    destination = Path(cfg['output_root']) / ('moment-fresh-confirmation-' + uuid4().hex[:12])
    destination.mkdir(parents=True, exist_ok=False)
    state = dict(status='running', study='moment-fresh-confirmation', config=cfg, selection_path=str(selection_path), selection_sha256=selection_sha, selected_method=selection['chosen_method'], no_training=True, no_retuning=True, scope='fresh query geometry/labels; same raw data and already inspected generating populations; no universal/newdataset claim')
    start = time.perf_counter()
    save_json(destination / 'run.json', state)
    try:
        providers = ResidentProviders(cfg)
        pd.DataFrame(providers.initialization).to_csv(destination / 'initialization.csv', index=False)
        pd.DataFrame(providers.costs).to_csv(destination / 'payload_costs.csv', index=False)
        pd.DataFrame([] if 'aggregate_relative_l1' in providers.methods else [dict(method='aggregate_relative_l1', reason='aggregate_dir blank; optional saved H-aware control not supplied; no replacement model trained')], columns=['method', 'reason']).to_csv(destination / 'skipped_baselines.csv', index=False)
        state['evaluated_methods'] = list(providers.methods)
        state['optional_ML_control_supplied'] = 'aggregate_relative_l1' in providers.methods
        plan, membership, lock = prepare_plan(cfg, providers, destination, selection_sha)
        state['score_sha256_before_labels'] = save_scores_before_labels(plan, membership, providers, destination)
        if (destination / 'direct_answer_control_lock.json').is_file():
            lock['direct_answer_control_lock_sha256'] = sha(destination / 'direct_answer_control_lock.json')
            lock['direct_answer_predictions_sha256'] = sha(destination / 'direct_answer_before_labels.csv')
            save_json(destination / 'protocol_lock.json', lock)
        state['protocol_lock_sha256'] = sha(destination / 'protocol_lock.json')
        state['phase'] = 'geometry_and_predictions_frozen_before_new_truth'
        save_json(destination / 'run.json', state)
        verify_selection(selection_path, cfg)
        for path, digest in lock['independent_confirmation_code_sha256'].items():
            if sha(path) != digest:
                raise ValueError('Confirmation implementation changed before labels')
        labels, label_seconds = build_labels(cfg, plan, destination)
        state['offline_new_truth_seconds'] = label_seconds
        result = evaluate_frozen_scores(labels, membership, destination, state['score_sha256_before_labels'], method_names=providers.methods)
        time_live(cfg, plan, result, providers, destination)
        if sha(selection_path) != selection_sha or sha(destination / 'protocol_lock.json') != state['protocol_lock_sha256']:
            raise ValueError('Frozen selection/protocol changed during final confirmation')
        verify_selection(selection_path, cfg)
        state.update(status='complete', total_seconds=time.perf_counter() - start, new_labels_sha256=sha(destination / 'new_queries.csv'), n_candidates=len(labels), eligible_unique_queries=int(labels.eligible.sum()), zero_truth=int(labels.zero_truth.sum()))
        save_json(destination / 'run.json', state)
        print('Completed frozen confirmation:', destination, flush=True)
        return destination
    except Exception as error:
        state.update(status='failed', error=f'{type(error).__name__}: {error}', total_seconds=time.perf_counter() - start)
        save_json(destination / 'run.json', state)
        print('Failed confirmation diagnostics:', destination, flush=True)
        raise
'''

In [ ]:
# Embedded helper: workflow.py
EMBEDDED19['workflow.py'] = r'''"""Portable construction/report wrapper for notebook19; no model training."""
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED
import importlib.metadata
import json
import shutil
import time
import numpy as np
import pandas as pd
from . import input_core, prepare_data, runtime_features, moment_metadata, budget_fine, analytic_moments_fast, confirm_portable

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False, default=str) + '\n')

def prepare_experiment(cfg, experiment_root, implementation_dir):
    """Validate inherited identities, construct exact metadata, then freeze."""
    experiment_root = Path(experiment_root)
    implementation_dir = Path(implementation_dir)
    started = time.perf_counter()
    inputs = experiment_root / 'input_validation'
    inputs.mkdir()
    state = dict(status='preparing', study='compact-distribution-critic', config=cfg, started_utc=datetime.now(timezone.utc).isoformat(), training=False, gate_fitting=False, selected_method='beta_gaussian_copula', fresh_query_seed=cfg['fresh_query_seed'], runtime_packages={name: importlib.metadata.version(name) for name in ('numpy', 'pandas', 'scipy')}, scope='Fresh rectangles and labels on the same authors dataset; inspected17 geometry excluded, populations already examined')
    write_json(experiment_root / 'run.json', state)
    inherited_cfg = dict(cfg, budget_bytes=0)
    data = input_core.load_inputs(inherited_cfg, inputs, state)
    source15 = Path(state['source15_run'])
    source17 = Path(state['source17_run'])
    source08d = Path(state['source08d_run'])
    original = json.loads((source08d / 'run.json').read_text())
    histogram_dir = Path(cfg['histogram_dir']) if cfg['histogram_dir'] else Path(state['original_histogram']['path']).parent
    payload_root = experiment_root / 'payloads'
    payload_root.mkdir()
    coarse_dir = payload_root / 'coarse'
    coarse_dir.mkdir()
    tick = time.perf_counter()
    coarse_files = []
    for record in data['raw_manifest']:
        source_id = int(record['source_id'])
        source_path = histogram_dir / f'src_{source_id}.txt'
        expected = original['input_sha256'][f'run/src_{source_id}.txt']
        input_core._input_verified(source_path, expected, 'original source histogram')
        hist = prepare_data.load_hist(source_path)
        path = coarse_dir / f'src_{source_id}.hist'
        size = runtime_features.save_coarse_payload(path, hist)
        coarse_files.append(dict(source_id=source_id, path=str(path), bytes=size, sha256=input_core.input_sha(path), original_export_sha256=expected))
    contexts = runtime_features.coarse_source_contexts(coarse_dir)
    coarse_bytes = sum((item['bytes'] for item in coarse_files))
    state['coarse_payload_construction_seconds'] = time.perf_counter() - tick
    write_json(coarse_dir / 'manifest.json', dict(status='complete', codec='12 SLHIST01', total_bytes=coarse_bytes, files=coarse_files))
    base = prepare_data.load_hist(state['original_histogram']['path'])
    base_path = payload_root / 'original_H.hist'
    original_bytes = runtime_features.save_coarse_payload(base_path, base)
    raw2_dir = payload_root / 'raw2'
    raw2_dir.mkdir()
    raw2_paths = []
    for record in data['raw_manifest']:
        name = Path(record['payload_path']).name
        choices = [Path(cfg['raw2_dir']) / name] if cfg['raw2_dir'] else [source15 / 'raw_payloads' / name, Path(record['payload_path'])]
        path = next((path for path in choices if path.is_file()), None)
        if path is None:
            raise FileNotFoundError('Saved15 raw2 payload missing: ' + name + '. Set raw2_dir to its unchanged folder')
        input_core._input_verified(path, record['payload_sha256'], 'saved15 raw2 source payload')
        copied = raw2_dir / name
        shutil.copyfile(path, copied)
        raw2_paths.append(copied)
    raw2_bytes = sum((path.stat().st_size for path in raw2_paths))
    moment_dir = payload_root / 'moments'
    raw_paths = {int(record['source_id']): Path(record['raw_path']) for record in data['raw_manifest']}
    raw_hashes = {int(record['source_id']): record['raw_sha256'] for record in data['raw_manifest']}
    tick = time.perf_counter()
    moments = moment_metadata.build_moments(contexts, raw_paths, moment_dir, raw_hashes)
    moment_seconds = time.perf_counter() - tick
    moment_bytes = sum((item['payload_bytes'] for item in moments.values()))
    candidate_auxiliary = coarse_bytes + moment_bytes
    expected = {str(record['raw_path']): dict(raw_sha256=record['raw_sha256'], expected_rows=record['expected_rows']) for record in data['raw_manifest']}
    candidate_total = original_bytes + candidate_auxiliary
    fine = budget_fine.build_fine_counts(list(raw_paths.values()), data['root'], [candidate_auxiliary, raw2_bytes, candidate_total], payload_root / 'fine', expected)
    fine_cost = json.loads((payload_root / 'fine/run.json').read_text())
    aggregate_dir = cfg.get('aggregate_dir', '').strip()
    run_cfg = dict(source15_dir=str(source15), source17_dir=str(source17), source08d_dir=str(source08d), raw_data_dir=str(Path(data['raw_manifest'][0]['raw_path']).parent), queries_file=state['old_query_file']['path'], histogram_dir=str(histogram_dir), raw2_dir=str(raw2_dir), coarse_dir=str(coarse_dir), moment_dir=str(moment_dir), aggregate_dir=aggregate_dir, fine164_path=fine[candidate_auxiliary]['path'], fine186_path=fine[raw2_bytes]['path'], fine164_n=fine[candidate_auxiliary]['n'], fine186_n=fine[raw2_bytes]['n'], fine247_path=fine[candidate_total]['path'], fine247_n=fine[candidate_total]['n'], use_optimized_resident=True, original_histogram_bytes=original_bytes, base_histogram_file=str(base_path), sa_core_file=str(implementation_dir / 'sa_core.py'), exact_counter_file=str(implementation_dir / 'exact_counter.py'), seeds=cfg['seeds'], fresh_query_seed=cfg['fresh_query_seed'], replication_cells=cfg['replication_cells'], timing_queries=cfg['timing_queries'], timing_repeats=cfg['timing_repeats'], output_root=str(experiment_root))
    artifacts = []
    paths = list(implementation_dir.glob('*.py')) + [base_path] + list(coarse_dir.glob('*.hist')) + list(moment_dir.glob('*.bin')) + raw2_paths
    paths.extend((Path(info['path']) for info in fine.values()))
    paths.extend((source15 / 'run.json', source17 / 'run.json', source15 / 'outer_splits.csv', source15 / 'raw_input_manifest.json', source17 / 'new_queries.csv', Path(run_cfg['queries_file'])))
    if aggregate_dir:
        for seed in cfg['seeds']:
            folder = Path(aggregate_dir) / f'aggregate_seed_{seed}' / 'aggregate_relative_l1'
            paths.extend((folder / 'model.txt', folder / 'schema.json'))
    for path in sorted(set((path.resolve() for path in paths))):
        if not path.is_file():
            raise FileNotFoundError('Missing selected artifact: ' + str(path))
        artifacts.append(dict(path=str(path), sha256=input_core.input_sha(path)))
    selection = dict(status='frozen', chosen_method='beta_gaussian_copula', fresh_query_seed=cfg['fresh_query_seed'], seeds=cfg['seeds'], replication_cells=cfg['replication_cells'], rationale='Pre-selected from old grouped inner-validation; this notebook does not select or tune on new labels', algorithm=analytic_moments_fast.protocol(), artifacts=artifacts, prior_development_evidence=dict(source='local pre-confirmation development, 2026-11-16; provenance only, not rerun by this notebook', copula_mean_old_inner_validation_mae_pp=10.861250813693387, copula_mean_old_inner_validation_auc=0.9040134900274222, matched_fine_mean_old_inner_validation_mae_pp=14.637523, matched_fine_mean_old_inner_validation_auc=0.8307763333333332), original_H_unchanged=True, learned_weights_bytes=0, accepted_risk_target=None, operational_gate_fitted=False, limitations=['Moment matching and Gaussian copulas have prior art; novelty is not established', 'Observed Pearson correlation is a plug-in copula parameter, not an exact joint covariance guarantee', 'The count reference and error proxy are approximate; physical coarse bounds are a separate audit', 'Source grids are retained as a union; no bounded heterogeneous ARM merge guarantee'])
    selection_path = experiment_root / 'selection.json'
    write_json(selection_path, selection)
    state.update(status='frozen_before_new_labels', selection_sha256=input_core.input_sha(selection_path), source_coarse_payload_bytes=coarse_bytes, moment_payload_bytes=moment_bytes, candidate_auxiliary_bytes=candidate_auxiliary, original_H_bytes=original_bytes, candidate_total_serialized_bytes=candidate_total, raw2_auxiliary_bytes=raw2_bytes, fine_controls=fine, moment_construction_seconds=moment_seconds, shared_fine_grid_construction_seconds=fine_cost['one_pass_construction_with_hashes_seconds'], preparation_seconds=time.perf_counter() - started, construction_scope='Moments scan raw sources once. All three fine controls share a second raw scan. Hash and initialization costs are separate from resident query time.')
    write_json(experiment_root / 'run.json', state)
    write_json(experiment_root / 'confirmation_config.json', run_cfg)
    return (selection_path, run_cfg, state)

def make_report(experiment_root, result_dir, state):
    """Descriptive English report, keeping cohorts and quality/cost distinct."""
    experiment_root = Path(experiment_root)
    result_dir = Path(result_dir)
    metrics = pd.read_csv(result_dir / 'metrics.csv')
    coverage = pd.read_csv(result_dir / 'fixed_coverage.csv')
    timing = pd.read_csv(result_dir / 'resident_timing_summary.csv')
    costs = pd.read_csv(result_dir / 'payload_costs.csv')
    labels = pd.read_csv(result_dir / 'new_queries.csv')
    paired = pd.read_csv(result_dir / 'paired_effects.csv')
    summary = metrics.groupby(['family', 'method'], as_index=False).agg(mean_seed_mae_pp=('mae_pp', 'mean'), mean_seed_auc=('auc_bad', 'mean'), min_queries_per_seed=('n_queries', 'min'), max_queries_per_seed=('n_queries', 'max'))
    summary.to_csv(result_dir / 'summary.csv', index=False)
    rows = []
    for family, part in summary.groupby('family'):
        for row in part.itertuples(index=False):
            rows.append(f'| {family} | {row.method} | {row.mean_seed_mae_pp:.3f} | {row.mean_seed_auc:.4f} | {row.min_queries_per_seed}–{row.max_queries_per_seed} |')
    skipped = result_dir / 'skipped_baselines.csv'
    skipped_note = ''
    if skipped.is_file() and len(pd.read_csv(skipped)):
        skipped_note = '\nThe optional saved H-aware LightGBM control was skipped. These results do not establish superiority over every learned evaluator.\n'
    text = '# Compact distribution critic: frozen fresh-query replication\n\nThe selected method keeps five quantized distribution moments per original source cell. It combines two moment-matched beta marginals with a fixed Gaussian copula, estimates a source-based reference count, and maps its disagreement with the unchanged ARM answer to an error proxy. It does not train a model or change the ARM answer.\n\n## Full eligible-cohort results\n\nNumbers are descriptive means across the saved seed memberships. Repeated seeds are not independent new datasets. Primary `new_groups` and supplementary `existing_test_groups` remain separate. Actual numerical-error metrics use the inherited offline cohort truth>=10; all zero/small-count candidates are retained in diagnostics.\n\n| Cohort | Evaluator | Error-prediction MAE (percentage points) | Bad-answer AUC | Eligible queries per seed |\n|---|---|---:|---:|---:|\n' + '\n'.join(rows) + skipped_note + '\n\nSA is a reliability ranking heuristic, so numerical relative-error MAE is undefined for SA. Lower MAE and higher AUC are desirable. The fixed-coverage table is a ranking diagnostic, not a fitted operational acceptance gate or a 5% risk guarantee. At100% coverage every evaluator has the same actual-answer violation rate because H is unchanged.\n\n## Cost and interpretation\n\n' + f"The candidate stores {state['source_coarse_payload_bytes']:,}B of exact source histograms plus {state['moment_payload_bytes']:,}B of moment payloads, for {state['candidate_auxiliary_bytes']:,}B of auxiliary information and {state['candidate_total_serialized_bytes']:,}B including the unchanged original histogram. Learned weights:0B. The matched finer control is selected by bytes alone; its actual payload and unused capacity are in payload_costs.csv and the parent run.json.\n\nMetadata construction took {state['moment_construction_seconds']:.3f}s. The three direct-count finer controls shared a separate {state['shared_fine_grid_construction_seconds']:.3f}s construction pass including hashes. These joint construction costs are not any individual control's standalone cost. Resident timing includes live H estimation, live summary-derived features, the analytic calculation and error-score mapping; initialization/loading is separate. It excludes raw fallback and does not establish E2E query speedup. CSVs report mean and p95 timing, repeat IDs and warmed resident caching.\n" + '\n## What remains unproved\n\nThese are fresh rectangles and exact raw labels on the same authors dataset. The source generating populations were inspected in17; they are not a new dataset or strict geographic holdout. Rerunning this notebook with the same seed reproduces the same geometry and is not another independent confirmation. Do not tune the seed, formula or threshold after viewing these labels.\n\nThe analytic reference is an approximation, not a confidence bound or a guaranteed exact answer. Quantized moments do not determine the full within-cell distribution. Plugging observed Pearson correlation into a Gaussian copula does not exactly preserve covariance with beta marginals. Adaptive histograms, moment matching and copulas have prior art; novelty still needs a focused literature comparison.\n\nSource grids remain a source union. The prototype does not solve bounded-size merging of arbitrary heterogeneous ARM histograms. A credible next step, if these quality/cost results persist, is to freeze this evaluator and validate it on a different raw dataset and query family, then measure exact-fallback E2E cost. A negative comparison against the stronger fine control must be reported rather than tuned away.\n'
    (result_dir / 'analysis_en.md').write_text(text)
    import matplotlib.pyplot as plt
    families = [f for f in ('new_groups', 'existing_test_groups') if f in set(summary.family)]
    fig, axes = plt.subplots(1, len(families), figsize=(7 * len(families), 4), squeeze=False)
    for ax, family in zip(axes[0], families):
        p = summary[(summary.family == family) & summary.mean_seed_mae_pp.notna()].sort_values('mean_seed_mae_pp')
        ax.barh(p.method, p.mean_seed_mae_pp)
        ax.invert_yaxis()
        ax.set_title(family)
        ax.set_xlabel('Error-prediction MAE (percentage points)')
    fig.tight_layout()
    fig.savefig(result_dir / 'error_prediction.png', dpi=160, bbox_inches='tight')
    plt.close(fig)
    fig, axes = plt.subplots(1, len(families), figsize=(7 * len(families), 4), squeeze=False)
    for ax, family in zip(axes[0], families):
        p = coverage[coverage.family == family].groupby(['method', 'nominal_coverage'], as_index=False).agg(coverage=('actual_coverage', 'mean'), risk=('violation_rate', 'mean'))
        for method, g in p.groupby('method'):
            ax.plot(100 * g.coverage, 100 * g.risk, marker='o', label=method)
        ax.set_title(family)
        ax.set_xlabel('Selected answers (%; complete ties)')
        ax.set_ylabel('Violation rate among selected answers (%)')
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(result_dir / 'risk_coverage.png', dpi=160, bbox_inches='tight')
    plt.close(fig)
    p = timing.groupby('method', as_index=False).agg(mean_ms=('mean_ms', 'mean'), p95_ms=('p95_ms', 'mean')).sort_values('mean_ms')
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(p))
    ax.bar(x - 0.18, p.mean_ms, 0.36, label='Mean')
    ax.bar(x + 0.18, p.p95_ms, 0.36, label='Mean of per-membership p95')
    ax.set_xticks(x, p.method, rotation=25, ha='right')
    ax.set_ylabel('Resident evaluator latency (ms)')
    ax.legend()
    fig.tight_layout()
    fig.savefig(result_dir / 'resident_latency.png', dpi=160, bbox_inches='tight')
    plt.close(fig)
    state.update(status='complete', confirmation_dir=str(result_dir), completed_utc=datetime.now(timezone.utc).isoformat(), confirmation_manifest_sha256=input_core.input_sha(result_dir / 'run.json'))
    write_json(experiment_root / 'run.json', state)
    return summary

def package_results(experiment_root, result_dir):
    experiment_root = Path(experiment_root)
    result_dir = Path(result_dir)
    archive = experiment_root / 'compact_distribution_results.zip'
    with ZipFile(archive, 'w', ZIP_DEFLATED) as stream:
        for path in sorted(result_dir.iterdir()):
            if path.is_file() and path.suffix.lower() in ('.csv', '.json', '.md', '.png'):
                stream.write(path, 'confirmation/' + path.name)
        for name in ('run.json', 'selection.json', 'confirmation_config.json'):
            path = experiment_root / name
            if path.is_file():
                stream.write(path, name)
        for path in (experiment_root / 'payloads').rglob('*manifest.json'):
            stream.write(path, str(path.relative_to(experiment_root)))
        fine = experiment_root / 'payloads/fine/run.json'
        if fine.is_file():
            stream.write(fine, 'payloads/fine/run.json')
    return archive
'''

In [ ]:
import hashlib
import importlib
import sys
from uuid import uuid4

_run_id19 = uuid4().hex[:12]
experiment_root19 = Path(cfg19["output_root"]).expanduser() / ("compact-distribution-critic-" + _run_id19)
experiment_root19.mkdir(parents=True, exist_ok=False)
_package19 = "_implementation_" + _run_id19
implementation_dir19 = experiment_root19 / _package19
implementation_dir19.mkdir()
(implementation_dir19 / "__init__.py").write_text("")
for name, source in EMBEDDED19.items():
    if hashlib.sha256(source.encode()).hexdigest() != EMBEDDED19_SHA256[name]:
        raise ValueError("Embedded implementation hash mismatch: " + name)
    (implementation_dir19 / name).write_text(source)
sys.path.insert(0, str(experiment_root19))
workflow19 = importlib.import_module(_package19 + ".workflow")
confirmation19 = importlib.import_module(_package19 + ".confirm_portable")
print("New run directory:", experiment_root19)

## Construct and freeze summaries before new labels

Source identities, original splits and old query geometry are verified first. Exact source masses must match raw rows.
The selected method's moments are encoded, saved, reloaded and charged by actual bytes. All finer controls share a
separate direct raw-data counting pass. Construction has no query-truth or score argument.

In [ ]:
selection_path19, confirmation_cfg19, preparation_state19 = workflow19.prepare_experiment(
    cfg19, experiment_root19, implementation_dir19
)
print("Frozen selection:", selection_path19)
print("Candidate auxiliary bytes:", preparation_state19["candidate_auxiliary_bytes"])
print("Candidate total bytes including unchanged H:", preparation_state19["candidate_total_serialized_bytes"])
print("Fine control grid sizes:", confirmation_cfg19["fine164_n"], confirmation_cfg19["fine186_n"])

## Freeze new geometry and predictions, then count raw truth

This step runs the authorized experiment. It saves query plans and all scores before exact labeling. The exact spatial
counter reads original point rows;32 fixed query IDs also receive an independent brute-force audit. No precomputed truth
lookup substitutes for this raw scan. Do not replace excluded truth<10 queries to improve the cohort.

In [ ]:
result_dir19 = confirmation19.run_confirmation(
    selection_path19, confirmation_cfg19, allow_label_generation=True
)
print("Completed confirmation:", result_dir19)

## Inspect full-cohort quality and measured costs

Lower numerical error MAE is better; higher bad-answer AUC is better. Always keep answer quality, selected-answer risk,
sample counts and storage/latency together. SA's score is not assigned numerical-error MAE. The optional learned-control
status appears in skipped_baselines.csv.

In [ ]:
summary19 = workflow19.make_report(experiment_root19, result_dir19, preparation_state19)
try:
    display(summary19)
except NameError:
    print(summary19.to_string(index=False))
print("English analysis:", result_dir19 / "analysis_en.md")
print("Figures:", result_dir19 / "error_prediction.png", result_dir19 / "risk_coverage.png", result_dir19 / "resident_latency.png")

## Export results

Share the ZIP below. It contains new-query predictions, aggregate metrics, paired effects, quality/ranking/timing tables,
costs, audits, protocol identities, English analysis and figures. Raw data, old model weights and temporary preparation
tables are excluded. The full private run folder keeps the implementation and persisted summary payloads.

In [ ]:
archive19 = workflow19.package_results(experiment_root19, result_dir19)
print("Share this file:", archive19)
print("Retained full run directory:", experiment_root19)